# 🎙️ RVC v2 — Audio Dataset Engineering Pipeline **PRO**

**من التنزيل إلى مجموعة بيانات جاهزة لتدريب RVC v2 — بأقصى جودة ممكنة مع المحافظة الكاملة على هوية المتحدث.**

هذا الدفتر ليس مجرد "تنظيف صوت" — إنه **خط معالجة هندسي متكامل (Production Pipeline)** مبني على 25 مرحلة (`STAGE 00` → `STAGE 24`)، كل مرحلة:

- تقيس أولًا ثم تقرر (لا معالجة إجبارية),
- توثّق كل قرار في قاعدة حالة SQLite + تقارير CSV + Manifest,
- قابلة لإعادة التشغيل (Resume) والذاكرة المؤقتة (Cache) المُبطلة تلقائيًا عند تغيّر الإعدادات,
- تحمي **هوية المتحدث** و**التفاصيل الصوتية** فوق كل اعتبار.

### المبادئ الهندسية غير القابلة للتفاوض
1. **هوية المتحدث أولًا** — أي معالجة تُخِلّ بالهوية تُرفض ويُحتفظ بالأصل.
2. **لا افتراضات** — كل قرار مبني على قياس (DNSMOS، بصمة ECAPA، تحليل طيفي) لا على "شهرة" نموذج.
3. **الأصل محفوظ دائمًا** — المصدر الخام لا يُعدَّل أبدًا، وكل النسخ المعالجة قابلة للتتبع إليه.
4. **الصدق** — لا قيم مُختلقة، ولا "SNR" بلا مرجع ضوضاء حقيقي، ولا Sample Peak يُسمّى True Peak.

### جدول المحتويات

| # | المرحلة | الوظيفة |
|---|---------|---------|
| 00 | Project Configuration | `CONFIG` مركزي + مُدقّق + قاعدة حالة SQLite |
| 01 | Environment Intelligence | كشف القدرات الفعلية + مدير اعتماديات ذكي (لا يكسر torch) |
| 02 | Source Acquisition | تنزيل قوي + أمان الأرشيف (ZIP-Slip / Bombs / تكرارات) |
| 03 | Audio Inventory | جرد كامل بـ ffprobe + **معرفات مستقرة** قائمة على المحتوى |
| 04 | Robust Decoding | فك ترميز موحّد + تحليل Stereo ذكي + فحص جودة المُفكّك |
| 05 | Signal Integrity | خصائص زمنية/طيفية + **True Peak حقيقي** + تصنيف الـ Clipping |
| 06 | Deduplication | 4 مستويات: Hash → Canonical → بصمة طيفية → احتواء جزئي |
| 07 | Speech Analysis | نشاط الكلام + إحصاءات النطق |
| 08 | Speaker Verification | ECAPA-TDNN + معايرة عتبة + حالات حدودية للمراجعة |
| 09 | Music / Noise / Interference | كشف متعدد المناطق (بداية/وسط/نهاية) + تصنيف 7 فئات |
| 10 | Conditional Separation | فصل موسيقى **فقط** للملفات المُعلَمة (BS-RoFormer / MelBand / UVR) |
| 11 | Adaptive Enhancement | محرك متعدد الفروع: كل مرشّح يبدأ من نفس المصدر |
| 12 | Conditional Dereverb | إزالة صدى **فقط** عند وجود دليل قياسي |
| 13 | Enhancement Selection | محرك قرار متعدد المعايير (موثّق الأوزان) |
| 14 | Identity Preservation | مقارنة بصمة المتحدث قبل/بعد لكل نسخة معالجة |
| 15 | Intelligent VAD | Silero + Hysteresis + Ensemble + حماية الكلام الهادئ |
| 16 | Utterance Segmentation | تقطيع عند **الصمت الطبيعي** + تدقيق الحدود + فُصِل حسب `CONFIG` |
| 17 | Segment Verification | فحص سلامة كل مقطع (بداية/نهاية/اكتمال/تداخل) |
| 18 | Sample Rate Preparation | SoXR VHQ — إعادة عينة **واحدة** فقط عند الحاجة |
| 19 | Final Export | Master نظيف + Clips منسّقة (bit-depth موصول فعليًا) |
| 20 | Multi-Dimensional QA | فحص **جميع** الملفات: تقني/كلامي/متحدث/إدراكي/طيفي |
| 21 | Diversity Analysis | تحليل تنوّع مجموعة البيانات (طبقة/مدة/جودة) |
| 22 | Dataset Split | Train/Evaluation/Quarantine بلا تسريب بين المجموعات |
| 23 | Manifest & Packaging | Provenance كامل + `dataset_index.csv` + ZIP |
| 24 | Final Audit | تدقيق نهائي + تقرير إنجاز + الإجابة عن 20 سؤالًا لكل ملف |

### طريقة التشغيل
1. شغّل الخلايا **من الأعلى إلى الأسفل** بالترتيب — كل الإعدادات في `CONFIG` بالخلية التالية.
2. في Google Colab: سيعمل الجلب والتثبيت تلقائيًا (GPU يُستخدم إن وُجد دون أي تغيير).
3. محليًا: ضع `Voice.zip` بجانب الدفتر — سيُكتشف تلقائيًا (أو اضبط `input.archive_url`).
4. أوقف وأعد التشغيل متى شئت — **المراحل المنجزة تُستأنف من الذاكرة المؤقتة** تلقائيًا.

> **ملاحظة الجودة:** كل خلية كود مسبوقة بخلية Markdown تشرح هدفها ومدخلاتها ومخرجاتها وأسلوب فشلها (Documentation-First).

## STAGE 00 — Project Configuration & Initialization

**الهدف:** نقطة التحكم الوحيدة للمشروع. كل سلوك في الخط يقرأ من هنا فقط — لا قيم مضمّنة داخل الدوال.

- **مدخلات:** إعدادات المستخدم.
- **مخرجات:** `CONFIG` مُتحقَّق منه عبر `validate_config()` + إصدار الخط `PIPELINE_VERSION` الذي يدخل في مفاتيح الذاكرة المؤقتة.
- **أوضاع الفشل:** قيم غير صالحة → يفشل مبكرًا مع رسائل واضحة قبل أي معالجة (Fail-Fast)، لا وسط الطريق.

### خريطة `CONFIG` (12 قسمًا)

| القسم | يتحكم في |
|-------|----------|
| `input` | رابط الأرشيف، المسارات المحلية، الامتدادات المدعومة، مهلات التنزيل |
| `runtime` | الجهاز (auto/cpu/cuda)، حجم الدفعة، العمال، حدود الذاكرة |
| `audio` | نمط RVC الهدف (48k/40k/32k)، bit-depth، حماية القمم، إدارة الجهارة **الاختيارية** |
| `speaker` | التحقق من المتحدث: المراجع، العتبات، سياسة الحالات الحدودية |
| `noise` | بوابات وسياسات التنقية وفروعها (UVR-DeNoise / noisereduce / DeepFilterNet / …) |
| `music` | بوابات كشف الموسيقى والفصل ونماذجه |
| `dereverb` | بوابة كشف الصدى ونموذج إزالته |
| `vad` | نموذج VAD وعتباته وفواصله الزمنية وحدود المقاطع |
| `dedup` | مستويات إزالة التكرار وعتباتها |
| `quality` | DNSMOS والفحوصات وعتبات القبول |
| `export` | مجلدات المخرجات والتقارير والتغليف |
| `pipeline` | Quick Test، الذاكرة المؤقتة، السجلات، اختيار المراحل |

> **قاعدة:** أي مفتاح هنا يجب أن يكون موصولًا فعليًا بشيفرات المراحل — إن وجدت مفتاحًا لا يؤثر في التنفيذ فهو خطأ هندسي (و`validate_config()` يفحص المفاتيح غير المعروفة).

In [ ]:
# -*- coding: utf-8 -*-
"""STAGE 00 — CONFIG: نقطة التحكم الوحيدة لخط المعالجة بأكمله."""
from pathlib import Path

PIPELINE_VERSION = "PRO-1.0.0"  # يدخل في مفاتيح الـ Cache — أي تغيير هنا يبطل النتائج القديمة

CONFIG = {
    # ================ A. INPUT — المصدر ================
    "input": {
        "archive_url": "https://drive.usercontent.google.com/download?id=1dXino3_rN6uV6RUGJpyfo3U4_bxxBXs8&export=download&authuser=0&confirm=t",
        # أسماء ملفات مقبولة لو نزّل المستخدم الأرشيف يدويًا بجانب الدفتر
        "manual_names": ["Voice.zip", "voice.zip", "VOICE.zip"],
        "local_archive": "",                # مسار صريح لأرشيف محلي (يفضّل على التنزيل إذا غير فارغ)
        "extensions": [".wav", ".mp3", ".ogg", ".opus", ".m4a", ".aac",
                       ".flac", ".mp4", ".webm", ".aiff", ".aif", ".wma"],
        "max_file_mb": 4096,                # رفض ملفات صوتية أكبر من هذا (حماية)
        "download": {
            "timeout_s": 120,               # مهلة الشبكة للطلب الواحد
            "retries": 4,                   # محاولات إعادة
            "backoff_s": 3.0,               # معامل التراجع الأُسّي
            "chunk_mb": 8,                  # حجم قطعة التدفق
            "min_zip_mb": 1,                # أصغر حجم منطقي للأرشيف
            "sha256": "",                   # إن عُرف مسبقًا يُتحقق منه (اختياري)
        },
    },

    # ================ B. RUNTIME — بيئة التشغيل ================
    "runtime": {
        "device": "auto",                   # auto | cpu | cuda
        "batch_size": 1,                    # معالجة الملفات تتابعيًا (حد الذاكرة)
        "max_workers": 1,                   # الخيوط المتوازية للمراحل القابلة للتوازي
        "memory_cleanup": True,             # gc + إفراغ ذاكرة CUDA بين الفروع
        "deterministic": True,             # تثبيت البذور
        "seed": 42,
    },

    # ================ C. AUDIO — الصيغة الهدف ================
    "audio": {
        # نمط توافق RVC — يحدد Sample Rate الهدف (لا فرض أعمى):
        #  rvc_v2_48k  : RVC v2 أصلي بتوليفات 48kHz (الأكثر شيوعًا، والأعلى استيعابًا للترددات)
        #  rvc_v2_40k  : RVC v2 بتوليفات 40kHz (توليفات Applio/RVC القديمة 40k)
        #  rvc_v2_32k  : RVC v2 بتوليفات 32kHz (خفيفة، هاتفية المصدر)
        "rvc_profile": "rvc_v2_48k",
        # bit-depth للتصدير (PCM): 16 | 24 — يُوصل فعليًا في STAGE 19
        "bit_depth": 16,
        "channels": 1,                      # RVC يتدرب على Mono
        # حماية القمم: يُطبَّق True-Peak (بفرط تعيين 4x) لا Sample-Peak
        "true_peak_ceiling_db": -1.0,      # سقف القمة الحقيقية عند التصدير
        # إدارة الجهارة: افتراضيًا "لا شيء" — نحافظ على الديناميكية الطبيعية.
        "loudness": {
            "mode": "none",                 # none | gain_only
            # gain_only: تعويض ربح ساكن فقط للملفات الشاذة دون تطبيع LUFS إجباري
            "target_lufs": None,            # مثال: -23.0 — لا يُستخدم إلا في mode=gain_only
            "max_boost_db": 6.0,            # حد أقصى لأي ربح
        },
    },

    # ================ D. SPEAKER — هوية المتحدث ================
    "speaker": {
        "verification_enabled": True,       # يعمل عند توفر speechbrain + الأوزان
        # مجلد تسجيلات مرجعية مؤكدة (wav/mp3، 5 ثوانٍ فأكثر لكل ملف).
        #  - إن وُجد: تُبنى بصمة المرجع منه.
        #  - إن لم يوجد: وضع Bootstrap — يحسب الخط بصمةً من أغلب وسط الأرشيف
        #    ويصنّف باقي الملفات بالنسبة إليها مع وسم bootstrap_reflected=True.
        "reference_dir": "",
        # عتبة التشابه: "auto" = تُعاير من توزع التشابه داخل الأرشيف نفسه
        # أو رقم صريح (0.0-1.0)
        "match_threshold": "auto",
        "min_speech_for_embedding_s": 3.0, # أقل كلام يُبنى عليه embedding موثوق
        "borderline_band": 0.05,            # عرض نطاق الحالات الحدودية حول العتبة
        "multi_speaker_policy": "review",   # review|quarantine — عند اكتشاف متحدثين
    },

    # ================ E. NOISE — التنقية التكيفية ================
    "noise": {
        "detection": {
            "dnsmos_bak_below": 3.55,       # BAK أقل من هذا = ضوضاء مؤثرة (DNSMOS P.835)
            "floor_estimate_above_db": -34.0,  # أرضية ضوضاء مُقدَّرة أعلى من هذا
            "quiet_speech_guard": True,     # لا تُعَلَّم ملفات كلام هادئ كضوضاء
        },
        "policy": "conditional",            # conditional — لا تنقية إلا ببوابة قياسية
        # فروع التنقية المرشحة (كلها تبدأ من نفس المصدر — مقارنة عادلة).
        # المرتبة = أولوية التجربة. المتاح يعمل، والمفقود يُتخطى بتوثيق.
        "branches": ["uvr_denoise", "noisereduce", "deepfilternet", "mossformer2", "resemble"],
        # قبول نتيجة التنقية بدل الأصل يتطلب تحسنًا صافيًا:
        "accept_margin": {
            "dnsmos_ovrl": 0.08,            # +OVRL أعلى بهذا القدر على الأقل
            "min_speaker_sim": 0.80,        # وبصمة متحدث ≥ هذا
        },
    },

    # ================ F. MUSIC — الموسيقى والفصل ================
    "music": {
        "detection": {
            "regions": 3,                   # فحص البداية والوسط والنهاية (لا عينة واحدة)
            # معايرة مقيسة فعليًا: هاتف/ميكروفون قريب يعطي 0.21-0.46 طاقة <120Hz بلا موسيقى
            # (جهارة الصوت القريبة) — الفصل عند 0.55 يميز الباص الموسيقي الحقيقي
            "flag_chroma_energy_ratio": 0.55,
            "flag_flatness": 0.30,          # مسطّح طيفي فوق هذا في غير الكلام = آلة/موسيقى
        },
        "policy": "conditional",            # لا فصل إلا للملفات المُعلَمة قياسيًا
        "separation_model": "auto",         # auto = أقوى المتاح بالترتيب أدناه
        "model_priority": [
            "model_bs_roformer_ep_368_sdr_12.9628.ckpt",     # BS-RoFormer — أعلى SDR مجتمعيًا
            "model_mel_band_roformer_ep_3005_sdr_11.4360.ckpt",  # MelBand RoFormer
            "UVR-MDX-NET-Inst_HQ_3.onnx",                    # MDX-Net — أخف
        ],
        "accept_margin_dnsmos_ovrl": 0.10,  # قبول الفصل يتطلب تحسن OVRL صافيًا
    },

    # ================ G. DEREVERB — الصدى ================
    "dereverb": {
        "detection": {
            "envelope_smooth_ratio": 0.60,  # نعومة الغلاف فوق هذا = صدى محتمل
            "decay_tail_ms": 240,           # ذيل اضمحلال أطول من هذا = صدى محتمل
        },
        "policy": "conditional",            # لا إزالة صدى إلا ببوابة قياسية
        "model": "UVR-DeEcho-DeReverb.pth",
        "accept_margin_dnsmos_ovrl": 0.10,
        "min_speaker_sim": 0.80,            # هوية المتحدث فوق هذا وإلا يُرفض
    },

    # ================ H. VAD — اكتشاف الكلام ================
    "vad": {
        "model": "silero",                  # silero | energy (احتياطي آلي عند غياب الأول)
        "threshold": 0.50,                   # عتبة الصعود للكلام
        "neg_threshold": 0.35,              # عتبة الهبوط (Hysteresis) — يمنع التذبذب
        "min_speech_ms": 250,               # أقصر كلام يُعتد به
        "min_silence_ms": 350,              # أقصر صمت يفصل مقطعين
        "speech_pad_ms": 120,               # هامش حول الكلام (تنفّس/حروف ختامية)
        "max_speech_s": 30.0,               # أقصى جملة قبل التقسيم (VAD داخلي)
        # حماية الكلام الهادئ: لا يُعد كلام RMS منخفض صمتًا إن ظهر كصوت مهموس
        "quiet_speech_rescue": True,
    },

    # ================ H2. SEGMENTATION — التقطيع ================
    "segmentation": {
        "min_clip_s": 2.0,                  # أقصر مقطع تدريب مطلوب (موصول فعليًا)
        "max_clip_s": 15.0,                 # أطول مقطع (RVC يقبل حتى ~15s افتراضيًا)
        "drop_below_s": 0.8,                # دون هذا يُهمل (كلام مبتور غير مفيد)
        "keep_short_between": True,         # بين drop_below و min: يُحفظ مع وسم short
        "merge_gap_s": 0.9,                 # دمج مقاطع تفصلها فجوة أقل من هذا
        "boundary_search_s": 1.5,           # نافذة البحث عن حد قطع طبيعي حول المركز
        "leading_silence_max_s": 0.6,       # سماحية صمت في بداية المقطع
        "trailing_silence_max_s": 0.6,      # سماحية صمت في نهايته
    },

    # ================ I. DEDUP — إزالة التكرارات ================
    "dedup": {
        "levels": ["file_hash", "canonical_hash", "acoustic_fingerprint", "containment"],
        "fingerprint": {
            "sr": 16000, "n_mels": 64, "fps": 4,   # بصمة خشنة مقاومة لإعادة الترميز
            "sim_threshold": 0.965,               # تشابه الكوساين فوق هذا = تكرار محتمل
            "duration_ratio_min": 0.92,           # مدد متقاربة فقط تدخل المقارنة الدقيقة
        },
        "containment": {
            "min_short_s": 2.0,            # أقصر ملف يُفحص كاحتواء جزئي
            "xcorr_threshold": 0.85,       # ارتباط مغلف متقاطع فوقه = ملف مضمن
            "envelope_fps": 50,            # دقة الغلاف للاحتواء
        },
    },

    # ================ J. QUALITY — الجودة ================
    "quality": {
        "dnsmos": {"enabled": True, "personalized": False},  # P.835 SIG/BAK/OVRL
        "min_file_dnsmos_ovrl": 2.00,      # أرضية القمامة الفعلية للملف الكامل
        "degradation_reject": 0.35,         # تدهور المقطع عن مداه بالمصدر: REJECTED
        "degradation_review": 0.15,         # أقل: NEEDS_REVIEW
        "min_speaker_sim_processed": 0.80, # بصمة النسخة المعالجة مقابل الأصل
        "max_true_peak_db": -0.8,          # قمة حقيقية أعلى من هذا: تحذير قص
        "max_dc_offset": 0.02,
        "max_clipping_ratio": 0.001,       # نسبة عينات مقصوصة تُعتبر مشكلة
        "review_low_confidence": 0.70,     # ثقة قرار أدنى من هذا: مراجعة بشرية
    },

    # ================ K. EXPORT — المخرجات ================
    "export": {
        "root": "RVC_Audio_Dataset",        # جذر شجرة المخرجات (بجانب الدفتر)
        "master": True, "clips": True,     # تصدير Master الكامل + Clips منسّقة
        "package_zip": True,                # تغليف نهائي ZIP
        "zip_name": "rvc_dataset.zip",
        "reports": True, "visualizations": True,
    },

    # ================ L. PIPELINE — سلوك التنفيذ ================
    "pipeline": {
        "quick_test": False,                # عينة ممثلة طبقيًا (ليست الأقصر!) — للتحقق السريع
        "quick_test": {
            "enabled": False,
            "n_files": 4,                   # عدد ملفات العينة
            "max_seconds_per_file": 90.0,   # قص العينة لتسريع الدورة (يخص الاختبار فقط)
        },
        "stages": "all",                    # "all" أو قائمة أرقام مثل [0,1,2]
        "use_cache": True,                  # تخطي المراحل المنجزة بمفتاح صالح
        "force_reprocess": [],              # قائمة مراحل تُعالَج قسرًا رغم الـ Cache
        "log_level": "INFO",
        "log_file": "pipeline_run.log",
    },
}

# ── تجاوزات الاختبار الآلي (Harness/CI فقط — لا تأثير على تشغيل الدفتر يدويًا) ──
_HARNESS_OVERRIDES = globals().get("__OVERRIDES__") or {}

def _apply_harness_overrides(overrides):
    """تطبيق تجاوزات مسطحة (a.b.c = قيمة) لقيم CONFIG — لأتمتة الاختبار فقط."""
    for dotted, val in overrides.items():
        keys = dotted.split(".")
        node = CONFIG
        for k in keys[:-1]:
            node = node.setdefault(k, {})
        node[keys[-1]] = val

_apply_harness_overrides(_HARNESS_OVERRIDES)

### STAGE 00.1 — المسارات، البذور، السجل، وتصنيف الأخطاء

**الهدف:** طبقة مشتركة تخدم كل المراحل: مسارات نسبية فقط (تعمل في Colab ومحليًا)، بذور مثبتة، سجل منظم، وتصنيف أخطاء صريح.

- **أهم قرارات التصميم:**
  - `RuntimeError` الأعمى ممنوع — لكل خطأ صنف معروف يميّز بين: اعتمادية اختيارية مفقودة، فشل نموذج، صوت غير صالح، نفاد موارد، وخطأ سلامة بيانات.
  - الكتابة الذرية `atomic_write_audio` في كل مخرجات الصوت: كتابة إلى ملف مؤقت ثم `os.replace` — لا ملف نصف مكتوب يُعتبر نتيجة أبدًا.
- **مخرجات:** `PATHS`, `LOGGER`, أصناف الأخطاء `PipelineError` وذرياتها.

In [ ]:
# -*- coding: utf-8 -*-
"""STAGE 00.1 — المسارات + البذور + السجل + تصنيف الأخطاء + كتابة ذرية."""
import gc as _gc
import hashlib as _hashlib
import json as _json
import logging as _logging
import os as _os
import random as _random
import shutil as _shutil
import sqlite3 as _sqlite3
import tempfile as _tempfile
import time as _time
import warnings as _warnings
from datetime import datetime as _dt
from pathlib import Path as _Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# ---- البذور (قابلية إعادة الإنتاج) ----
if CONFIG["runtime"]["deterministic"]:
    _random.seed(CONFIG["runtime"]["seed"])
    np.random.seed(CONFIG["runtime"]["seed"])
    try:
        import torch as _torch_mod
        _torch_mod.manual_seed(CONFIG["runtime"]["seed"])
        if _torch_mod.cuda.is_available():
            _torch_mod.cuda.manual_seed_all(CONFIG["runtime"]["seed"])
    except ImportError:
        pass  # torch اختياري — يوثق لاحقًا في STAGE 01

# ---- المسارات (نسبية إلى مجلد الدفتر — تعمل في Colab ومحليًا) ----
NB_DIR = _Path.cwd()
EXPORT_ROOT = NB_DIR / CONFIG["export"]["root"]

def _mk(p):
    p.mkdir(parents=True, exist_ok=True)
    return p

PATHS = {
    "nb_dir": NB_DIR,
    "archive": NB_DIR / "data" / "archives",        # الأرشيف كما نزل
    "raw": NB_DIR / "data" / "raw",                 # الصوت المستخرج (لا يُعدَّل أبدًا)
    "work": NB_DIR / "work",                        # الوسائط والذاكرة المؤقتة
    "decoded": None,                                 # يُملأ في STAGE 04
    "export_root": EXPORT_ROOT,
    "models_cache": NB_DIR / "models_cache",        # أوزان النماذج المؤقتة
    "reports": EXPORT_ROOT / "reports",
    "visualizations": EXPORT_ROOT / "visualizations",
    "cache": NB_DIR / "work" / "cache",
}
for _k in ("archive", "raw", "work", "models_cache", "reports", "visualizations", "cache"):
    _mk(PATHS[_k])

# ---- السجل المنظم (ملف + تيار، دون تكرار المعالجات عند إعادة التشغيل) ----
def _build_logger():
    lg = _logging.getLogger("rvc_pipeline")
    lg.setLevel(getattr(_logging, CONFIG["pipeline"]["log_level"], _logging.INFO))
    if lg.handlers:                       # منع التكرار عند إعادة تشغيل الخلية
        return lg
    fmt = _logging.Formatter("%(asctime)s | %(levelname)-8s | %(message)s", "%H:%M:%S")
    fh = _logging.FileHandler(NB_DIR / CONFIG["pipeline"]["log_file"], encoding="utf-8")
    fh.setFormatter(fmt); fh.setLevel(_logging.DEBUG)
    sh = _logging.StreamHandler()
    sh.setFormatter(fmt); sh.setLevel(_logging.INFO)
    lg.addHandler(fh); lg.addHandler(sh)
    lg.propagate = False
    return lg

LOGGER = _build_logger()
LOGGER.info("=" * 60)
LOGGER.info("PIPELINE %s — START | cwd=%s", PIPELINE_VERSION, NB_DIR)

# ---- تصنيف الأخطاء (خطأ منطقي لا يُخفى أبدًا كنجاح) ----
class PipelineError(Exception):
    """الأساس — كل أخطاء الخط معروفة الصنف."""

class OptionalDependencyMissing(PipelineError):
    """اعتمادية اختيارية غائبة — تُتخطى الميزة بتوثيق لا بفشل صامت."""

class ModelLoadError(PipelineError):
    """فشل تحميل نموذج أو أوزانه (تنزيل، إصدار، فساد)."""

class InvalidAudioError(PipelineError):
    """ملف صوتي غير قابل للفك أو تالف بعد الفك."""

class DataIntegrityError(PipelineError):
    """خرق سلامة: طول غير متسق، NaN/Inf، مخرج مفقود بعد نجاح مُعلن."""

class ResourceExhaustedError(PipelineError):
    """نفاد ذاكرة/قرص — قابل للمحاولة بجهاز/دفعة مختلفة."""

def _warn_once(msg, category=None):
    """تحذير مرة واحدة لكل رسالة — دون إخفاء عالمي للتحذيرات."""
    key = str(category or "") + msg
    if key not in _WARNED:
        _WARNED.add(key)
        _warnings.warn(msg, category or RuntimeWarning, stacklevel=2)

_WARNED = set()

# ---- أدوات هاش مشتركة ----
def sha256_bytes(data: bytes) -> str:
    return _hashlib.sha256(data).hexdigest()

def sha256_file(path, chunk=1 << 20) -> str:
    h = _hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

# ---- كتابة صوت ذرية (لا ملف نصف مكتوب يعتبر نتيجة) ----

### الأدوات الذرية والمشتركة

كتابة صوت/JSON/بايت **ذرية** (`tmp → fsync → rename`) لكل مخرجات الخط (لا ملف نصف مكتوب يُعتبر نتيجة أبدًا)، مع دوال dB وحراسة الذاكرة وطابع الوقت.

In [ ]:
def atomic_write_audio(path, y, sr, subtype="FLOAT"):
    """كتابة صوتية ذرية: tmp → fsync → rename.

    subtype حقيقي من soundfile (PCM_16/PCM_24/FLOAT) — لا إعدادات شكلية.
    """
    path = _Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + f".tmp{_os.getpid()}")
    try:
        import soundfile as _sf
        # الصيغة صراحة (الاسم المؤقت يفقد الامتداد) — WAV/FLAC حسب الامتداد النهائي
        fmt = "WAV" if path.suffix.lower() == ".wav" else None
        if fmt:
            _sf.write(str(tmp), y, sr, subtype=subtype, format=fmt)
        else:
            _sf.write(str(tmp), y, sr, subtype=subtype)
        with open(tmp, "rb") as f:
            _os.fsync(f.fileno())
        _os.replace(tmp, path)
    finally:
        if tmp.exists():
            tmp.unlink(missing_ok=True)
    return path

def atomic_write_json(path, obj):
    path = _Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + f".tmp{_os.getpid()}")
    tmp.write_text(_json.dumps(obj, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    _os.replace(tmp, path)
    return path

def atomic_write_bytes(path, data: bytes):
    path = _Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + f".tmp{_os.getpid()}")
    tmp.write_bytes(data)
    _os.replace(tmp, path)
    return path

def db_to_scalar(x, floor=1e-12):
    """خطي → dBFS مع أرضية ضد اللانهاية."""
    x = np.asarray(x, dtype=np.float64)
    return 20.0 * np.log10(np.maximum(np.abs(x), floor))

def memory_cleanup():
    """تحرير الذاكرة بين الفروع الثقيلة (يحترم إعداد runtime)."""
    if CONFIG["runtime"]["memory_cleanup"]:
        _gc.collect()
        try:
            import torch as _t
            if _t.cuda.is_available():
                _t.cuda.empty_cache(); _t.cuda.ipc_collect()
        except ImportError:
            pass

def now_iso() -> str:
    return _dt.now().isoformat(timespec="seconds")

### STAGE 00.2 — قاعدة الحالة SQLite + محرك الذاكرة المؤقتة (Cache Engine)

**الهدف:** قابلية إيقاف واستئناف حقيقية. كل مرحلة لكل ملف تُسجَّل في قاعدة `work/state.db` مع:

- **مفتاح ذاكرة مؤقتة صالح** = `SHA-256(معرف المصدر + المرحلة + هاش إعدادات المرحلة + PIPELINE_VERSION + إصدارات المكتبات المستخدمة)` — أي تغيير مؤثر **يبطل النتيجة القديمة تلقائيًا** (لا يكفي وجود الملف).
- **تحقق صلاحية عند القراءة**: النتيجة المخزنة تُستخدم فقط إذا كان المخرج موجودًا وبصمته SHA-256 مطابقة. مخرج مفقود أو متغير = إعادة معالجة.
- **الإيداعية (Idempotent)**: إعادة تشغيل مرحلة منجزة بنفس المفتاح لا تُعيد العمل.

**مخرجات:** `STATE` (كائن `StateDB`) و`StageCache` و`stage_config_key()`.

In [ ]:
# -*- coding: utf-8 -*-
"""STAGE 00.2 — قاعدة الحالة SQLite + محرك ذاكرة مؤقتة بمفاتيح صالحة."""
import json as _json
import sqlite3 as _sq

class StateDB:
    """قاعدة حالة الخط: ملفات، مراحل، مقاييس، قرارات، أخطاء — بدفعة ذرية واحدة لكل مكالمة."""

    def __init__(self, db_path):
        self.path = str(db_path)
        self.conn = _sq.connect(self.path, timeout=60)
        self.conn.executescript("""
        PRAGMA journal_mode=WAL;
        CREATE TABLE IF NOT EXISTS stages(
            fid TEXT, stage TEXT, status TEXT,
            cache_key TEXT, input_sha TEXT, output_path TEXT, output_sha TEXT,
            started TEXT, finished TEXT, duration_s REAL, error TEXT,
            PRIMARY KEY (fid, stage));
        CREATE TABLE IF NOT EXISTS metrics(fid TEXT, stage TEXT, metric TEXT,
            value REAL, PRIMARY KEY (fid, stage, metric));
        CREATE TABLE IF NOT EXISTS decisions(fid TEXT, stage TEXT, decision TEXT,
            reason TEXT, ts TEXT);
        CREATE TABLE IF NOT EXISTS errors(id INTEGER PRIMARY KEY AUTOINCREMENT,
            fid TEXT, stage TEXT, error_class TEXT, severity TEXT, message TEXT, ts TEXT);
        """)
        self.conn.commit()

    # ── المراحل ──
    def stage_cache_ok(self, fid, stage, cache_key, output_paths=(), force=False):
        """هل توجد نتيجة صالحة؟ (status=ok + كل المخرجات المسجلة موجودة فعلاً على القرص)"""
        if force or not CONFIG["pipeline"]["use_cache"]:
            return False
        row = self.conn.execute(
            "SELECT status, output_path FROM stages WHERE fid=? AND stage=? AND cache_key=?",
            (fid, stage, cache_key)).fetchone()
        if not row or row[0] != "ok" or not row[1]:
            return False
        for o in _json.loads(row[1]):
            if not _Path(o).exists():
                return False  # مخرج حُذف أو تلاعب به → إبطال النتيجة
        return True

    def stage_done(self, fid, stage, cache_key, outputs, duration_s, input_sha=""):
        outs = [str(o) for o in outputs]
        self.conn.execute(
            "INSERT OR REPLACE INTO stages VALUES (?,?,?,?,?,?,?,?,?,?,?)",
            (fid, stage, "ok", cache_key, input_sha, _json.dumps(outs), "",
             now_iso(), now_iso(), float(duration_s), None))
        self.conn.commit()

    def stage_skipped(self, fid, stage, cache_key):
        self.conn.execute(
            "INSERT OR REPLACE INTO stages (fid,stage,status,cache_key,started,finished) "
            "VALUES (?,?,?,?,?,?)", (fid, stage, "skipped", cache_key, now_iso(), now_iso()))
        self.conn.commit()

    def stage_failed(self, fid, stage, error_class, message):
        self.conn.execute(
            "INSERT OR REPLACE INTO stages (fid,stage,status,error,started,finished) "
            "VALUES (?,?,?,?,?,?)", (fid, stage, "failed", f"[{error_class}] {message}",
            now_iso(), now_iso()))
        self.conn.execute("INSERT INTO errors (fid,stage,error_class,severity,message,ts) "
                           "VALUES (?,?,?,?,?,?)",
                           (fid, stage, error_class, "error", message[:2000], now_iso()))
        self.conn.commit()

    def log_error(self, fid, stage, error_class, severity, message):
        self.conn.execute("INSERT INTO errors (fid,stage,error_class,severity,message,ts) "
                          "VALUES (?,?,?,?,?,?)",
                          (fid, stage, error_class, severity, message[:2000], now_iso()))
        self.conn.commit()

    def metric(self, fid, stage, metric, value):
        try:
            val = float(value)
        except (TypeError, ValueError):
            return
        self.conn.execute("INSERT OR REPLACE INTO metrics VALUES (?,?,?,?)",
                          (fid, stage, metric, val))
        self.conn.commit()

    def decision(self, fid, stage, decision, reason):
        self.conn.execute("INSERT INTO decisions VALUES (?,?,?,?,?)",
                          (fid, stage, decision, reason, now_iso()))
        self.conn.commit()

    def stage_config_key(self, stage, cfg_fragment):
        """مفتاح صالح: مصدر + مرحلة + إعدادات + إصدار خط + إصدارات المكتبات."""
        return sha256_bytes(_json.dumps(
            {"stage": stage, "cfg": cfg_fragment, "v": PIPELINE_VERSION,
             "libs": LIB_VERSIONS}, sort_keys=True, default=str).encode())[:24]

LIB_VERSIONS = {}   # تُملأ في STAGE 01 بعد اكتشاف البيئة — التعريف المبكر يضمن أمان الترتيب

STATE = StateDB(PATHS["work"] / "state.db")

### الطبقة الثانية للدولة: مفاتيح ذاكرة مؤقتة صالحة + مشغّل المراحل `run_stage_cached` — أي تغيير في الإعدادات/الإصدارات يبطل النتيجة القديمة تلقائيًا، والفشل يُصنّف لا يُخفى.

In [ ]:
import json as _j

def stage_cache_key(stage, cfg_fragment):
    return STATE.stage_config_key(stage, cfg_fragment)

def run_stage_cached(fid, stage, cfg_fragment, fn, outputs, force=False):
    """نفّذ مرحلة مع ذاكرة مؤقتة صالحة + معايرة الأداء.
    
    fn: كالِبل يعيد dict من المخرجات الإضافية (يُعاد كما هو عند الصلاحية المخزنة None)
    outputs: مسارات المخرجات التي تُفحص وجودها للصلاحية.
    """
    key = stage_cache_key(stage, cfg_fragment)
    t0 = _time.time()
    if STATE.stage_cache_ok(fid, stage, key, outputs, force=force):
        STATE.stage_skipped(fid, stage, key)
        LOGGER.debug("cache-hit %s/%s", fid, stage)
        return {"status": "cached", "duration_s": 0.0}
    try:
        result = fn() or {}
        STATE.stage_done(fid, stage, key, outputs, _time.time() - t0)
        return {"status": "ok", "duration_s": _time.time() - t0, **result}
    except Exception as e:  # تصنيف معروف الصنف — لا نجاح صامت
        STATE.stage_failed(fid, stage, type(e).__name__, str(e))
        raise


def _jsonify(obj):
    """تحويل تكراري إلى JSON-safe (numpy → قيم/قوائم خام)."""
    import numpy as _npd
    if isinstance(obj, dict):
        return {str(k): _jsonify(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_jsonify(v) for v in obj]
    if isinstance(obj, _npd.ndarray):
        return obj.tolist()
    if isinstance(obj, (_npd.floating, _npd.integer)):
        return obj.item()
    if isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj
    return str(obj)

def _numpyfy(obj):
    """إرجاع numpy float32 للأهداف والقوائم المتداخلة (بصمات/أبعاد)."""
    import numpy as _npd
    if isinstance(obj, list):
        try:
            return _npd.asarray(obj, dtype=_npd.float32)
        except (ValueError, TypeError):
            return [_numpyfy(v) for v in obj]
    return obj

def file_cache(stage: str, fid: str, cfg_fragment, compute, *, decode=None):
    """ذاكرة مؤقتة JSON على مستوى الملف للمراحل الثقيلة (خارج decode أيضًا).

    المفتاح: المرحلة + fid (متضمنًا بصمة المصدر) + هاش إعدادات المرحلة +
    إصدار الخط + إصدارات المكتبات — أي تغيير مؤثر يبطل النتيجة تلقائيًا.
    decode: محوّل اختياري (مثل _numpyfy للبصمات).
    """
    key = sha256_bytes(_j.dumps(
        {"stage": stage, "fid": fid, "cfg": cfg_fragment, "v": PIPELINE_VERSION,
         "libs": LIB_VERSIONS}, sort_keys=True, default=str).encode())[:24]
    path = PATHS["cache"] / "files" / stage / f"{fid}_{key}.json"
    if CONFIG["pipeline"]["use_cache"] and stage not in CONFIG["pipeline"]["force_reprocess"] \
            and path.exists():
        try:
            data = _j.loads(path.read_text(encoding="utf-8"))
            return decode(data) if decode else data
        except Exception as e:
            LOGGER.debug("cache-read فشل %s: %s — إعادة حساب", path.name, e)
    result = compute()
    try:
        path.parent.mkdir(parents=True, exist_ok=True)
        tmp = path.with_suffix(path.suffix + f".tmp{_os.getpid()}")
        tmp.write_text(_j.dumps(_jsonify(result), ensure_ascii=False), encoding="utf-8")
        _os.replace(tmp, path)
    except Exception as e:
        LOGGER.debug("cache-write فشل %s: %s", path.name, e)
    return result

### STAGE 00.3 — مُدقّق الإعدادات `validate_config()`

**الهدف:** فشل مبكر (Fail-Fast) — أي قيمة غير صالحة أو تعارض يُكشف **قبل بدء أي معالجة**، لا في منتصف ساعة معالجة.

**ما يُفحص:** أنماط `rvc_profile` المقبولة، `bit_depth` الموصول بالتصدير، تراتب حدود التقطيع (`drop_below ≤ min ≤ max`)، عتبات VAD المنطقية (الهستيريسيس: `neg_threshold < threshold`)، نطاقات العتبات 0-1، مسارات صالحة، وقيم السياسات المعروفة.

**مخرجات:** `VALIDATION_ISSUES` — قائمة فارغة تعني `CONFIG` صالحًا، وإلا يفشل التنفيذ فورًا.

In [ ]:
# -*- coding: utf-8 -*-
"""STAGE 00.3 — مُدقّق CONFIG: كل شيء يفشل مبكرًا لا متأخرًا."""

RVC_PROFILES = {  # طبقة توافق RVC — Sample Rate الهدف لكل نمط توليفات
    "rvc_v2_48k": {"target_sr": 48000, "note": "RVC v2 — توليفات 48kHz (الأعلى جودة)"},
    "rvc_v2_40k": {"target_sr": 40000, "note": "RVC v2 — توليفات 40kHz"},
    "rvc_v2_32k": {"target_sr": 32000, "note": "RVC v2 — توليفات 32kHz"},
}

def validate_config(cfg=None) -> list:
    """يعيد قائمة مشاكل نصية. فارغة = صالح."""
    C = cfg or CONFIG
    issues = []

    # A. INPUT
    d = C["input"]["download"]
    if d["retries"] < 0: issues.append("input.download.retries سالب")
    if d["timeout_s"] < 10: issues.append("input.download.timeout_s قصير جدًا (<10s)")
    if d["chunk_mb"] < 1: issues.append("input.download.chunk_mb < 1")

    # C. AUDIO
    if C["audio"]["rvc_profile"] not in RVC_PROFILES:
        issues.append(f"audio.rvc_profile غير معروف: {C['audio']['rvc_profile']}")
    if C["audio"]["bit_depth"] not in (16, 24):
        issues.append("audio.bit_depth يجب أن يكون 16 أو 24 (PCM)")
    if C["audio"]["channels"] != 1:
        issues.append("audio.channels يجب أن يكون 1 — RVC يتدرب Mono")
    if not (-6.0 <= C["audio"]["true_peak_ceiling_db"] <= -0.1):
        issues.append("audio.true_peak_ceiling_db خارج [-6, -0.1]")
    L = C["audio"]["loudness"]
    if L["mode"] not in ("none", "gain_only"):
        issues.append("audio.loudness.mode غير معروف")
    if L["mode"] == "gain_only" and not L["target_lufs"]:
        issues.append("loudness.mode=gain_only يتطلب target_lufs")

    # H. VAD — Hysteresis منطقي
    V = C["vad"]
    if not (0.0 < V["threshold"] < 1.0): issues.append("vad.threshold خارج (0,1)")
    if not (0.0 <= V["neg_threshold"] < V["threshold"]):
        issues.append("vad.neg_threshold يجب أن يكون أقل من threshold (Hysteresis)")
    if V["min_speech_ms"] > V["min_silence_ms"]:
        issues.append("vad.min_speech_ms > min_silence_ms — غير منطقي")
    if V["max_speech_s"] < 2.0: issues.append("vad.max_speech_s قصير جدًا")

    # H2. SEGMENTATION — تراتب صارم موصول فعليًا بـ STAGE 16
    S = C["segmentation"]
    if not (0 < S["drop_below_s"] <= S["min_clip_s"] <= S["max_clip_s"]):
        issues.append("segmentation: يجب drop_below_s <= min_clip_s <= max_clip_s")
    if S["max_clip_s"] > 30.0:
        issues.append("segmentation.max_clip_s > 30s — خارج نطاق RVC المريح")
    if S["merge_gap_s"] < 0: issues.append("segmentation.merge_gap_s سالب")
    if S["leading_silence_max_s"] < 0 or S["trailing_silence_max_s"] < 0:
        issues.append("segmentation: سماحيات الصمت سالبة")

    # D. SPEAKER
    SP = C["speaker"]
    if SP["match_threshold"] != "auto" and not (0.5 <= float(SP["match_threshold"]) < 1.0):
        issues.append("speaker.match_threshold يجب 'auto' أو رقمًا في [0.5, 1)")
    if SP["min_speech_for_embedding_s"] < 1.0:
        issues.append("speaker.min_speech_for_embedding_s قصير جدًا لـ ECAPA")

    # E/F/G — السياسات المعروفة فقط
    if C["noise"]["policy"] not in ("conditional", "off"):
        issues.append("noise.policy يجب conditional أو off — التنقية الإجبارية ممنوعة")
    if C["music"]["policy"] not in ("conditional", "off"):
        issues.append("music.policy يجب conditional أو off")
    if C["dereverb"]["policy"] not in ("conditional", "off"):
        issues.append("dereverb.policy يجب conditional أو off")
    for name in C["noise"]["branches"]:
        if name not in ("uvr_denoise", "noisereduce", "deepfilternet", "mossformer2", "resemble"):
            issues.append(f"noise.branches: فرع غير معروف {name}")

    # J. QUALITY
    Q = C["quality"]
    if not (1.0 <= Q["min_file_dnsmos_ovrl"] <= 5.0):
        issues.append("quality.min_file_dnsmos_ovrl خارج [1,5]")
    if not (0.05 <= Q["degradation_review"] <= Q["degradation_reject"] <= 1.5):
        issues.append("quality: يجب 0.05 <= degradation_review <= degradation_reject")

    # L. PIPELINE
    if C["pipeline"]["stages"] != "all":
        if not isinstance(C["pipeline"]["stages"], (list, tuple)):
            issues.append("pipeline.stages يجب 'all' أو قائمة أرقام")

    return issues

VALIDATION_ISSUES = validate_config()
if VALIDATION_ISSUES:
    for _i in VALIDATION_ISSUES:
        LOGGER.error("CONFIG غير صالح: %s", _i)
    raise DataIntegrityError(f"CONFIG فشل في التحقق: {len(VALIDATION_ISSUES)} مشكلة — صلحها قبل المتابعة")

TARGET_SR = RVC_PROFILES[CONFIG["audio"]["rvc_profile"]]["target_sr"]
LOGGER.info("CONFIG صالح ✓ — النمط: %s (target_sr=%d Hz, bit-depth=%d)",
            CONFIG["audio"]["rvc_profile"], TARGET_SR, CONFIG["audio"]["bit_depth"])

## STAGE 01 — Environment & Dependency Intelligence

**الهدف:** كشف القدرات **الفعلية** بالقياس لا بالتخمين من اسم البيئة، ثم مدير اعتماديات ذكي:

- **الطبقات:** `core` (تحليل DSP أساسي) / `vad` / `speaker` / `enhance` / `quality` / `separation` — تُثبَّت الناقص فقط عند الحاجة، **ولا يُعاد تثبيت PyTorch أبدًا** إن كان موجودًا (حماية توافق CUDA في Colab).
- **الاختبار بعد التثبيت:** استيراد فعلي + إصدار + توافق + `pip check` — لا افتراض أن التثبيت نجح.
- **النتيجة:** جدول قدرات `CAPS` + `LIB_VERSIONS` الذي يدخل في مفاتيح الـ Cache (تغيير مكتبة ⇒ بطلان نتائج المراحل المعتمدة عليها).

**مخرجات:** `CAPS` (قدرات فعلية)، `LIB_VERSIONS`، وتقرير بيئة.

> **ملاحظة:** النماذج الثقيلة الاختيارية (DeepFilterNet / ClearerVoice / resemble-enhance / WhisperX) لا تُثبَّت تلقائيًا أبدًا — تُكتشف إن وُجدت، وإلا تعمل الفروع المتاحة ويتوثّق الغياب (الشفافية قبل الافتراض).

In [ ]:
# -*- coding: utf-8 -*-
"""STAGE 01 — كشف البيئة الفعلي + إدارة الاعتماديات الذكية."""
import importlib as _il
import platform as _platform
import shutil as _sh
import subprocess as _sp
import sys as _sys

def _pkg_version(mod_name):
    try:
        m = _il.import_module(mod_name)
        return getattr(m, "__version__", "ok")
    except Exception:
        return None

def detect_environment() -> dict:
    """قياس فعلي للقدرات — لا استنتاج من أسماء البيئات."""
    env = {
        "python": _sys.version.split()[0],
        "os": f"{_platform.system()} {_platform.release()}",
        "machine": _platform.machine(),
        "cpu_count": _os.cpu_count(),
        "ram_gb": None,
        "cwd": str(NB_DIR),
        "in_colab": False,
        "ffmpeg": None, "ffprobe": None,
        "cuda": False, "gpu_name": None, "gpu_vram_gb": None,
    }
    try:  # RAM
        with open("/proc/meminfo") as f:
            for line in f:
                if line.startswith("MemTotal"):
                    env["ram_gb"] = round(int(line.split()[1]) / 1e6, 1)
                    break
    except Exception:
        pass
    env["in_colab"] = "google.colab" in _sys.modules or bool(_os.environ.get("COLAB_RELEASE_TAG"))
    for tool in ("ffmpeg", "ffprobe"):
        p = _sh.which(tool)
        env[tool] = str(p) if p else None
    try:  # GPU عبر torch فقط (لا nvidia-smي إلا إذا وُجد)
        import torch
        env["torch"] = torch.__version__
        if torch.cuda.is_available():
            env["cuda"] = True
            env["gpu_name"] = torch.cuda.get_device_name(0)
            env["gpu_vram_gb"] = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    except ImportError:
        env["torch"] = None
    env["disk_free_gb"] = round(_sh.disk_usage(str(NB_DIR)).free / 1e9, 1)
    return env

ENV = detect_environment()

### تنفيذ كشف البيئة

`ENV` يُقاس فعليًا (Python/OS/CPU/RAM/GPU عبر اختبار CUDA حقيقي/ffmpeg/مساحة القرص) — لا استنتاج من اسم البيئة.

In [ ]:
def select_device(pref: str) -> str:
    """auto → cuda إن توفرت فعليًا ووُجدت torch، وإلا cpu."""
    if pref == "cpu":
        return "cpu"
    try:
        import torch
        if pref == "cuda" or (pref == "auto" and torch.cuda.is_available()):
            # اختبار فعلي لا اسمي
            _ = torch.zeros(1).to("cuda")
            return "cuda"
    except Exception:
        pass
    return "cpu"

DEVICE = select_device(CONFIG["runtime"]["device"])

# ---- تعريف الاعتماديات بالطبقات (تثبيت عند الحاجة فقط) ----
DEPS = {
    "core":      {"mods": ["numpy", "pandas", "scipy", "librosa", "soundfile",
                            "matplotlib", "tqdm", "soxr"], "pips": []},
    "vad":       {"mods": ["silero_vad"], "pips": ["silero-vad"], "needs": "torch"},
    "speaker":   {"mods": ["speechbrain"], "pips": ["speechbrain"], "needs": "torch"},
    "enhance":   {"mods": ["noisereduce"], "pips": ["noisereduce"]},
    "quality":   {"mods": ["onnxruntime"], "pips": ["onnxruntime"]},
    "separation": {"mods": ["audio_separator"], "pips": ["audio-separator[cpu]"]},
    # اختيارية ثقيلة — لا تُثبّت تلقائيًا أبدًا:
    "optional_heavy": {
        "deepfilternet": {"mods": ["DF"], "pips": ["DeepFilterNet"]},
        "clearervoice": {"mods": ["clearvoice"], "pips": ["clearervoice"]},
        "resemble": {"mods": ["resemble_enhance"], "pips": ["resemble-enhance"]},
        "whisperx": {"mods": ["whisperx"], "pips": ["whisperx"]},
    },
}

def ensure_torch():
    """torch موجودة في Colab دائمًا تقريبًا. محليًا: عجلة CPU فقط — لا نكسر شيء."""
    if _pkg_version("torch"):
        return True
    if ENV["in_colab"]:
        return False  # غريب — لكن لا نعبث ببيئة Colab
    try:
        _sp.run([_sys.executable, "-m", "pip", "install", "-q", "--no-input",
                 "torch", "torchaudio", "--index-url",
                 "https://download.pytorch.org/whl/cpu"], check=True, timeout=1800)
        return True
    except Exception as e:
        LOGGER.warning("لم تُثبّت torch: %s — ميزات GPU/النماذج العصبية ستُعطّل", e)
        return False

def install_tier(tier: str, auto: bool = True) -> bool:
    """تثبيت طبقة عند نقصها. يعيد True إن أصبحت متاحة."""
    spec = DEPS[tier]
    missing = [m for m in spec["mods"] if _pkg_version(m) is None]
    if not missing:
        return True
    if spec.get("needs") == "torch" and not ensure_torch():
        return False
    if not (auto and spec.get("pips")):
        return False
    LOGGER.info("تثبيت طبقة '%s': %s", tier, spec["pips"])
    try:
        _sp.run([_sys.executable, "-m", "pip", "install", "-q", "--no-input",
                 *spec["pips"]], check=True, timeout=1800)
    except Exception as e:
        LOGGER.warning("فشل تثبيت %s: %s — تعمل البدائل", tier, e)
        return False
    return all(_pkg_version(m) for m in spec["mods"])

### إطار الاعتماديات الذكي وتنفيذه

تعريف الأجهزة والاعتماديات بالطبقات مع حماية torch الموجودة (لا كسر CUDA)، ثم تثبيت الناقص عند الحاجة فقط وفحص المكونات الاختيارية الثقيلة.

In [ ]:
# ---- تنفيذ السياسة: core إلزامي، والباقي عند الحاجة ----
AUTO_INSTALL = True  # Colab/محلي: الطبقات الخفيفة تُثبّت تلقائيًا (لا تشمل الثقيلة)
for _tier in ("core", "vad", "speaker", "enhance", "quality", "separation"):
    _ok = install_tier(_tier, auto=AUTO_INSTALL)
    if not _ok:
        LOGGER.warning("طبقة '%s' غير متاحة — تعمل البدائل الاحتياطية ويتوثق الغياب", _tier)

### جدول القدرات الفعلية `CAPS`

اكتشاف المكونات الاختيارية الثقيلة (اكتشافًا فقط — لا تثبيتًا) ثم تجميع القدرات الفعلية وإصدارات المكتبات التي تدخل مفاتيح الذاكرة المؤقتة، وتقرير البيئة المختصر.

In [ ]:
# فحص الكتل الاختيارية الثقيلة (اكتشاف فقط — لا تثبيت)
OPTIONAL_HEAVY = {name: _pkg_version(spec["mods"][0]) for name, spec in DEPS["optional_heavy"].items()}
for _n, _v in OPTIONAL_HEAVY.items():
    if _v:
        LOGGER.info("اكتُشفت مكونة اختيارية مثبّتة مسبقًا: %s (%s)", _n, _v)

CAPS = {
    "device": DEVICE,
    "torch": _pkg_version("torch"),
    "silero_vad": _pkg_version("silero_vad"),
    "speechbrain": _pkg_version("speechbrain"),
    "onnxruntime": _pkg_version("onnxruntime"),
    "noisereduce": _pkg_version("noisereduce"),
    "audio_separator": _pkg_version("audio_separator"),
    "soxr": _pkg_version("soxr"),
    "ffmpeg": bool(ENV["ffmpeg"]),
    "ffprobe": bool(ENV["ffprobe"]),
    "dnsmos_weights": None,  # يُملأ في STAGE 20 عند توفر الأوزان
    **{f"opt_{k}": bool(v) for k, v in OPTIONAL_HEAVY.items()},
}

LIB_VERSIONS.update({k: str(v) for k, v in CAPS.items() if v and k not in ("device",)})

print("── تقرير البيئة " + "─" * 40)
print(f"Python {ENV['python']} | {ENV['os']} | CPU×{ENV['cpu_count']} | RAM {ENV['ram_gb']}GB | "
      f"Disk {ENV['disk_free_gb']}GB free")
print(f"الجهاز المختار: {DEVICE}" + (f" ({ENV['gpu_name']}, {ENV['gpu_vram_gb']}GB VRAM)" if DEVICE == "cuda" else ""))
print(f"ffmpeg: {'✓' if ENV['ffmpeg'] else '✗'} | ffprobe: {'✓' if ENV['ffprobe'] else '✗'}")
print("القدرات:", ", ".join(f"{k}={'✓' if v else '✗'}" for k, v in CAPS.items() if isinstance(v, (bool, str)) and k not in ("device",)))
if not CAPS["silero_vad"]:
    print("⚠ غياب Silero VAD → البديل الطاقي الاحتياطي سيعمل (أقل دقة للكلام الهادئ)")
if not CAPS["speechbrain"]:
    print("⚠ غياب SpeechBrain → التحقق من هوية المتحدث يُعطّل (بدون بدائل متكافئة)")
if not CAPS["onnxruntime"]:
    print("⚠ غياب onnxruntime → DNSMOS يُعطّل (قرارات الجودة تعتمد على القياسات الطيفية)")

## STAGE 02 — Source Acquisition & Archive Security

**الهدف:** جلب الأرشيف بطريقة إنتاجية (تدفق + استئناف + إعادة محاولة أُسّية + تحقق حجم/هاش + كتابة ذرية)، ثم **استخراج آمن** ضد كل هجمات الأرشيف المعروفة.

**ترتيب المصادر:** (1) أرشيف محلي صريح `input.local_archive` ← (2) اسم يدوي بجانب الدفتر `input.manual_names` ← (3) تنزيل `input.archive_url`. روابط Google Drive المؤقتة تنتهي — الدفتر لا "ينهار بصمت": يشرح الحل (التنزيل اليدوي) ويستمر إن وُجد الملف.

**أمان الاستخراج (يُوثَّق كل قرار):**

| الهجوم / الحالة | الدفاع |
|---|---|
| ZIP-Slip / Path Traversal | رفض أي مسار عضو يخرج عن مجلد الاستخراج (`resolve` + فحص الأصل) |
| ZIP Bomb | نسبة ضغط > 200× أو محتوى متوقع > 20GB → رفض |
| أعضاء مكررون | الاحتفاظ بالأول + توثيق الاستبعاد (لا فقد بلا أثر) |
| أعضاء مشفرون | استبعاد موثّق (لا فك عنيف) |
| أحجام شاذة | حد `input.max_file_mb` لكل ملف صوتي |
| ملفات غير صوتية | تُفصل عن الصوتيات ولا تُفقد (مجلد `_ignored`) |

**مخرجات:** `ARCHIVE_PATH`, `RAW_FILES` (قائمة مسارات الصوت الخام), `EXTRACT_REPORT` (كل عضو: مستخرج/مستبعد + سبب).

In [ ]:
# -*- coding: utf-8 -*-
"""STAGE 02 — تنزيل قوي + استخراج آمن منظم بالتقارير."""
import zipfile as _zipfile
import urllib.request as _urlreq
import urllib.error as _urlerr

def robust_download(url: str, dest: _Path, *, expected_sha256: str = "",
                    min_mb: float = 1.0, retries: int = 4, timeout_s: int = 120,
                    backoff_s: float = 3.0, chunk_mb: int = 8) -> _Path:
    """تنزيل تدفقي ذري: tmp → تحقق (حجم/هاش/نوع محتوى) → rename.

    فشل مؤقت → إعادة محاولة بتراجع أُسّي (يحترم Retry-After).
    """
    D = CONFIG["input"]["download"]
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > min_mb * 1e6:
        LOGGER.info("الأرشيف موجود مسبقًا: %s (%.1f MB) — تخطي التنزيل", dest.name, dest.stat().st_size / 1e6)
        return dest
    tmp = dest.with_suffix(dest.suffix + ".part")
    last_err = None
    for attempt in range(1, max(1, retries) + 1):
        try:
            req = _urlreq.Request(url, headers={"User-Agent": "Mozilla/5.0 (pipeline)"})
            with _urlreq.urlopen(req, timeout=timeout_s) as r, open(tmp, "wb") as f:
                total = int(r.headers.get("Content-Length") or 0)
                got = 0
                while True:
                    chunk = r.read(chunk_mb * (1 << 20))
                    if not chunk:
                        break
                    f.write(chunk); got += len(chunk)
            if got < min_mb * 1e6:
                raise DataIntegrityError(f"حجم منزّل صغير جدًا: {got/1e6:.2f}MB < {min_mb}MB (رابط منتهٍ محتمل)")
            if expected_sha256:
                actual = sha256_file(tmp)
                if actual != expected_sha256:
                    raise DataIntegrityError(f"SHA-256 غير مطابق: {actual[:16]}…")
            _os.replace(tmp, dest)
            LOGGER.info("نُزّل %s (%.1f MB)", dest.name, got / 1e6)
            return dest
        except (_urlerr.URLError, _urlerr.HTTPError, TimeoutError, ConnectionError, DataIntegrityError) as e:
            last_err = e
            LOGGER.warning("محاولة تنزيل %d/%d فشلت: %s", attempt, retries, e)
            _time.sleep(min(backoff_s * (2 ** (attempt - 1)), 60))
        except Exception as e:
            raise PipelineError(f"فشل تنزيل غير متوقع: {e}") from e
    raise PipelineError(
        f"تعذر تنزيل الأرشيف بعد {retries} محاولات: {last_err}\n"
        f"→ الحل: نزّل الأرشيف يدويًا وضعه بجانب الدفتر باسم من: {CONFIG['input']['manual_names']}")

def find_local_archive() -> _Path | None:
    """الأولوية: مسار صريح ← أسماء يدوية بجانب الدفتر ← مجلد الأرشيفات."""
    explicit = CONFIG["input"]["local_archive"]
    if explicit and _Path(explicit).exists():
        return _Path(explicit)
    for name in CONFIG["input"]["manual_names"]:
        for base in (NB_DIR, PATHS["archive"]):
            cand = base / name
            if cand.exists() and cand.stat().st_size > 0:
                return cand
    zips = sorted(PATHS["archive"].glob("*.zip"))
    return zips[0] if zips else None

ARCHIVE_PATH = find_local_archive()
if ARCHIVE_PATH is None:
    D = CONFIG["input"]["download"]
    ARCHIVE_PATH = robust_download(
        CONFIG["input"]["archive_url"], PATHS["archive"] / "Voice.zip",
        expected_sha256=D["sha256"], min_mb=D["min_zip_mb"], retries=D["retries"],
        timeout_s=D["timeout_s"], backoff_s=D["backoff_s"], chunk_mb=D["chunk_mb"])

ARCHIVE_SHA = sha256_file(ARCHIVE_PATH)
LOGGER.info("الأرشيف: %s | %.1f MB | sha256=%s…", ARCHIVE_PATH.name,
            ARCHIVE_PATH.stat().st_size / 1e6, ARCHIVE_SHA[:16])

### تنفيذ الاستخراج الآمن وتجميع الملفات الصوتية

استخراج محصّن ضد ZIP-Slip/القنابل/الأسماء المكررة/المشفرة، مع فصل غير الصوتي إلى `data/_ignored/` (لا فقد) وجمع `RAW_FILES` النهائية.

In [ ]:
# -*- coding: utf-8 -*-
"""STAGE 02.1 — استخراج آمن: ZIP-Slip / Bombs / أعضاء مكررون / مشفرون / غير صوتيين."""

SECURE_ZIP_LIMITS = {
    "max_members": 20000,          # عدد أعضاء معقول لأرشيف صوتي
    "max_total_gb": 20,            # حجم مفكوك معقول
    "max_compression_ratio": 200,  # نسبة ضغط فوقها = قنبلة محتملة
}

def secure_extract(zip_path: _Path, out_dir: _Path, allowed_exts) -> dict:
    """استخراج محصّن. يعيد تقريرًا لكل عضو: kept / skipped + سبب. يحفظ مسار العضو الأصلي."""
    out_dir.mkdir(parents=True, exist_ok=True)
    ignored_dir = out_dir.parent / "_ignored"
    L = SECURE_ZIP_LIMITS
    report = {"kept": [], "skipped": [], "audio": [], "errors": []}
    with _zipfile.ZipFile(zip_path) as zf:
        infos = zf.infolist()
        if len(infos) > L["max_members"]:
            raise DataIntegrityError(f"عدد أعضاء ZIP شاذ: {len(infos)}")
        seen_names = set()
        total_out = 0
        for info in infos:
            name = info.filename
            if info.is_dir():
                continue
            # 1) ZIP-Slip: المسار المُحلَّل يجب أن يبقى داخل out_dir
            target = (out_dir / name).resolve()
            if not str(target).startswith(str(out_dir.resolve())):
                report["skipped"].append({"member": name, "reason": "path_traversal"})
                continue
            # 2) أسماء مكررة: الأول يفوز، والباقي يوثَّق ولا يُفقد بصمت
            if name.lower() in seen_names:
                report["skipped"].append({"member": name, "reason": "duplicate_member_name"})
                continue
            seen_names.add(name.lower())
            # 3) تشفير
            if info.flag_bits & 0x1:
                report["skipped"].append({"member": name, "reason": "encrypted"})
                continue
            # 4) قنبلة ضغط
            usize, csize = info.file_size, max(info.compress_size, 1)
            if usize / csize > L["max_compression_ratio"] or usize > L["max_total_gb"] * 1e9:
                report["skipped"].append({"member": name, "reason": f"zip_bomb_suspect ratio={usize/csize:.0f}x"})
                continue
            total_out += usize
            if total_out > L["max_total_gb"] * 1e9:
                raise DataIntegrityError("المجموع المفكوك تجاوز الحد الآمن")
            ext = _Path(name).suffix.lower()
            if ext in allowed_exts and usize <= CONFIG["input"]["max_file_mb"] * 1e6:
                target.parent.mkdir(parents=True, exist_ok=True)
                with zf.open(info) as src, open(target, "wb") as dst:
                    _shutil.copyfileobj(src, dst)
                report["kept"].append({"member": name, "out": str(target), "size": usize})
            else:
                # ملف غير صوتي أو ضخم: يُفصل بلا فقد
                ignored_dir.mkdir(parents=True, exist_ok=True)
                alt = ignored_dir / _Path(name).name
                with zf.open(info) as src, open(alt, "wb") as dst:
                    _shutil.copyfileobj(src, dst)
                report["skipped"].append({
                    "member": name, "reason": "non_audio_or_oversize",
                    "moved_to": str(alt)})
    report["audio"] = sorted(
        [k["out"] for k in report["kept"]], key=str)
    return report

try:
    EXTRACT_REPORT = secure_extract(ARCHIVE_PATH, PATHS["raw"],
                                    set(CONFIG["input"]["extensions"]))
except _zipfile.BadZipFile as e:
    raise InvalidAudioError(f"أرشيف ZIP تالف: {ARCHIVE_PATH.name}: {e}") from e
except DataIntegrityError as e:
    raise DataIntegrityError(f"الأرشيف رُفض لأسباب أمنية: {e}") from e

# تجنّب الاحتواء الذاتي: إن كان الأرشيف مستخرَجًا مسبقًا داخل raw/ فلا نستخرج مجددًا داخل raw/raw
RAW_FILES = []
for p in sorted(PATHS["raw"].rglob("*")):
    if p.is_file() and p.suffix.lower() in CONFIG["input"]["extensions"] and not p.name.endswith(".part"):
        RAW_FILES.append(p)

LOGGER.info("استخراج: %d صوتي | %d مستبعد (%s)", len(RAW_FILES), len(EXTRACT_REPORT["skipped"]),
            "; ".join({s["reason"] for s in EXTRACT_REPORT["skipped"]}) or "لا استبعادات")
print(f"RAW_FILES = {len(RAW_FILES)} ملف صوتي جاهز للجرد")

## STAGE 03 — Audio Inventory & Metadata Extraction

**الهدف:** جرد دقيق لكل ملف عبر **ffprobe الصادم** (لا المكتبات المتساهلة) + معلومات الحاوية والترميز.

**معرفات مستقرة (Stable IDs):** لا ترقيم تسلسلي يتغير بتغير ترتيب الملفات — المعرف يُشتق من `SHA-256(مسار العضو الأصلي داخل الأرشيف + بصمة المصدر)`:
- نفس الملف بنفس المسار = نفس المعرف **دائمًا** عبر إعادة التشغيل،
- معالجة تصادمية شبه مستحيلة (12 خانة هاش) — وأي تصادم يفحص صراحةً.

**لكل ملف يُستخرج:** الامتداد، الحجم، الحاوية، الترميز (codec)، معدل العينة، القنوات، Bit Depth، المدة، Bitrate، بصمة SHA-256 الكاملة، ومسار العضو الأصلي داخل الأرشيف (Provenance).

**مخرجات:** `INVENTORY` (DataFrame) + `reports/inventory.csv` — كل ملف له `fid` مستقر.

In [ ]:
# -*- coding: utf-8 -*-
"""STAGE 03 — جرد ffprobe دقيق + معرفات مستقرة قائمة على المحتوى."""
import soundfile as sf
import json as _json

def ffprobe_info(path) -> dict:
    """استخراج بيانات ffprobe الفعلية — المصدر الأوثق للحاويات (مع fallback لـ soundfile)."""
    if not ENV["ffprobe"]:
        try:  # بديل soundfile عند غياب ffprobe (يوثّق الحد: بيانات حاوية أقل)
            info = sf.info(str(path))
            return {"container": Path(path).suffix.lstrip("."), "codec": info.format,
                    "sr": info.samplerate, "channels": info.channels,
                    "bit_depth": info.subtype or "", "duration_s": info.frames / info.samplerate,
                    "bitrate_kbps": round(Path(path).stat().st_size * 8 / max(info.frames / info.samplerate, 1e-9) / 1000),
                    "lossless": "pcm" in (info.subtype or ""), "probe_warning": "soundfile_fallback"}
        except Exception as e:
            return {"error": f"ffprobe مفقود وsoundfile فشل: {e}"}
    r = _sp.run([ENV["ffprobe"] or "ffprobe", "-v", "error", "-print_format", "json",
                 "-show_format", "-show_streams", str(path)],
                capture_output=True, text=True, timeout=60)
    if r.returncode != 0:
        return {"error": r.stderr.strip()[:200]}
    data = _json.loads(r.stdout or "{}")
    streams = [s for s in data.get("streams", []) if s.get("codec_type") == "audio"]
    fmt = data.get("format", {})
    a = streams[0] if streams else {}
    return {
        "container": fmt.get("format_name", ""),
        "codec": a.get("codec_name", ""),
        "sr": int(a["sample_rate"]) if a.get("sample_rate") else None,
        "channels": a.get("channels"),
        "bit_depth": a.get("bits_per_sample") or a.get("bits_per_raw_sample"),
        "duration_s": float(fmt.get("duration") or a.get("duration") or 0),
        "bitrate_kbps": round(int(fmt.get("bit_rate") or 0) / 1000),
        "lossless": (a.get("codec_name") in ("flac", "alac", "pcm_s16le", "pcm_s24le", "aiff")),
        "probe_warning": "" if streams else "no_audio_stream",
    }

def stable_fid(source_sha: str, rel_member: str) -> str:
    """معرف مستقر قائم على المحتوى + المسار الأصلي — لا يتغير بترتيب الملفات."""
    return "F" + sha256_bytes(f"{rel_member}::{source_sha}".encode())[:12]

def build_inventory(files) -> "list[dict]":
    rows = []
    for p in files:
        p = _Path(p)
        sha = sha256_file(p)
        rel = str(p.relative_to(PATHS["raw"]))
        row = {
            "fid": stable_fid(sha, rel),
            "member_path": rel,
            "orig_name": p.name,
            "path": str(p),
            "file_sha256": sha,
            "size_mb": round(p.stat().st_size / 1e6, 2),
        }
        row.update(ffprobe_info(p))
        rows.append(row)
    return rows

INVENTORY_ROWS = build_inventory(RAW_FILES)
INVENTORY = pd.DataFrame(INVENTORY_ROWS)

if len(INVENTORY) == 0:
    raise DataIntegrityError("لا توجد ملفات صوتية بعد الاستخراج — تحقق من الأرشيف والإعدادات")
if not INVENTORY["fid"].is_unique:
    # تصادم معرفات: نطيل الهاش حتى التفرد (يُوثَّق)
    LOGGER.warning("تصادم معرفات نادر — تمت إطالة المعرفات")
    INVENTORY["fid"] = ["F" + sha256_bytes(f"{r.member_path}::{r.file_sha256}".encode())[:16]
                        for r in INVENTORY.itertuples()]

INVENTORY.to_csv(PATHS["reports"] / "inventory.csv", index=False)
LOGGER.info("الجرد: %d ملفًا | إجمالي المدة %.1f دقيقة",
            len(INVENTORY), INVENTORY["duration_s"].sum() / 60)

_disp = INVENTORY[["fid", "orig_name", "codec", "sr", "channels", "duration_s", "bitrate_kbps", "size_mb"]].copy()
_disp["duration_s"] = _disp["duration_s"].round(1)
try:
    from IPython.display import display as _ipd
    _ipd(_disp)
except ImportError:
    print(_disp.to_string(index=False))

## STAGE 04 — Robust Audio Decoding

**الهدف:** تمثيل موحّد (mono float32) لكل ملف **مع حفظ معدل العينة الأصلي** — لا فرض 48kHz هنا (الفرض يحدث مرة واحدة لاحقًا في STAGE 18 فقط عند الحاجة).

**قرارات التصميم المهمة:**

1. **فك ترميز بمسار واحد متسق:** soundfile أولًا (الأدق WAV/FLAC)، ثم **ffmpeg-PCM32 عند الحاجة** (OGG/OPUS/M4A…) بمعدل العينة الأصلي — المسار الاحتياطي لا يغيّر معدل العينة (إصلاح التناقض التاريخي بين المسارين).
2. **Stereo ذكي لا أعمى:** قبل الدمج نحسب RMS لكل قناة + الارتباط + الطور بين القنوات. إذا كان الارتباط سالبًا بقوة (anti-phase) أو اختلفت القنوات بقوة → نختار أفضل قناة (أعلى RMS سليمًا) بدل متوسط يدمر الإشارة بالإلغاء. القرار يُوثّق لكل ملف.
3. **Decoder QA:** NaN/Inf، طول صفر، مدة تتجاوز ffprobe بفارق >5%، قيم خارج [-1,1] → ملف غير صالح (استثناء صريح لا نجاح صامت).
4. **ذاكرة مؤقتة بصلاحية:** decoded cache بمفتاح بصمة المصدر — الملف لا يُعاد فكّه عند إعادة التشغيل إلا إذا تغيّر المصدر.

**مخرجات:** `decoded/` (WAV FLOAT بمعدل العينة الأصلي) + `REGISTRY` + `reports/diagnostics.csv` أولي.

In [ ]:
# -*- coding: utf-8 -*-
"""STAGE 04 — فك ترميز متسق + تحليل Stereo ذكي + فحص جودة المُفكّك."""
import soundfile as sf
import librosa

DECODE_CACHE = PATHS["work"] / "01_decoded"; DECODE_CACHE.mkdir(parents=True, exist_ok=True)

def ffmpeg_decode(path):
    """فك ترميز ffmpeg الصحيح: معدل العينة الأصلي يُقرأ من ffprobe لا يُفرض."""
    info = ffprobe_info(path)
    sr = info.get("sr") or 16000
    tmp = DECODE_CACHE / (sha256_bytes(str(path).encode())[:16] + "_ff.wav")
    r = _sp.run([ENV["ffmpeg"] or "ffmpeg", "-y", "-loglevel", "error", "-i", str(path),
                 "-map", "a:0", "-ac", "1", "-vn", "-c:a", "pcm_f32le",
                 "-ar", str(sr), str(tmp)],
                capture_output=True, text=True, timeout=600)
    if r.returncode != 0 or not tmp.exists():
        raise InvalidAudioError(f"ffmpeg فشل: {Path(path).name}: {r.stderr[:200]}")
    y, s = sf.read(tmp, dtype="float32", always_2d=False)
    tmp.unlink(missing_ok=True)
    if s != sr:  # ffmpeg أحترمه لكن نتحقق
        y = librosa.resample(y, orig_sr=s, target_sr=sr)
    return y.astype(np.float32), int(sr)

def analyze_stereo(path):
    """تحليل قنوات: RMS لكل قناة + ارتباط + كشف anti-phase (إلغاء محتمل)."""
    try:
        y2, sr = sf.read(str(path), dtype="float32", always_2d=True)
    except Exception:
        return None
    if y2.ndim != 2 or y2.shape[1] < 2:
        return None
    L, R = y2[:, 0], y2[:, 1]
    rms_l, rms_r = float(np.sqrt(np.mean(L ** 2))), float(np.sqrt(np.mean(R ** 2)))
    corr = float(np.corrcoef(L, R)[0, 1]) if (rms_l > 0 and rms_r > 0) else 0.0
    return {"stereo": True, "rms_l": rms_l, "rms_r": rms_r, "corr": corr,
            "anti_phase": corr < -0.8, "identical": corr > 0.9999}

def smart_decode(path):
    """فك موحّد: soundfile ← ffmpeg، مع Stereo ذكي وفحص جودة المُفكّك."""
    stereo = analyze_stereo(path)
    mono_strategy, y = "mono_source", None
    try:  # المسار الأساسي — soundfile
        y_multi, sr = sf.read(str(path), dtype="float32", always_2d=True)
        if y_multi.shape[1] == 2 and stereo is not None:
            corr = stereo["corr"]
            if stereo.get("anti_phase"):
                mono_strategy = "best_channel_anti_phase"
                y = y_multi[:, 0] if stereo["rms_l"] >= stereo["rms_r"] else y_multi[:, 1]
            elif abs(corr) < 0.5:  # قناتان مختلفتان جداً (مثل مقابلة بقناتين)
                mono_strategy = "best_channel_uncorrelated"
                y = y_multi[:, 0] if stereo["rms_l"] >= stereo["rms_r"] else y_multi[:, 1]
            else:
                mono_strategy = "average_channels"
                y = y_multi.mean(axis=1)
        else:
            y = y_multi[:, 0] if y_multi.ndim == 2 else y_multi
    except Exception as e:
        LOGGER.debug("soundfile لم يستطع %s (%s) ← ffmpeg", Path(path).name, e)
        y = None
    if y is None or len(y) == 0:  # المسار الاحتياطي — متسق مع الأساسي (معدل أصلي)
        y, sr = ffmpeg_decode(path)
    # ── Decoder QA: لا يعتبر سليمًا لمجرد أنه قُرئ ──
    if not np.all(np.isfinite(y)):
        raise InvalidAudioError(f"NaN/Inf بعد فك {Path(path).name}")
    if np.max(np.abs(y) if len(y) else 0) > 1.0 + 1e-6:
        LOGGER.warning("قيم خارج [-1,1] في %s — قصّ رقمي في المصدر؟", Path(path).name)
    return y.astype(np.float32), int(sr), mono_strategy

def decode_stage_file(fid, src_path, file_sha, out_dir):
    """فك ملف واحد مع ذاكرة مؤقتة صالحة (بصمة المصدر جزء من المفتاح)."""
    out = out_dir / f"{fid}.wav"
    def _do():
        y, sr, strategy = smart_decode(src_path)
        atomic_write_audio(out, y, sr, subtype="FLOAT")
        return {"sr": sr, "dur_s": len(y) / sr, "strategy": strategy, "y": None}
    res = run_stage_cached(fid, "decode",
                           {"sha": file_sha, "ext": Path(src_path).suffix.lower()},
                           _do, [out], force="decode" in CONFIG["pipeline"]["force_reprocess"])
    if res["status"] == "cached":
        info = sf.info(str(out))
        res.update({"sr": info.samplerate, "dur_s": info.frames / info.samplerate, "strategy": "cached"})
    return out, res

### التنفيذ: Quick Test بعينة طبقية (وليست الأقصر!) ثم فك الترميز لكل ملف مع فحص QA وفشل موثّق لا صامت.

In [ ]:
# ── عينة Quick Test ممثلة طبقيًا (وليس الأقصر!) ──
def stratified_sample(df, n):
    """اختيار ممثلة حسب المدة والجهارة المحتملة — تنوّع حقيقي للعينة."""
    if n >= len(df):
        return df.index.tolist()
    q = pd.qcut(df["duration_s"].rank(method="first"), q=min(n, len(df)),
                labels=False, duplicates="drop")
    idx = list(df.groupby(q).apply(lambda g: g.index[0], include_groups=False))
    if len(idx) < n:
        rest = [i for i in df.index if i not in idx]
        idx += rest[: n - len(idx)]
    return idx[:n]

QT = CONFIG["pipeline"]["quick_test"]
DEC_FILES = INVENTORY.copy()
if QT["enabled"]:
    sel = stratified_sample(DEC_FILES, QT["n_files"])
    DEC_FILES = DEC_FILES.loc[sel].reset_index(drop=True)
    print(f"⚡ Quick Test: عينة طبقية = {len(DEC_FILES)} ملفات (قصّ حتى {QT['max_seconds_per_file']} ث)")

rows = []
for r in tqdm(DEC_FILES.itertuples(), total=len(DEC_FILES), desc="فك الترميز", unit="ملف"):
    try:
        out, res = decode_stage_file(r.fid, r.path, r.file_sha256, DECODE_CACHE)
        dur = res["dur_s"]
        if QT["enabled"] and dur > QT["max_seconds_per_file"]:  # قص العينة فقط (للاختبار)
            y, sr = sf.read(str(out), dtype="float32")
            y = y[: int(sr * QT["max_seconds_per_file"])]
            atomic_write_audio(out, y, sr, subtype="FLOAT")
            dur = len(y) / sr
        rows.append({**{k: getattr(r, k) for k in
                        ("fid", "member_path", "orig_name", "path", "file_sha256",
                         "size_mb", "codec", "sr", "channels", "duration_s", "bitrate_kbps")},
                     "wav_path": str(out), "dec_sr": res["sr"], "dec_dur_s": round(dur, 2),
                     "mono_strategy": res["strategy"], "decode_status": "ok"})
    except (InvalidAudioError, PipelineError) as e:
        LOGGER.error("فشل فك %s: %s", r.orig_name, e)
        rows.append({**{k: getattr(r, k) for k in ("fid", "member_path", "orig_name", "path",
                                                    "file_sha256", "size_mb", "codec", "sr",
                                                    "channels", "duration_s", "bitrate_kbps")},
                     "wav_path": None, "dec_sr": None, "dec_dur_s": 0.0,
                     "mono_strategy": None, "decode_status": f"failed: {e}"})

REGISTRY = pd.DataFrame(rows)
REGISTRY.to_csv(PATHS["reports"] / "decode_report.csv", index=False)
_ok = REGISTRY[REGISTRY["decode_status"] == "ok"]
print(f"فُكّ بنجاح: {len(_ok)}/{len(REGISTRY)} | إجمالي المدة: {_ok['dec_dur_s'].sum()/60:.1f} دقيقة")
if REGISTRY["mono_strategy"].ne("mono_source").any():
    print(REGISTRY.loc[REGISTRY["mono_strategy"].ne("mono_source"),
                        ["orig_name", "mono_strategy"]].to_string(index=False))
if len(_ok) == 0:
    raise DataIntegrityError("لا يوجد ملف واحد صالح لفك الترميز — تحقق من الأرشيف")

## STAGE 05 — Audio Integrity Inspection

**الهدف:** قياسات موضوعية شاملة **قبل أي تعديل** — هذه القياسات تقود كل قرارات المعالجة اللاحقة.

### ما يُقاس (لكل ملف)

**مجال الزمن:** القمة، RMS، Crest Factor، إزاحة DC، نسبة الصمت، **True Peak حقيقي** (بفرط تعيين 4× عبر مرشح متعدد الأطوار — لا Sample Peak مسمّى زورًا)، Dynamic Range.

**تصنيف Clinging المنفصل (إصلاح الخلط التاريخي):**
| الصنف | المعنى | الطريقة |
|---|---|---|
| `source_clipping` | قص في **المصدر نفسه** | عينات مسطحة عند ±(1-ε) متتالية ≥3 قبل أي معالجة |
| `processing_clipping` | قص **أحدثته معالجة** | مقارنة قمم النسخة المعالجة بالمصدر |
| `export_clipping` | قص عند التصدير | فحص الملف المكتوب بعد الكتابة |

**الضوضاء — بصدق:** لا نسمّي المئين العاشر لطاقة الإطارات "SNR". نُقدّر **أرضية الضوضاء** (noise floor) من إطارات **اللاكلام الفعلي** (عبر VAD) — ويوسم الناتج `noise_floor_est` (تقدير) لا "SNR" لأنه بلا مرجع إشارة نظيفة حقيقي (Intrusive metrics مثل PESQ/STOI ممنوعة بلا مرجع — نلتزم شروط المقياس).

**مجال التردد:** Spectral Centroid / Rolloff / Bandwidth / Flatness / Entropy، توزيع طاقة النطاقات (منخفض/كلام/عالي) — تخدم كشف الموسيقى والصدى لاحقًا.

**الكلام:** نسبة النشاط الصوتي، عدد الجمل، متوسط طول الجملة (تخدم مرحلتي VAD والتقطيع).

**مخرجات:** `DIAG` (DataFrame) + `reports/diagnostics.csv` + رسوم توزيعات تشخيصية.

In [ ]:
# -*- coding: utf-8 -*-
"""STAGE 05 — قياسات سلامة الإشارة: زمن/طيف/قص/ضوضاء بصدق تام."""
import librosa as _lr
from scipy import signal as _sig

def true_peak_db(y, sr, oversample=4):
    """True Peak تقديريًا بإعادة تعينة 4× polyphase — التسمية صادقة (ليست Sample Peak)."""
    if len(y) == 0:
        return -np.inf
    y_up = _sig.resample_poly(y, up=oversample, down=1)
    tp = float(np.max(np.abs(y_up)))
    return float(db_to_scalar(tp))

def sample_peak_db(y):
    return float(db_to_scalar(np.max(np.abs(y))) if len(y) else -np.inf)

def rms_db(y):
    return float(db_to_scalar(np.sqrt(np.mean(np.square(y)))) if len(y) else -np.inf)

def detect_source_clipping(y, threshold=0.985, run=3):
    """قص المصدر: عينات مسطحة متتالية قرب القمة (كلا الجهتين)."""
    if len(y) == 0:
        return {"clip_ratio": 0.0, "max_run": 0, "clipped": False}
    a = np.abs(y) >= threshold
    if not a.any():
        return {"clip_ratio": 0.0, "max_run": 0, "clipped": False}
    # أطول سلسلة متتالية
    padded = np.concatenate(([False], a, [False]))
    diff = np.diff(padded.astype(np.int8))
    starts, ends = np.where(diff == 1)[0], np.where(diff == -1)[0]
    max_run = int((ends - starts).max()) if len(starts) else 0
    return {"clip_ratio": float(a.mean()), "max_run": max_run,
            "clipped": bool(max_run >= run)}

def noise_floor_estimate(y, sr, vad_speech_spans=None, frame_ms=50):
    """أرضية ضوضاء مُقدَّرة من إطارات اللاكلام (VAD-aware) — يوسم تقديرًا لا SNR."""
    if len(y) < sr * 0.2:
        return -np.inf, 0
    hop = int(sr * frame_ms / 1000)
    n_frames = max(1, len(y) // hop)
    frames = y[: n_frames * hop].reshape(n_frames, hop)
    frms_db = 20 * np.log10(np.sqrt((frames ** 2).mean(axis=1)) + 1e-12)
    if vad_speech_spans:  # نستبعد إطارات الكلام إن توفرت فواصل VAD
        t = (np.arange(n_frames) + 0.5) * frame_ms / 1000.0
        mask = np.ones(n_frames, dtype=bool)
        for (s, e) in vad_speech_spans:
            mask &= ~((t >= s) & (t <= e))
        if mask.sum() >= max(3, n_frames // 20):
            frms_db = frms_db[mask]
    if len(frms_db) == 0:
        return -np.inf, 0
    # أرضية = مئين 20 من إطارات اللاكلام (متانة ضد بقاء كلام متسرب)
    return float(np.percentile(frms_db, 20)), int(len(frms_db))

def spectral_features(y, sr, n_fft=2048):
    """خصائص طيفية تخدم كشف الموسيقى/الصدى/الجودة."""
    if len(y) < n_fft:
        y = np.pad(y, (0, n_fft - len(y)))
    S = np.abs(_lr.stft(y, n_fft=n_fft, hop_length=n_fft // 2)) ** 2
    freqs = _lr.fft_frequencies(sr=sr, n_fft=n_fft)
    centroid = float(_lr.feature.spectral_centroid(S=S, sr=sr).mean())
    rolloff = float(_lr.feature.spectral_rolloff(S=S, sr=sr, roll_percent=0.85).mean())
    flatness = float(_lr.feature.spectral_flatness(S=S).mean())
    bandwidth = float(_lr.feature.spectral_bandwidth(S=S, sr=sr).mean())
    total = S.sum() + 1e-12
    low = float(S[(freqs < 300)].sum() / total) if (freqs < 300).any() else 0.0
    speech_band = float(S[((freqs >= 300) & (freqs <= 3400))].sum() / total)
    high = float(S[(freqs > 3400)].sum() / total) if (freqs > 3400).any() else 0.0
    return {"centroid_hz": centroid, "rolloff_hz": rolloff, "flatness": flatness,
            "bandwidth_hz": bandwidth, "low_ratio": low, "speech_band_ratio": speech_band,
            "high_ratio": high}

def envelope_smoothness(y, sr, win_ms=100):
    """نعومة الغلاف الزمني — مؤشر ارتداد/صدى (يوسم مؤشرًا لا قياسًا مطلقًا)."""
    hop = max(1, int(sr * win_ms / 1000))
    n = max(1, len(y) // hop)
    env = 20 * np.log10(np.sqrt((y[: n * hop].reshape(n, hop) ** 2).mean(axis=1)) + 1e-12)
    if len(env) < 4:
        return 0.0
    d1 = np.abs(np.diff(env))
    d2 = np.abs(np.diff(env, 2))
    # صدى = ذيول اضمحلال ناعمة طويلة: انحراف منحدر أول منخفض + طاقة ذيل
    return float(np.mean(d2) / (np.mean(d1) + 1e-9))

def measure_file(wav_path, sr):
    """كل قياسات الملف في بنية واحدة — المصدر الوحيد لكل القرارات."""
    y, s = sf.read(str(wav_path), dtype="float32")
    clip = detect_source_clipping(y)
    spec = spectral_features(y, s)
    env_smooth = envelope_smoothness(y, s)
    return {
        "dur_s": len(y) / s,
        "sample_peak_db": sample_peak_db(y),
        "true_peak_est_db": true_peak_db(y, s),
        "rms_db": rms_db(y),
        "crest_db": sample_peak_db(y) - rms_db(y),
        "dc_offset": float(np.mean(y)),
        "zero_cross_rate": float(np.mean(np.abs(np.diff(np.signbit(y))))),
        "silence_ratio": float(np.mean(20 * np.log10(np.abs(y) + 1e-12) < -60)),
        "clip_ratio": clip["clip_ratio"],
        "clip_max_run": clip["max_run"],
        "source_clipping": clip["clipped"],
        "noise_floor_est_db": None,  # يُملأ بعد VAD (Stage 07)
        "env_smoothness": env_smooth,
        **{f"spec_{k}": v for k, v in spec.items()},
    }

### تشغيل القياسات لكل ملف وحفظها في `DIAG` + قاعدة الحالة — هذه الأرقام تقود كل القرارات اللاحقة.

In [ ]:
_MEASURE_CFG = {k: v for k, v in CONFIG["vad"].items()}
diag_rows = []
for r in tqdm(REGISTRY.itertuples(), total=len(REGISTRY), desc="قياس السلامة", unit="ملف"):
    if r.decode_status != "ok":
        continue
    try:
        m = file_cache("measure", r.fid, {"vad": _MEASURE_CFG},
                       lambda: measure_file(r.wav_path, r.dec_sr))
        for k, v in m.items():
            STATE.metric(r.fid, "integrity", k, v)
        diag_rows.append({"fid": r.fid, "orig_name": r.orig_name, "wav_path": r.wav_path,
                          "dec_sr": r.dec_sr, **m})
    except Exception as e:
        STATE.log_error(r.fid, "integrity", type(e).__name__, "error", str(e))
        LOGGER.error("قياس %s فشل: %s", r.orig_name, e)

DIAG = pd.DataFrame(diag_rows)
DIAG.to_csv(PATHS["reports"] / "diagnostics_base.csv", index=False)
if len(DIAG):
    _cols = ["orig_name", "dur_s", "true_peak_est_db", "rms_db", "noise_floor_est_db",
             "silence_ratio", "source_clipping", "spec_flatness"]
    print(DIAG[[c for c in _cols if c in DIAG.columns]].round(2).to_string(index=False))

## STAGE 06 — Multi-Level Audio Deduplication

**المشكلة الحقيقية:** الأرشيفات الصوتية تحتوي تكرارات **مقنّعة**: نسخة OGG وأخرى MP3 لنفس التسجيل، ملف مدمج (`merged`) يضم عدة رسائل، أو نسخة مقصوصة الحواف. الملفات المتطابقة بالبایت وحدها لا تكشف شيئًا من هذا.

### المستويات الأربعة (من الرخيص إلى الغالي — كل مستوى يضيّق المرشحين للموالي)

| المستوى | يكشف | الطريقة | التكلفة |
|---|---|---|---|
| **L1: File Hash** | نسخ متطابقة بايتيًا | SHA-256 تدفقي | O(n) |
| **L2: Canonical Hash** | إعادة ترميز لنفس المحتوى (OGG↔MP3↔WAV) | هاش PCM بعد توحيد (mono/16k/float32) | O(n) فك |
| **L3: Acoustic Fingerprint** | نفس التسجيل بعد تعديل/ضغط مختلف | بصمة Log-Mel خشنة (4 fps) + كوساين + قيد نسبة المدة | O(n²) على البصمات فقط |
| **L4: Containment** | ملف كامل مضمن داخل ملف أطول (مثل ملف مدمج) | ارتباط متقاطع منزلق لمغلف RMS (50fps) | أزواج مرشحة فقط |

**تحذير هندسي مضمّن في الكود:** نفس المتحدث ينطق نفس الجملة مرتين **ليس تكرارًا** — لذلك يستخدم المستوى L3 قيد نسبة المدة (≥0.92) **و** تشابه البصمة الزمنية الكاملة (وليس متوسط الطيف فقط)، وL4 يتطلب احتواء الغلاف عبر **كامل** طول الملف القصير. بصمة ECAPA (المتحدث) **لا تُستخدم في إزالة التكرار أبدًا** — فقط محتوى الموجة.

**اختيار النسخة الأفضل داخل العنقود:** قرار موزون (جودة الإشارة الفعلية المقاسة + سلامة القمم + الاكتمال) — **لا** الامتداد WAV/FLAC كمعيار وحيد (ملف MP3 من مصدر أنظف قد يتفوق).

**مخرجات:** `DEDUP_CLUSTERS` + `KEPT`/`EXCLUDED` + `reports/deduplication.csv` (كل استبعاد بسببه الكامل) + الملفات المستبعدة تُحفظ في `quarantine/duplicate/` (لا حذف نهائي).

### STAGE 06.0 — معيار اختيار النسخة الأفضل داخل عنقود التكرار

قرار موزون على **القياسات الفعلية** (سلامة القمم، الاكتمال، النطاق الترددي المقيس، أرضية الضوضاء) — **لا** على امتداد الملف (WAV/FLAC ليس دليل جودة بالضرورة: نسخة MP3 من مصدر أنظف قد تتفوق).

In [ ]:
# -*- coding: utf-8 -*-
"""STAGE 06.1 — اختيار النسخة الأفضل داخل عنقود تكرار: قرار موزون قياسي لا شكلي."""

def copy_quality_score(fid, files_df):
    """درجة جودة النسخة من القياسات الفعلية (DIAG إن توفرت) — لا امتداد الملف."""
    d = DIAG.set_index("fid").loc[fid] if fid in DIAG["fid"].values else None
    f = files_df.set_index("fid").loc[fid]
    score = 0.0
    # 1) سلامة الإشارة (أعلى وزن) — لا قص مصدر
    score += 0.35 * (0.0 if (d is not None and bool(d["source_clipping"])) else 1.0)
    # 2) اكتمال المحتوى — مدة أطول = محتوى أكمل
    dur = float(f["dec_dur_s"])
    score += 0.25 * min(dur / 60.0, 1.0)
    # 3) نطاق ترددي أعلى فعليًا (spec_rolloff مقيس لا امتداد)
    roll = float(d["spec_rolloff_hz"]) if d is not None else 4000.0
    score += 0.25 * min(roll / 8000.0, 1.0)
    # 4) ضوضاء أقل (أرضية أعمق)
    if d is not None and np.isfinite(d["noise_floor_est_db"] if d["noise_floor_est_db"] is not None else -np.inf):
        score += 0.15 * min(max((float(d["noise_floor_est_db"]) + 60) / 30, 0), 1.0)
    else:
        score += 0.15 * 0.5
    return score

def pick_best_copy(cluster_fids, files_df, _cache_canon=None):
    """أفضل نسخة = أعلى درجة جودة فعلية (تعادل → الأسبق بالمعرف للاستقرار)."""
    scored = sorted(((-copy_quality_score(f, files_df), f) for f in cluster_fids))
    return scored[0][1]

### دوال إزالة التكرارات (المستويات الأربعة)

L1 هاش الملف، L2 هاش PCM الموحد (mono/16k)، L3 بصمة طيفية زمنية خشنة + قيد نسبة المدة (نفس الجملة المنطوقة مرتين **ليست** تكرارًا)، L4 كشف الاحتواء الجزئي بغلاف RMS منزلق — كلها وظائف خالصة قابلة للاختبار.

In [ ]:
# -*- coding: utf-8 -*-
"""STAGE 06 — إزالة التكرارات رباعية المستويات مع توثيق كامل."""
import soxr

D = CONFIG["dedup"]

def canonical_pcm(wav_path, sr_target=16000):
    """PCM الموحد للمقارنة الصوتية: mono 16k float32 (يبطل فروق الحاوية/القنوات/المعدل)."""
    y, sr = sf.read(str(wav_path), dtype="float32")
    if sr != sr_target:
        y = soxr.resample(y, sr, sr_target, quality="VHQ")
    return y

def canonical_hash(y):
    return sha256_bytes(np.asarray(y, dtype="<f4").tobytes())

def acoustic_fingerprint(y, sr=16000, n_mels=64, fps=4):
    """بصمة زمنية-طيفية خشنة: مقاومة لإعادة الترميز، حساسة للاختلاف الصوتي الحقيقي."""
    import librosa as lr
    S = lr.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels, n_fft=1024,
                                   hop_length=max(1, sr // fps), power=2.0)
    S_db = lr.power_to_db(S + 1e-10)
    # توحيد L2 لكل إطار + إزالة المتوسط الزمني (يعمم ضد إزاحة طيفية مستقرة)
    S_db -= S_db.mean(axis=1, keepdims=True)
    return S_db.T.astype(np.float32)  # [frames, n_mels]

def fp_cosine(fa, fb):
    """تشابه كوساين بين بصمتين كاملتين (إطارات كاملة — ليس متوسطًا)."""
    n = min(len(fa), len(fb))
    if n < 8:
        return 0.0
    a, b = fa[:n].ravel(), fb[:n].ravel()
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na == 0 or nb == 0:
        return 0.0
    return float(np.dot(a, b) / (na * nb))

def envelope(y, sr, fps=50):
    hop = max(1, sr // fps)
    n = max(1, len(y) // hop)
    e = np.sqrt((y[: n * hop].reshape(n, hop) ** 2).mean(axis=1)) + 1e-9
    e_db = 20 * np.log10(e)
    return (e_db - e_db.mean()) / (e_db.std() + 1e-9)  # تطبيع z: مقارنة شكل لا مستوى

def best_ncc(needle, hay, min_overlap=None):
    """أقصى ارتباط متقاطع منزلق معياري لغلاف قصير داخل طويل (إطارات متساوية الطول)."""
    n = len(needle)
    if min_overlap is None:
        min_overlap = max(8, int(n * 0.9))
    best, pos = -1.0, -1
    step = max(1, n // 400)  # دقة بحث عملية
    for i in range(0, max(1, len(hay) - min_overlap + 1), step):
        seg = hay[i: i + n]
        if len(seg) < n:      # ذيل مبتور — الطول يجب أن يطابق needle تمامًا
            if len(seg) < min_overlap:
                break
            needle_e = needle[: len(seg)]
        else:
            needle_e = needle
        ns = np.linalg.norm(seg)
        if ns == 0:
            continue
        c = float(np.dot(needle_e, seg) / (np.linalg.norm(needle_e) * ns + 1e-12))
        if c > best:
            best, pos = c, i
    return best, pos

### المستويات الثلاثة الأولى (File Hash → Canonical Hash → بصمة طيفية) — كل استبعاد بسببه ومرجعه الأفضل.

In [ ]:
# ══ تنفيذ المستويات على REGISTRY ══
_files = REGISTRY[REGISTRY["decode_status"] == "ok"].copy()
_excl = []      # (fid, reason, cluster_id, better_fid)
_canon = {}     # fid → (hash, y, fp)
_DPCFG = {"sr": D["fingerprint"]["sr"], "n_mels": D["fingerprint"]["n_mels"],
          "fps": D["fingerprint"]["fps"]}
for r in tqdm(_files.itertuples(), total=len(_files), desc="بصمات إزالة التكرار", unit="ملف"):
    y = canonical_pcm(r.wav_path)
    h = canonical_hash(y)
    def _fp():
        return acoustic_fingerprint(y)   # y محفوظ في الإغلاق (closure)
    fp = file_cache("dedup_fp", r.fid, _DPCFG, _fp, decode=_numpyfy)
    _canon[r.fid] = (h, y, fp)
    del y

# ── L1: هاش الملف الخام (نسخ متطابقة بايتيًا) ──
_by_file_sha = {}
for r in _files.itertuples():
    _by_file_sha.setdefault(r.file_sha256, []).append(r.fid)
for sha, group in _by_file_sha.items():
    if len(group) > 1:
        best = pick_best_copy(group, _files, _canon)
        for f in group:
            if f != best:
                _excl.append((f, "L1_file_hash_duplicate", sha[:12], best))

# ── L2: هاش PCM الموحد (إعادة ترميز لنفس المحتوى) ──
L12 = {}
for r in _files.itertuples():
    ch = _canon[r.fid][0]
    L12.setdefault(ch, []).append(r.fid)
for ch, group in L12.items():
    if len(group) > 1:
        already = {e[0] for e in _excl}
        alive = [f for f in group if f not in already]
        if len(alive) <= 1:
            continue
        best = pick_best_copy(alive, _files, _canon)
        for f in alive:
            if f != best:
                _excl.append((f, "L2_canonical_hash_duplicate", ch[:12], best))

# ── L3: بصمة طيفية (نفس التسجيل بعد تعديل مختلف) ──
FP = D["fingerprint"]
survivors = [r.fid for r in _files.itertuples()
             if r.fid not in {e[0] for e in _excl}]
for i in range(len(survivors)):
    for j in range(i + 1, len(survivors)):
        a, b = survivors[i], survivors[j]
        ya, yb = _canon[a][1], _canon[b][1]
        ratio = min(len(ya), len(yb)) / max(len(ya), len(yb))
        if ratio < FP["duration_ratio_min"]:
            continue  # مدد مختلفة جدًا — ليسا نسختين لنفس التسجيل
        sim = fp_cosine(_canon[a][2], _canon[b][2])
        if sim >= FP["sim_threshold"]:
            already = {e[0] for e in _excl}
            pair = [f for f in (a, b) if f not in already]
            if len(pair) <= 1:
                continue  # شبكة أمان: لا يُستبعد طرفا زوج متبادل كلاهما أبدًا
            best = pick_best_copy(pair, _files, _canon)
            other = pair[1] if best == pair[0] else pair[0]
            _excl.append((other, f"L3_acoustic_fingerprint_dup sim={sim:.3f}", f"fp{int(sim*1000)}", best))

### المستوى الرابع: كشف الاحتواء الجزئي والقرار النهائي

الملفات القصيرة تُفحص ضد الأطول بغلاف RMS منزلق (L4)، ثم القرار النهائي: `KEPT`/`QUARANTINE_DUP` مع حفظ المستبعدات في `quarantine/duplicate/` (لا حذف بيانات أبدًا) وتقرير كامل الأسباب.

In [ ]:
# ── L4: احتواء جزئي (ملف مضمن في أطول) + سياسة الحاوية المركّبة ──
CD = D["containment"]
excl_set = {e[0] for e in _excl}
finalists = [r for r in _files.itertuples() if r.fid not in excl_set]
finalists.sort(key=lambda r: -r.dec_dur_s)
_envs = {r.fid: envelope(_canon[r.fid][1], 16000, fps=CD["envelope_fps"]) for r in finalists}
# العلاقات: جزء قصير ⊂ حاوية طويلة
_containment = []   # (short_row, long_row, ncc)
for i, short in enumerate(finalists):
    if short.dec_dur_s < CD["min_short_s"] or short.fid in excl_set:
        continue
    for longr in finalists:
        if longr.fid == short.fid or longr.dec_dur_s <= short.dec_dur_s * 1.05:
            continue
        if short.dec_dur_s / longr.dec_dur_s > 0.75:
            continue  # ليست علاقة احتواء واضحة
        score, pos = best_ncc(_envs[short.fid], _envs[longr.fid])
        if score >= CD["xcorr_threshold"]:
            _containment.append((short, longr, score))
            LOGGER.debug("L4 علاقة: %s ⊂ %s (ncc=%.3f)", short.orig_name, longr.orig_name, score)
            break

# سياسة الحاوية: إن غطّت الأجزاء ~كل مدة الحاوية فهي "حاوية مركّبة" (merged) —
# تُستبعد الحاوية (إعادة ترميز أدنى جودة) وتُحفظ الأجزاء الأصلية الأفضل.
_parts_by_container = {}
for short, longr, score in _containment:
    _parts_by_container.setdefault(longr.fid, []).append((short, score))
for longr in finalists:
    parts = _parts_by_container.get(longr.fid, [])
    if not parts:
        continue
    covered = sum(p.dec_dur_s for p, _ in parts)
    if covered >= 0.85 * longr.dec_dur_s:
        # حاوية مركّبة: كل محتواها متوفر كملفات مستقلة أفضل جودة
        _excl.append((longr.fid,
                      f"L4_composite_container (أجزاؤه {covered:.0f}s من {longr.dec_dur_s:.0f}s "
                      f"متوفرة مستقلة بأفضل جودة)",
                      "ct_comp", parts[0][0].fid))
        excl_set.add(longr.fid)
        LOGGER.info("L4: %s حاوية مركّبة — استُبعدت وحُفظت أجزاؤها الأصلية", longr.orig_name)
        for p, _ in parts:
            # الأجزاء لا تُستبعد (هي المصادر الأفضل)
            if any(e[0] == p.fid and e[3] == longr.fid for e in _excl):
                _excl[:] = [e for e in _excl if not (e[0] == p.fid and e[3] == longr.fid)]
                excl_set.discard(p.fid)

# العلاقات المتبقية (حاوية ذات محتوى فريد): الجزء القصير مكرر داخلها → يُستبعد
for short, longr, score in _containment:
    if longr.fid in excl_set or short.fid in excl_set:
        continue
    _excl.append((short.fid, f"L4_contained_in {longr.fid} ncc={score:.3f}",
                  f"ct{int(score*1000)}", longr.fid))
    excl_set.add(short.fid)
    LOGGER.info("L4: %s مضمّن داخل %s (ncc=%.3f) والحاوية تحوي محتوى فريدًا",
                 short.orig_name, longr.orig_name, score)

# ── القرار النهائي ──
_dedup_excl = pd.DataFrame([{"fid": f, "reason": reason, "cluster": cluster, "kept_instead": better}
                             for f, reason, cluster, better in _excl],
                            columns=["fid", "reason", "cluster", "kept_instead"])
if _dedup_excl.empty:
    KEPT = _files.reset_index(drop=True)
    QUARANTINE_DUP = _files.iloc[0:0].reset_index(drop=True)
else:
    KEPT = _files[~_files["fid"].isin(_dedup_excl["fid"])].reset_index(drop=True)
    QUARANTINE_DUP = _files[_files["fid"].isin(_dedup_excl["fid"])].reset_index(drop=True)

# حفظ المستبعدات (لا حذف بيانات مصدر أبدًا)
dup_dir = PATHS["export_root"] / "quarantine" / "duplicate"
dup_dir.mkdir(parents=True, exist_ok=True)
for r in QUARANTINE_DUP.itertuples():
    dst = dup_dir / Path(r.wav_path).name
    if not dst.exists():
        _shutil.copy2(r.wav_path, dst)

_dedup_excl.to_csv(PATHS["reports"] / "deduplication.csv", index=False)
for _, row in _dedup_excl.iterrows():
    STATE.decision(row["fid"], "dedup", "exclude_duplicate", row["reason"])
LOGGER.info("إزالة التكرارات: استُبعد %d وأُبقي %d | أسباب: %s",
            len(_dedup_excl), len(KEPT),
            _dedup_excl["reason"].str.split().str[0].value_counts().to_dict() if len(_dedup_excl) else "لا شيء")
print(f"KEPT = {len(KEPT)} ملف فريد | مستبعد {len(_dedup_excl)} (محفوظة في quarantine/duplicate)")

## STAGE 07 — Speech & Speaker Analysis

**الهدف:** لكل ملف: إحصاءات النشاط الصوتي (نسبة الكلام، عدد الجمل، متوسط طولها) عبر **Silero VAD** (إن توفر) وإلا البديل الطاقي الاحتياطي (موثّق النقص للكلام الهادئ).

- هذه الفواصل تخدم: تقدير أرضية الضوضاء الفعلي (إطارات اللاكلام)، حماية الكلام الهادئ، والكشف عن ملفات "بلا كلام تقريبًا" (استبعاد مبكر موثّق).
- **بنية VAD موحدة `speech_spans()`**: كل المراحل اللاحقة (التقطيع، القياس، الاحتواء) تستدعيها بنفس التكوين — لا نسخ مكررة من منطق VAD.

**مخرجات:** `SPEECH_STATS` + فواصل كلام مخزنة تُستدعى لاحقًا + إكمال `noise_floor_est_db` في `DIAG`.

In [ ]:
# -*- coding: utf-8 -*-
"""STAGE 07 — تحليل النشاط الصوتي الموحد (VAD واحد لكل الخط)."""
import librosa as librosa

_VAD_MODEL = None  # دورة حياة النموذج: يُحمّل مرة واحدة ويُعاد استخدامها

def vad_model():
    """تحميل Silero مرة واحدة لكل جلسة (دورة حياة موحدة للنماذج)."""
    global _VAD_MODEL
    if _VAD_MODEL is None:
        if not CAPS["silero_vad"]:
            raise OptionalDependencyMissing("silero-vad غير متاح — استخدم energy_vad")
        from silero_vad import load_silero_vad
        _VAD_MODEL = load_silero_vad()
    return _VAD_MODEL

def energy_vad_spans(y, sr, noise_db=None, on_db=-38.0, off_db=-42.0,
                     min_speech_ms=250, min_silence_ms=350):
    """البديل الاحتياطي الطاقي — هستيريسيس ثنائي العتبة (معلَن النقص: أضعف مع الكلام الهادئ)."""
    hop = max(1, int(sr * 0.01))
    n = max(1, len(y) // hop)
    env = 20 * np.log10(np.sqrt((y[: n * hop].reshape(n, hop) ** 2).mean(axis=1)) + 1e-12)
    if noise_db is not None:
        on_db, off_db = noise_db + 8, noise_db + 4
    spans, start = [], None
    for i, e in enumerate(env):
        t = i * 0.01
        if start is None and e > on_db:
            start = t
        elif start is not None and e < off_db and t - start >= min_speech_ms / 1000:
            if (t - 0.01) - start >= min_speech_ms / 1000:
                spans.append((start, t - 0.01))
            start = None
    if start is not None:
        spans.append((start, len(y) / sr))
    # دمج الفجوات القصيرة
    merged = []
    for s, e in spans:
        if merged and s - merged[-1][1] < min_silence_ms / 1000:
            merged[-1] = (merged[-1][0], e)
        else:
            merged.append((s, e))
    return [(s, e) for s, e in merged if (e - s) >= min_speech_ms / 1000]

def speech_spans(y, sr, cfg=None):
    """فواصل الكلام الموحدة: Silero (هستيريسيس كامل التوصيل من CONFIG) أو البديل الطاقي.
    
    كل عتبات VAD تُقرأ من CONFIG['vad'] فعليًا — لا قيم مضمّنة (إصلاح الخلل التاريخي).
    """
    V = cfg or CONFIG["vad"]
    if CAPS["silero_vad"]:
        import torch
        ts = get_speech_timestamps_torch(y, sr, V)
        return ts
    _warn_once("Silero VAD غير متاح — البديل الطاقي الاحتياطي يعمل (أدنى دقة للكلام الهادئ)")
    return energy_vad_spans(y, sr,
                            min_speech_ms=V["min_speech_ms"],
                            min_silence_ms=V["min_silence_ms"])

def get_speech_timestamps_torch(y, sr, V):
    """Silero مع كل معاملات CONFIG موصولة فعليًا (هستيريسيس + حدود زمنية)."""
    from silero_vad import get_speech_timestamps
    import torch as t
    y16 = y if sr == 16000 else soxr.resample(y, sr, 16000, quality="VHQ")
    wav = t.from_numpy(np.ascontiguousarray(y16, dtype=np.float32))
    ts = get_speech_timestamps(
        wav, vad_model(),
        threshold=float(V["threshold"]),
        neg_threshold=float(V["neg_threshold"]),
        min_speech_duration_ms=int(V["min_speech_ms"]),
        min_silence_duration_ms=int(V["min_silence_ms"]),
        speech_pad_ms=int(V["speech_pad_ms"]),
        max_speech_duration_s=float(V["max_speech_s"]),
        return_seconds=True)
    # إعادة تحويل الأزمنة إلى معدل الملف الأصلي إن لزم (البصمة الزمنية كما هي بالثواني)
    return [(s["start"], s["end"]) for s in ts]

def spans_total(spans):
    return float(sum(e - s for s, e in spans))

### تشغيل التحليل: إحصاءات الكلام + إكمال أرضية الضوضاء الفعلية (من إطارات اللاكلام) + حماية الكلام الهادئ + حجر الملفات بلا كلام.

In [ ]:
# ══ تشغيل التحليل لكل ملف ══
speech_rows, span_cache = {}, {}
_VADCFG = dict(CONFIG["vad"])
for r in tqdm(KEPT.itertuples(), total=len(KEPT), desc="تحليل الكلام", unit="ملف"):
    y, sr = sf.read(r.wav_path, dtype="float32")
    spans = file_cache("vad_spans", r.fid, _VADCFG, lambda: speech_spans(y, sr))
    span_cache[r.fid] = spans
    speech_dur = spans_total(spans)
    nf_est, nf_frames = noise_floor_estimate(y, sr, vad_speech_spans=spans)
    # حماية الكلام الهادئ: أرضية الضوضاء تُستخدم مع حراسة (Quiet Speech Guard)
    quiet_speech = bool(nf_est > CONFIG["noise"]["detection"]["floor_estimate_above_db"]
                        and CONFIG["noise"]["detection"]["quiet_speech_guard"]
                        and speech_dur / max(len(y) / sr, 1e-9) > 0.4)
    if quiet_speech:
        LOGGER.info("%s: كلام هادئ (أرضية %.0f dB لكن نسبة كلام %.2f) — حماية Quiet Guard",
                    r.orig_name, nf_est, speech_dur / (len(y) / sr))
    speech_rows[r.fid] = {
        "fid": r.fid, "orig_name": r.orig_name,
        "speech_s": round(speech_dur, 2),
        "speech_ratio": round(speech_dur / max(len(y) / sr, 1e-9), 3),
        "n_utterances": len(spans),
        "avg_utt_s": round(speech_dur / len(spans), 2) if spans else 0.0,
        "noise_floor_est_db": round(nf_est, 1) if np.isfinite(nf_est) else None,
        "nf_estimate_frames": nf_frames,
        "quiet_speech_flag": quiet_speech,
    }
    for k, v in speech_rows[r.fid].items():
        STATE.metric(r.fid, "speech", k, v if isinstance(v, (int, float, bool)) else None)
    del y

SPEECH_STATS = pd.DataFrame(list(speech_rows.values()))
DIAG = DIAG.merge(SPEECH_STATS[["fid", "speech_ratio", "speech_s", "n_utterances",
                                 "avg_utt_s", "noise_floor_est_db", "quiet_speech_flag"]],
                   on="fid", how="left", suffixes=("", "_vad"))
DIAG["noise_floor_est_db"] = DIAG["noise_floor_est_db_vad"].combine_first(DIAG["noise_floor_est_db"])
DIAG = DIAG.drop(columns=[c for c in DIAG.columns if c.endswith("_vad")])
DIAG.to_csv(PATHS["reports"] / "diagnostics.csv", index=False)

# ملفات بلا كلام كافٍ → حجر صحي مبكر موثّق (لا كلام = لا قيمة تدريبية)
_speech_min_s = 2.0
_no_speech = SPEECH_STATS[SPEECH_STATS["speech_s"] < _speech_min_s]["fid"].tolist()
for f in _no_speech:
    STATE.decision(f, "speech_analysis", "quarantine_no_speech",
                   f"كلام فعلي {SPEECH_STATS.set_index('fid').loc[f, 'speech_s']}s < {_speech_min_s}s")
if _no_speech:
    LOGGER.warning("ملفات بلا كلام كافٍ (حجر صحي): %s", _no_speech)
    KEPT = KEPT[~KEPT["fid"].isin(_no_speech)].reset_index(drop=True)
print(SPEECH_STATS[["orig_name", "speech_s", "speech_ratio", "n_utterances",
                     "noise_floor_est_db", "quiet_speech_flag"]].to_string(index=False))

## STAGE 08 — Target Speaker Verification (ECAPA-TDNN)

**الهدف:** لا يدخل التدريب إلا تسجيلات المتحدث المستهدف. النموذج: **SpeechBrain ECAPA-TDNN** (VoxCeleb، 192-dim) — نموذج تحقق متحدث مفتوح راسخ.

### منهجية المرجع (بلا اختراع بصمة)
1. **مرجع صريح:** ضع تسجيلات مؤكدة في مجلد واضبط `speaker.reference_dir` → centroid من بصماتها.
2. **Bootstrap (إن لم يوجد مرجع):** النظام يحسب البصمات لكل الملفات، **يجمّعها (agglomerative clustering)**، وإن هيمن عنقود واحد (أرشيف رسائل صوتية لمتحدث واحد) → يتخذ الـ medoid بصمةً ووسم `bootstrap=True`. **هذا وسم صريح يحتاج تأكيدًا بشريًا** — لا يعتبر إثبات هوية.

### العتبة — معايرة لا رقم سحري
- `match_threshold: auto` → تُعاير من التوزع الفعلي: عتبة = عتمة داخل-عنقود مقابل بين-عنقود (لوكتشفت عناقيد متعددة)، أو بوابة سفلية قوية (0.55) مع ملاحظة معايرة.
- التصنيف: `match` / `probable` / `borderline` / `mismatch` + `insufficient_speech` — **الحالات الحدودية تذهب للمراجعة البشرية (قائمة Review)، لا استبعادًا آليًا**.

### إصلاح خطير من المواصفات
تشابه ECAPA **لا يُستخدم وحده أبدًا** لإثبات الأفضلية أو الاستبعاد النهائي (البصمة وحدها لا تميز نفس الجملة من نفس الشخص) — ولا يُستخدم في إزالة التكرارات (مستحيل منطقيًا: نفس الشخص = تشابه عالٍ بطبيعته).

**مخرجات:** `SPEAKER` (DataFrame: تشابه/تصنيف/ثقة) + `reports/speaker_verification.csv` + بصمة المرجع محفوظة.

In [ ]:
# -*- coding: utf-8 -*-
"""STAGE 08 — التحقق من هوية المتحدث المستهدف (ECAPA-TDNN + معايرة)."""
_ECAPA = None

_ECAPA_MODEL_DIR = None

def ecapa():
    """دورة حياة نموذج واحدة (تحميل كسول + إعادة استخدام).

    تحميل محلي أولًا (HF_HUB_OFFLINE) إن كانت الأوزان مخزنة مسبقًا — يمنع
    التعليق على فحوصات HuggingFace الشبكية عند تقييد المعدل. الإنترنت تُستخدم
    فقط عند غياب الأوزان محليًا.
    """
    global _ECAPA
    if _ECAPA is None:
        if not CAPS["speechbrain"]:
            raise OptionalDependencyMissing("speechbrain غير متاح")
        from speechbrain.inference.speaker import EncoderClassifier
        savedir = PATHS["models_cache"] / "ecapa"
        local_ready = (savedir / "hyperparams.yaml").exists()
        _prev = _os.environ.get("HF_HUB_OFFLINE")
        if local_ready:
            _os.environ["HF_HUB_OFFLINE"] = "1"   # تحميل محلي صرفًا (لا شبكة)
        try:
            _ECAPA = EncoderClassifier.from_hparams(
                source="speechbrain/spkrec-ecapa-voxceleb",
                savedir=str(savedir),
                run_opts={"device": DEVICE if DEVICE != "auto" else "cpu"})
        except Exception:
            if not local_ready:
                raise          # لم تكن هناك أوزان محلية — فشل التنزيل موثق
            _os.environ.pop("HF_HUB_OFFLINE", None)   # إعادة محاولة عبر الشبكة
            _ECAPA = EncoderClassifier.from_hparams(
                source="speechbrain/spkrec-ecapa-voxceleb",
                savedir=str(savedir),
                run_opts={"device": DEVICE if DEVICE != "auto" else "cpu"})
        finally:
            if _prev is None:
                _os.environ.pop("HF_HUB_OFFLINE", None)
            else:
                _os.environ["HF_HUB_OFFLINE"] = _prev
    return _ECAPA

_EMB_CHUNK_S = 30.0   # نافذة البصمة للملفات الطويلة (ECAPA مبني لجمل 3-10s)

def ecapa_embedding(y, sr, min_s=3.0):
    """بصمة ECAPA (192-dim) — دون الحد الأدنى: None (لا بصمة زائفة).

    الملفات الطويلة تُبصم بنوافذ 30s ثم يُحسب متوسط البصمات المعيارية:
    1) الاستدلال الكامل على مقاطع عدة دقائق أبطأ بأوامر مقدار على CPU،
    2) المتوسط النافذي أثبت ضد نافذة شاذة واحدة (ضوضاء/نشاز).
    """
    import torch
    y16 = y if sr == 16000 else soxr.resample(y, sr, 16000, quality="VHQ")
    if len(y16) < 16000 * min_s:
        return None  # كلام غير كافٍ — لا بصمة زائفة
    chunk = int(16000 * _EMB_CHUNK_S)
    if len(y16) <= chunk:
        with torch.inference_mode():
            rel = ecapa().encode_batch(torch.from_numpy(
                np.ascontiguousarray(y16, dtype=np.float32)))
            emb = rel.squeeze().cpu().numpy().astype(np.float32)
        n = np.linalg.norm(emb)
        return emb / n if n > 0 else None
    # نوافذ 30s — متوسط بصمات معيارية
    embs = []
    for i in range(0, len(y16) - 16000 * int(min_s), int(chunk * 0.9)):  # تراكب 10%
        seg = np.ascontiguousarray(y16[i: i + chunk], dtype=np.float32)
        if len(seg) < 16000 * int(min_s):
            break
        with torch.inference_mode():
            rel = ecapa().encode_batch(torch.from_numpy(seg))
            e_ = rel.squeeze().cpu().numpy().astype(np.float32)
        n = np.linalg.norm(e_)
        if n > 0:
            embs.append(e_ / n)
        if i + chunk >= len(y16):
            break
    if not embs:
        return None
    avg = np.stack(embs).mean(axis=0)
    n = np.linalg.norm(avg)
    return avg / n if n > 0 else None

def cosine(a, b):
    if a is None or b is None:
        return None
    return float(np.dot(a, b))

### بناء بصمة المرجع: مرجع صريح إن وُجد مجلد `speaker.reference_dir`، وإلا بصمات كل الملفات تمهيدًا لوضع Bootstrap.

In [ ]:
# ══ 1) بناء بصمة المرجع ══
SPK = CONFIG["speaker"]
REFERENCE_EMB, REFERENCE_MODE = None, None
ref_dir = SPK.get("reference_dir") or ""
ref_embs = []
if ref_dir and _Path(ref_dir).exists():
    for p in sorted(_Path(ref_dir).glob("*")):
        if p.suffix.lower() in CONFIG["input"]["extensions"]:
            try:
                y, sr = sf.read(str(p), dtype="float32")
                e = ecapa_embedding(y if y.ndim == 1 else y.mean(axis=1), sr)
                if e is not None:
                    ref_embs.append(e)
            except Exception as ex:
                LOGGER.warning("تعذر تضمين مرجع %s: %s", p.name, ex)
if ref_embs:
    R = np.stack(ref_embs)                       # centroid + medoid
    REFERENCE_EMB = R.mean(axis=0); REFERENCE_EMB /= np.linalg.norm(REFERENCE_EMB)
    REFERENCE_MODE = "explicit_reference"
    LOGGER.info("بصمة مرجع صريحة من %d تسجيلًا", len(ref_embs))

# ══ 2) بصمات الملفات ══
emb_map = {}
if CONFIG["speaker"]["verification_enabled"] and (CAPS["speechbrain"]):
    _EMBCFG = {"min_s": SPK["min_speech_for_embedding_s"], "chunk_s": 30.0}
    for r in tqdm(KEPT.itertuples(), total=len(KEPT), desc="بصمات المتحدث", unit="ملف"):
        try:
            def _emb():
                y, sr = sf.read(r.wav_path, dtype="float32")
                spans = span_cache.get(r.fid, [])
                e = ecapa_embedding(y, sr, min_s=SPK["min_speech_for_embedding_s"])
                if e is None and spans:
                    y_speech = np.concatenate([y[int(s * sr): int(e_ * sr)] for s, e_ in spans
                                               if int(e_ * sr) > int(s * sr)])
                    e = ecapa_embedding(y_speech, sr, min_s=SPK["min_speech_for_embedding_s"])
                del y
                return e if e is not None else []   # None→[] للـ JSON
            e = file_cache("ecapa_emb", r.fid, _EMBCFG, _emb, decode=_numpyfy)
            emb_map[r.fid] = (e if len(e) == 192 else None) if e is not None else None
        except OptionalDependencyMissing:
            emb_map[r.fid] = None
            break
        except Exception as e:
            LOGGER.warning("بصمة %s فشلت: %s", r.orig_name, e)
            emb_map[r.fid] = None

### وضع Bootstrap (بوسم صريح يحتاج تأكيدك) + معايرة العتبة من التوزع الفعلي + التصنيف الخماسي — الحدودي يذهب للمراجعة البشرية لا للاستبعاد الآلي.

In [ ]:
# ══ 3) Bootstrap عند غياب المرجع (بوسم صريح يحتاج تأكيدًا) ══
if REFERENCE_EMB is None and emb_map and any(v is not None for v in emb_map.values()):
    from sklearn.cluster import AgglomerativeClustering
    ids = [f for f, v in emb_map.items() if v is not None]
    X = np.stack([emb_map[f] for f in ids])
    if len(ids) >= 3:
        cl = AgglomerativeClustering(n_clusters=2, metric="cosine", linkage="average").fit(X)
        lbl = cl.labels_
        if (lbl == 0).sum() / len(lbl) >= 0.7 or (lbl == 1).sum() / len(lbl) >= 0.7:
            dom = 0 if (lbl == 0).sum() >= (lbl == 1).sum() else 1
            dom_ids = [f for f, l in zip(ids, lbl) if l == dom]
            sims = [cosine(emb_map[a], emb_map[b]) for i, a in enumerate(dom_ids)
                    for b in dom_ids[i + 1:]]
            medoid = dom_ids[int(np.argmax([sum(cosine(emb_map[a], emb_map[b]) for b in dom_ids if b != a)
                                            for a in dom_ids]))]
            REFERENCE_EMB = emb_map[medoid]
            REFERENCE_MODE = f"bootstrap_medoid (cluster={len(dom_ids)}/{len(ids)}, sim={np.mean(sims):.3f})"
            LOGGER.warning("⚠ Bootstrap: لا مرجع صريح — اتخذ medoid بصمةً. أكد الهوية عبر speaker.reference_dir")
        else:
            LOGGER.warning("عناقيد متعددة بلا مرجع — التصنيف يعتمد كوساين القرابة النسبية فقط (غير حاسم)")
    else:
        REFERENCE_EMB = emb_map[ids[0]]
        REFERENCE_MODE = "bootstrap_single_file"

# ══ 4) عتبة معايرة + تصنيف ══
def calibrate_threshold(ids, embs):
    if SPK["match_threshold"] != "auto":
        return float(SPK["match_threshold"]), "explicit_config"
    sims = np.array([cosine(embs[a], embs[b]) for i, a in enumerate(ids)
                     for b in ids[i + 1:]] or [0.0])
    if REFERENCE_MODE and REFERENCE_MODE.startswith("bootstrap") and len(sims):
        # عتبة من فجوة التوزع داخل-الأرشيف (متانة ضد تشابه الجمل نفسها)
        lo = np.percentile(sims, 10); hi = np.percentile(sims, 90)
        thr = float(max(0.55, lo - 0.05))
        return thr, f"auto (intra-archive q10={lo:.3f} q90={hi:.3f})"
    return 0.55, "auto_fallback (لا توزع كافٍ للمعايرة)"

sp_rows = []
if REFERENCE_EMB is not None:
    ids = list(emb_map.keys())
    THR, thr_src = calibrate_threshold(ids, emb_map)
    band = SPK["borderline_band"]
    for r in KEPT.itertuples():
        e = emb_map.get(r.fid)
        if e is None:
            cls, sim, conf = "insufficient_speech", None, 0.0
        else:
            sim = cosine(e, REFERENCE_EMB)
            conf = min(abs(sim - THR) / band * 0.5 + 0.5, 1.0)
            if sim >= THR + band: cls, conf = "match", 1.0
            elif sim >= THR: cls = "probable"
            elif sim >= THR - band: cls = "borderline"
            else: cls = "mismatch"
        sp_rows.append({"fid": r.fid, "orig_name": r.orig_name,
                        "speaker_sim": None if sim is None else round(sim, 4),
                        "speaker_class": cls,
                        "decision_confidence": round(conf, 3),
                        "threshold": round(THR, 4), "threshold_source": thr_src,
                        "reference_mode": REFERENCE_MODE})
        STATE.metric(r.fid, "speaker", "sim", sim if sim is not None else -1)
        STATE.decision(r.fid, "speaker", cls, f"sim={sim} thr={THR:.3f} ({thr_src})")
    SPEAKER = pd.DataFrame(sp_rows)
    _counts = SPEAKER["speaker_class"].value_counts().to_dict()
    LOGGER.info("المتحدث: عتبة %.3f (%s) | توزع: %s", THR, thr_src, _counts)
    print(SPEAKER[["orig_name", "speaker_sim", "speaker_class",
                   "decision_confidence"]].to_string(index=False))
    SPEAKER.to_csv(PATHS["reports"] / "speaker_verification.csv", index=False)
else:
    SPEAKER = pd.DataFrame(columns=["fid", "orig_name", "speaker_sim", "speaker_class",
                                     "decision_confidence", "threshold", "threshold_source",
                                     "reference_mode"])
    if not CAPS["speechbrain"]:
        print("⚠ التحقق من المتحدث معطّل: speechbrain غير متاح — لا إثبات هوية يعمل")
    else:
        print("⚠ لا بصمات متاحة — تعذر بناء المرجع")

# التصنيفات غير الحاسمة: مراجعة بشرية (لا استبعاد آلي من الحالات الحدودية)
REVIEW_SPEAKER = SPEAKER[SPEAKER["speaker_class"].isin(["borderline", "mismatch", "insufficient_speech"])] \
    if len(SPEAKER) else pd.DataFrame()
if len(REVIEW_SPEAKER):
    LOGGER.warning("ملفات للمراجعة البشرية (متحدث): %d", len(REVIEW_SPEAKER))

### محرك القياس المشترك — DNSMOS P.835 (Microsoft, ONNX محليًا)

**الهدف:** مقياس جودة إدراكي موضوعي يعمل محليًا بلا خدمات سحابية:

- **النموذجان الرسميان** من مستودع Microsoft DNS-Challenge: `sig_bak_ovr.onnx` (SIG/BAK/OVRL) و`model_v8.onnx` (P.808).
- **المنطق مطابق للمصدر الرسمي** `dnsmos_local.py`: نوافذ 9.01s بقفزات 1s، Melspec بمعاملات النموذج، ومعايرة Polyfit الرسمية (غير المخصصة).
- **دورة حياة الأوزان:** تنزيل مرة واحدة إلى `models_cache/dnsmos` — فشل التنزيل = تعطيل موثّق (لا مقياس مختلق أبدًا) وتعمل بقية الخط بقياسات طيفية.

**الاستخدامات داخل الخط:** بوابات التنقية/الفصل/الصدى (قرارات القبول المشروطة) + QA النهائي + مقارنة الفروع (Benchmark).

In [ ]:
# -*- coding: utf-8 -*-
"""محرك DNSMOS P.835 محلي (ONNX) — منطق مطابق للمصدر الرسمي مع تحقق دورة حياة الأوزان."""

DNSMOS_URLS = {
    "sig_bak_ovr.onnx": "https://raw.githubusercontent.com/microsoft/DNS-Challenge/master/DNSMOS/DNSMOS/sig_bak_ovr.onnx",
    "model_v8.onnx": "https://raw.githubusercontent.com/microsoft/DNS-Challenge/master/DNSMOS/DNSMOS/model_v8.onnx",
}
DNSMOS_DIR = PATHS["models_cache"] / "dnsmos"
_DNSMOS = {"primary": None, "p808": None, "failed": False}

def _download_dnsmos_weights() -> bool:
    DNSMOS_DIR.mkdir(parents=True, exist_ok=True)
    for name, url in DNSMOS_URLS.items():
        dest = DNSMOS_DIR / name
        if dest.exists() and dest.stat().st_size > 100_000:
            continue
        try:
            robust_download(url, dest, min_mb=0.0002, retries=2, timeout_s=60)
        except Exception as e:
            LOGGER.warning("تعذر تنزيل أوزان DNSMOS (%s): %s — المقياس يُعطّل", name, e)
            _DNSMOS["failed"] = True
            return False
    return True

def dnsmos_sessions():
    """جلسات ONNX (تحميل كسول مرة واحدة)."""
    if _DNSMOS["failed"] or not CONFIG["quality"]["dnsmos"]["enabled"]:
        return None
    if _DNSMOS["primary"] is None:
        if not CAPS["onnxruntime"]:
            LOGGER.warning("onnxruntime غير متاح — DNSMOS معطل (قرارات الجودة بالقياس الطيفي)")
            _DNSMOS["failed"] = True
            return None
        if not _download_dnsmos_weights():
            return None
        import onnxruntime as ort
        _DNSMOS["primary"] = ort.InferenceSession(str(DNSMOS_DIR / "sig_bak_ovr.onnx"),
                                                   providers=["CPUExecutionProvider"])
        _DNSMOS["p808"] = ort.InferenceSession(str(DNSMOS_DIR / "model_v8.onnx"),
                                                providers=["CPUExecutionProvider"])
        CAPS["dnsmos_weights"] = True
    return _DNSMOS

# معايرة Polyfit الرسمية (غير المخصصة) — من dnsmos_local.py بالمستودع الرسمي
_P_SIG = np.poly1d([-0.08397278, 1.22083953, 0.0052439])
_P_BAK = np.poly1d([-0.13166888, 1.60915514, -0.39604546])
_P_OVR = np.poly1d([-0.06766283, 1.11546468, 0.04602535])

def dnsmos_full(y, sr):
    """DNSMOS P.835 كامل: SIG/BAK/OVRL معايرة + P808 — أو None عند التعطيل الموثّق.
    
    المنطق مطابق للمصدر الرسمي: نوافذ 9.01s، قفز 1s، mel بمعاملات النموذج.
    """
    sess = dnsmos_sessions()
    if sess is None:
        return None
    y = y if y.ndim == 1 else y.mean(axis=1)
    if sr != 16000:
        y = soxr.resample(y, sr, 16000, quality="VHQ")
    fs = 16000
    L = int(9.01 * fs)
    if 0 < len(y) < L:
        # مقاطع أقصر من النافذة: حشو صمت (المصدر الرسمي يعمل بالإكمال الدوري وهو
        # مصمم لملفات > 9.01s — تطبيقه على المقاطع القصيرة يولّد وصلة مصطنعة
        # تعاقب DNSMOS عليها بلا سبب حقيقي). القياس يوسم n_hops=1.
        y = np.concatenate([y, np.zeros(L - len(y), dtype=np.float32)])
    n_hops = max(1, int(np.floor(len(y) / fs) - 9.01) + 1)
    sig_r, bak_r, ovr_r, p808 = [], [], [], []
    for idx in range(n_hops):
        seg = y[int(idx * fs): int((idx + 9.01) * fs)]
        if len(seg) < L:
            break
        x = np.ascontiguousarray(seg, dtype=np.float32)[np.newaxis, :]
        s, b, o = sess["primary"].run(None, {"input_1": x})[0][0]
        sig_r.append(s); bak_r.append(b); ovr_r.append(o)
        m = librosa.feature.melspectrogram(y=seg[:-160], sr=fs, n_fft=321, hop_length=160,
                                           n_mels=120, power=2.0)
        m_db = (librosa.power_to_db(m, ref=np.max) + 40) / 40
        p808.append(sess["p808"].run(None, {"input_1": m_db.T[np.newaxis].astype(np.float32)})[0][0][0])
    if not ovr_r:
        return None
    sig_m, bak_m, ovr_m = np.mean(sig_r), np.mean(bak_r), np.mean(ovr_r)
    return {"SIG": float(_P_SIG(sig_m)), "BAK": float(_P_BAK(bak_m)),
            "OVRL": float(_P_OVR(ovr_m)), "P808": float(np.mean(p808)),
            "n_hops": len(ovr_r)}

### قياس DNSMOS الفعلي لكل ملف — بوابة كل قرارات الجودة لاحقًا (فشل التنزيل = تعطيل موثّق، لا مقياس مختلق).

In [ ]:
# ── قياس DNSMOS لكل الملفات المتبقية (بوابة كل قرارات الجودة اللاحقة) ──
dnsmos_rows = []
_DNCFG = dict(CONFIG["quality"]["dnsmos"])
for r in tqdm(KEPT.itertuples(), total=len(KEPT), desc="DNSMOS P.835", unit="ملف"):
    def _dns():
        y, sr = sf.read(r.wav_path, dtype="float32")
        res = dnsmos_full(y, sr)
        del y
        return res or {"SIG": None, "BAK": None, "OVRL": None, "P808": None, "n_hops": 0}
    res = file_cache("dnsmos", r.fid, _DNCFG, _dns)
    dnsmos_rows.append({"fid": r.fid, "orig_name": r.orig_name, **res})
    for k in ("SIG", "BAK", "OVRL", "P808"):
        if res.get(k) is not None:
            STATE.metric(r.fid, "dnsmos", k, res[k])

DNSMOS_DF = pd.DataFrame(dnsmos_rows)
if DNSMOS_DF["OVRL"].notna().any():
    print(DNSMOS_DF[["orig_name", "SIG", "BAK", "OVRL", "P808"]].round(3).to_string(index=False))
    LOGGER.info("DNSMOS متوسط الأرشيف: OVRL=%.3f BAK=%.3f SIG=%.3f",
                DNSMOS_DF["OVRL"].mean(), DNSMOS_DF["BAK"].mean(), DNSMOS_DF["SIG"].mean())
else:
    LOGGER.warning("DNSMOS غير متاح — بوابات الجودة تعمل بالقياسات الطيفية فقط (موثّق)")

## STAGE 09 — Music, Noise & Interference Classification

**إصلاح الخلل التاريخي:** فحص عينة واحدة من منتصف الملف لا يكفي — الموسيقى قد تدخل ثوانٍ فقط. هنا **فحص متعدد المناطق**: البداية والوسط والنهاية (3 مناطق) + مناطق **الكلام الفعلي** (VAD-aware).

**الإشارات الطيفية لكل منطقة:** Flatness (آلة موسيقية مستمرة أعلى)، طاقة Chroma (نغمات موسيقية)، طاقة النطاق المنخفض (<120Hz — آلات bass)، Onset rate (إيقاع). **ليست حاسمة منفردة** — التصنيف النهائي يجمع الإشارات بقواعد موزونة، والمناطق تُوسم.

**التصنيفات السبعة (تحدد مسار المعالجة لاحقًا):**
1. `clean_speech` — لا تدخل.
2. `speech_plus_music` → فصل مشروط (STAGE 10).
3. `speech_plus_stationary_noise` → تنقية تكيفية (STAGE 11).
4. `speech_plus_nonstationary_noise` → تنقية تكيفية (فرع أقوى).
5. `speech_plus_competing_speaker` → **ليس فصل موسيقى!** — يُوسم للمعالجة المتخصصة/المراجعة (فصل الغناء ≠ عزل متحدث هدف).
6. `speech_plus_reverb` → إزالة صدى مشروطة (STAGE 12).
7. `mixed_conditions` → تُعالج الفروع بحسب الإشارات المكتشفة مجتمعة.

> **حدود معلنة:** كشف المتحدث المتنافس يعتمد على تحليل VAD (تداخل كثيف غير متوقع لنشأة الكلام + بصمة ECAPA منفصلة عند توفرها) — وسيلة استدلال أولية تُراجع بشريًا في الحالات الحرجة، لا حكمًا نهائيًا.

**مخرجات:** `CLASSIFICATION` (DataFrame لكل ملف: درجات المناطق + التصنيف) + `reports/noise_analysis.csv`.

In [ ]:
# -*- coding: utf-8 -*-
"""STAGE 09 — كشف وتصنيف متعدد المناطق: موسيقى/ضوضاء/صدى/متحدث متنافس."""

MUS = CONFIG["music"]["detection"]

def region_features(y, sr):
    """خصائص منطقة واحدة (زمنية/طيفية موجهة لكشف الموسيقى)."""
    if len(y) < sr * 0.5:
        return None
    import librosa as lr
    S = np.abs(lr.stft(y, n_fft=2048, hop_length=1024)) ** 2
    freqs = lr.fft_frequencies(sr=sr, n_fft=2048)
    low_ratio = float(S[freqs < 120].sum() / S.sum() + 1e-12)
    flatness = float(lr.feature.spectral_flatness(S=S).mean())
    chroma = lr.feature.chroma_stft(S=S ** 0.5, sr=sr)
    chroma_energy = float((chroma ** 2).sum(axis=0).mean())
    speech = spans_total(speech_spans(y, sr)) / max(len(y) / sr, 1e-9)
    return {"flatness": flatness, "low_ratio": low_ratio,
            "chroma_energy": chroma_energy, "speech_ratio": speech}

def reverb_tail_ms(y, sr, spans, noise_floor_db=None, hop_ms=20, drop_db=12.0, max_ms=700.0):
    """وسيط زمن هبوط الغلاف بعد الكلام — مؤشر صدى فيزيائي صادق.

    قيود القياس الصادقة:
    1) البحث محدود بـ max_ms (لا مشي حتى نهاية الملف) —
    2) إن كان هدف الهبوط (مستوى الكلام - drop_db) تحت أرضية الضوضاء فالصدى
       مقنّع بالضوضاء ولا يمكن قياسه — يُسجل 0 (لا قياس زائف)،
    3) عدم الهبوط خلال السقف = لا ذيل قابل للقياس (كلام متواصل) — لا يسجل.
    """
    if not spans:
        return 0.0
    hop = max(1, int(sr * hop_ms / 1000))
    n_frames = len(y) // hop
    if n_frames < 4:
        return 0.0
    env = 20 * np.log10(np.sqrt((y[: n_frames * hop].reshape(n_frames, hop) ** 2).mean(axis=1)) + 1e-12)
    tails = []
    max_f = int(max_ms / hop_ms)
    for s, e in spans:
        end_f = min(int(e * 1000 / hop_ms), n_frames - 1)
        if end_f <= 2 or end_f >= n_frames - 1:
            continue
        level = env[max(0, end_f - 5): end_f].mean()
        target = level - drop_db
        if noise_floor_db is not None and target < noise_floor_db + 3.0:
            continue  # الهدف تحت الأرضية — ذيل مقنّع بالضوضاء، لا قياس صادق ممكن
        f = end_f
        while f < min(n_frames, end_f + max_f) and env[f] > target:
            f += 1
        if f < min(n_frames, end_f + max_f) or (f < n_frames and env[min(f, n_frames-1)] <= target):
            tails.append(min((f - end_f) * hop_ms, max_ms))
    return float(np.median(tails)) if tails else 0.0

def analyze_regions(wav_path, n_regions=3):
    """فحص البداية/الوسط/النهاية + VAD كامل الملف مرة واحدة + ذيل الصدى الفيزيائي."""
    y, sr = sf.read(str(wav_path), dtype="float32")
    dur = len(y) / sr
    regions = {}
    if dur < 3 * 1.5:
        regions["all"] = region_features(y, sr)
    else:
        seg = dur / 3
        for i, name in enumerate(("start", "middle", "end")):
            s = int(i * seg * sr); e = int(min(len(y), (i + 1) * seg * sr))
            regions[name] = region_features(y[s:e], sr)
    full_spans = speech_spans(y, sr)
    return regions, y, sr, full_spans

def classify_file(regions, diag_row):
    """قواعد موزونة على مناطق متعددة — التصنيف السباعي + درجات."""
    feats = [v for v in regions.values() if v]
    if not feats:
        return {"condition": "insufficient_audio", "music_score": 0.0,
                "noise_score": 0.0, "reverb_score": 0.0}
    music_hits = sum(1 for f in feats if
                     f["flatness"] > MUS["flag_flatness"] or
                     f["low_ratio"] > MUS["flag_chroma_energy_ratio"])
    music_score = music_hits / len(feats)
    noise_score = 0.0
    nf = diag_row.get("noise_floor_est_db") if diag_row is not None else None
    if nf is not None and np.isfinite(nf) and nf is not None:
        noise_score = float(np.clip((-nf - 25) / 20, 0, 1))  # أرضية أعلى من -45dB → درجة
    # صدى: ذيل اضمحلال فيزيائي بعد الكلام (متانة أعلى من نعومة الغلاف المجردة)
    tail = diag_row.get("reverb_tail_ms") if diag_row is not None else None
    reverb_score = float(np.clip(((tail or 0.0) - 150.0) / 250.0, 0, 1)) if tail else 0.0
    # متحدث متنافس: إشارة استدلالية من تحليل العناقيد (STAGE 08) فقط —
    # مؤشرات onset الطاقية غير موثوقة (تحسب مقاطع الكلام نفسها) فلا تُستخدم.
    competing_hint = bool(diag_row.get("multi_speaker_hint")) if diag_row is not None else False
    # القرار: أسبقية للموسيقى (أخطر على التدريب) ثم الضوضاء/الصدى القياسية
    if music_score >= 0.5:
        cond = "speech_plus_music"
    elif competing_hint:
        cond = "speech_plus_competing_speaker"
    elif noise_score > 0.6 and reverb_score > 0.6:
        cond = "mixed_conditions"
    elif reverb_score > 0.6:
        cond = "speech_plus_reverb"
    elif noise_score > 0.6:
        cond = "speech_plus_stationary_noise" if noise_score < 0.85 else \
               "speech_plus_nonstationary_noise"
    else:
        cond = "clean_speech"
    return {"condition": cond, "music_score": round(music_score, 3),
            "noise_score": round(noise_score, 3), "reverb_score": round(reverb_score, 3),
            "reverb_tail_ms": round(float(tail or 0.0), 1),
            "competing_speaker_hint": competing_hint}

cls_rows = {}
_CLSCFG = {"music": CONFIG["music"]["detection"], "vad": dict(CONFIG["vad"]),
           "speaker_classes": (SPEAKER["speaker_class"].value_counts().to_dict()
                               if len(SPEAKER) else {}),
           "reverb": CONFIG["dereverb"]["detection"]}
for r in tqdm(KEPT.itertuples(), total=len(KEPT), desc="تصنيف الحالة", unit="ملف"):
    def _cls():
        drow = DIAG[DIAG["fid"] == r.fid].iloc[0].to_dict() if (DIAG["fid"] == r.fid).any() else None
        regions, y, sr, full_spans = analyze_regions(r.wav_path, n_regions=MUS["regions"])
        multi_hint = False
        if len(SPEAKER):
            sp = SPEAKER.set_index("fid").loc[r.fid] if r.fid in SPEAKER["fid"].values else None
            drow = dict(drow or {})
            drow["multi_speaker_hint"] = bool(sp is not None and sp["speaker_class"] in
                                              ("borderline", "mismatch", "insufficient_speech"))
        if drow is not None:
            _nf = drow.get("noise_floor_est_db")
            _nf = float(_nf) if _nf is not None and np.isfinite(float(_nf)) else None
            drow["reverb_tail_ms"] = reverb_tail_ms(y, sr, full_spans, noise_floor_db=_nf)
        out = classify_file(regions, drow)
        out["_tail_ms"] = drow.get("reverb_tail_ms") if drow else None
        del y
        return out
    res = file_cache("classify", r.fid, _CLSCFG, _cls)
    cls_rows[r.fid] = {"fid": r.fid, "orig_name": r.orig_name,
                       **{k: v for k, v in res.items() if not k.startswith("_")}}
    STATE.decision(r.fid, "classification", cls_rows[r.fid]["condition"],
                   f"music={cls_rows[r.fid]['music_score']} noise={cls_rows[r.fid]['noise_score']} "
                   f"reverb={cls_rows[r.fid]['reverb_score']} tail={res.get('_tail_ms')}")

CLASSIFICATION = pd.DataFrame(list(cls_rows.values()))
CLASSIFICATION.to_csv(PATHS["reports"] / "noise_analysis.csv", index=False)
print(CLASSIFICATION[["orig_name", "condition", "music_score", "noise_score",
                       "reverb_score"]].to_string(index=False))
print("\nتوزع الحالات:", CLASSIFICATION["condition"].value_counts().to_dict())

## STAGE 10 — Conditional Music Separation

**القاعدة:** **لا فصل إلا للأدلة** — الفصل يدمر التفاصيل الصوتية إذا لم يلزم. الملفات وحدها التي صُنّفت `speech_plus_music` تدخل هنا.

### دورة حياة النموذج (إصلاح إعادة التحميل لكل ملف)
- كائن `AudioSeparator` **واحد** يُنشأ مرة، والنماذج تُحمّل/تُفرَّغ عند الطلب عبر `MODEL_REGISTRY` — لا تنزيل متكرر ولا تحميل لكل استدعاء.
- النموذج يُختار من **قائمة النماذج المدعومة فعليًا** في الإصدار المثبت (لا أسماء مكتوبة يدويًا قد لا توجد) بترتيب `model_priority` من CONFIG.
- بعد كل استعمال: تفريغ GC/VRAM (احترام بيئات الذاكرة المحدودة).

### فحص نتائج الفصل (لا ثقة عمياء باسم "Vocals")
- تُفحص الـ stems الناتجة **فعليًا**: الملف موجود، الطول والـ Sample Rate مطابقان للمصدر (تفاوت >5% = فشل).
- يُختار الـ stem الصوتي بالاسم **ويُتحقق منه قياسًا**: أعلى نسبة كلام (VAD) + أعلى تشابه ECAPA مع المصدر — لا بالاسم وحده.
- **قبول الفصل مشروط:** DNSMOS OVRL للـ stem المختار يجب أن يتفوق على المصدر بهامش `accept_margin_dnsmos_ovrl`، وإلا يُرفض الفصل (الفصل الخاطئ أسوأ من الموسيقى).

**مخرجات:** `SEP_RESULTS` (لكل ملف: استُخدم نموذج؟ أي stem؟ مقاييس قبل/بعد) + `reports/separation_report.csv`.

In [ ]:
# -*- coding: utf-8 -*-
"""STAGE 10 — فصل موسيقى مشروط بدورة حياة نماذج صحيحة وفحص stems فعلي."""

_MODEL_REGISTRY = {"sep": None, "loaded_model": None}

def separator_instance():
    """Separator واحد لكل جلسة (دورة حياة النموذج — إصلاح الإنشاء لكل استدعاء)."""
    if _MODEL_REGISTRY["sep"] is None:
        if not CAPS["audio_separator"]:
            raise OptionalDependencyMissing("audio_separator غير متاح")
        from audio_separator.separator import Separator
        _MODEL_REGISTRY["sep"] = Separator(
            output_dir=str(PATHS["cache"]),
            model_file_dir=str(PATHS["models_cache"] / "sep"),
            log_level=30,
            normalization_threshold=1.0,   # لا ربح داخلي — ندير المستويات بأنفسنا بشفافية
        )
    return _MODEL_REGISTRY["sep"]

def load_priority_model():
    """اختيار أول نموذج متاح فعليًا من قائمة الإصدار المثبت (لا أسماء يدوية ميتة).
    
    الصيغة الفعلية لـ list_supported_model_files: {pack: {اسم: {filename, stems, ...}}}
    """
    sep = separator_instance()
    supported = set()
    try:
        for pack in sep.list_supported_model_files().values():
            if isinstance(pack, dict):
                for m in pack.values():
                    if isinstance(m, dict) and m.get("filename"):
                        supported.add(m["filename"])
    except Exception as e:
        LOGGER.warning("تعذر جلب قائمة النماذج المدعومة: %s", e)
    # وعيًا بالجهاز: على CPU نبدأ بنماذج ONNX الخفيفة — BS-RoFormer (كتلي عصبي)
    # أبطأ بعشرات المرات على CPU (ساعات لكل ملف) بينما MDX-Net ONNX عملي تمامًا
    priority = list(CONFIG["music"]["model_priority"])
    if DEVICE == "cpu":
        priority.sort(key=lambda m: not m.endswith(".onnx"))  # ONNX أولًا على CPU
    for name in priority:
        if not supported or name in supported:
            if _MODEL_REGISTRY["loaded_model"] != name:
                sep.load_model(name)
                _MODEL_REGISTRY["loaded_model"] = name
            return name
    raise ModelLoadError(f"لا نموذج فصل متاح من: {priority}")


def align_stem_to_source(y_stem, sr_stem, y_src, sr_src):
    """مطابقة stem مع المصدر: إعادة عينة (نماذج VR تعالج داخليًا عند 44.1k) + محاذاة الطول.
    
    لا يعتبر اختلاف المعدل فشلًا — هذه سلوك معروف لنماذج VR Arch — بل نعيده بصدق إلى معدل المصدر.
    """
    if sr_stem != sr_src:
        y_stem = soxr.resample(y_stem, sr_stem, sr_src, quality="VHQ")
    n = len(y_src)
    if len(y_stem) >= n:
        y_stem = y_stem[:n]
    else:
        y_stem = np.pad(y_stem, (0, n - len(y_stem)))
    return y_stem.astype(np.float32)

def separate_vocals(wav_path, fid):
    """فصل + فحص فعلي للـ stems: طول/SR متسقان، واختيار الـ stem الصوتي بالقياس."""
    out = PATHS["cache"] / f"sep_{fid}"
    out.mkdir(parents=True, exist_ok=True)
    model_name = load_priority_model()
    sep = separator_instance()
    sep.output_dir = str(out)
    stems = sep.separate(str(wav_path))
    src_info = sf.info(str(wav_path))
    y_src_ref, sr_ref = sf.read(str(wav_path), dtype="float32")
    cand_paths = [out / s for s in stems]
    candidates = []
    for p in cand_paths:
        if not p.exists():
            LOGGER.warning("stem مفقود: %s", p.name); continue
        info = sf.info(str(p))
        if abs(info.frames / info.samplerate - src_info.frames / src_info.samplerate) > max(0.5, 0.05 * src_info.frames / src_info.samplerate):
            LOGGER.warning("stem بطول غير متسق: %s", p.name); continue
        y, sr = sf.read(str(p), dtype="float32")
        y = y.mean(axis=1) if y.ndim > 1 else y
        y = align_stem_to_source(y, sr, y_src_ref, sr_ref)  # مطابقة المصدر (مدة+معدل)
        spans = speech_spans(y, sr_ref)
        sp_ratio = spans_total(spans) / max(len(y) / sr_ref, 1e-9)
        candidates.append({"path": p, "y": y, "sr": sr_ref, "speech_ratio": sp_ratio})
    if not candidates:
        raise ModelLoadError(f"فشل الفصل لـ {fid}: لا stems صالحة")
    # اختيار الـ stem الصوتي بالقياس: أعلى كلام
    candidates.sort(key=lambda c: -c["speech_ratio"])
    best = candidates[0]
    return {"model": model_name, "stem_path": str(best["path"]),
            "speech_ratio": best["speech_ratio"], "y": best["y"], "sr": best["sr"]}

def dnsmos_ovr_if_available(y, sr):
    """DNSMOS OVRL سريع إن توفرت الأوزان (يُعاد استخدامه في مراحل أخرى عبر dnsmos_full)."""
    res = dnsmos_full(y, sr)
    return res["OVRL"] if res else None

### تنفيذ الفصل المشروط: للملفات الموسومة فقط، مع مقارنة OVRL قبل/بعد وقرار قبول بهامش — ثم تفريغ الذاكرة.

In [ ]:
sep_results = []
if CONFIG["music"]["policy"] == "off":
    LOGGER.info("سياسة الفصل: معطلة من CONFIG")
else:
    music_fids = CLASSIFICATION.loc[CLASSIFICATION["condition"] == "speech_plus_music", "fid"].tolist()
    LOGGER.info("فصل مشروط: %d ملفًا عليه دليل موسيقى", len(music_fids))
    for r in tqdm(KEPT[KEPT["fid"].isin(music_fids)].itertuples(),
                  total=len(music_fids), desc="فصل الموسيقى", unit="ملف"):
        try:
            out_wav = PATHS["cache"] / f"sep_{r.fid}" / f"{r.fid}_vocals.wav"
            def _do():
                res = separate_vocals(r.wav_path, r.fid)
                atomic_write_audio(out_wav, res["y"], res["sr"], subtype="FLOAT")
                return {}
            cache_res = run_stage_cached(r.fid, "separation",
                                          {"model_priority": CONFIG["music"]["model_priority"]},
                                          _do, [out_wav],
                                          force="separation" in CONFIG["pipeline"]["force_reprocess"])
            y_sep, sr_sep = sf.read(str(out_wav), dtype="float32")
            ovr_sep = dnsmos_ovr_if_available(y_sep, sr_sep)
            y_src, sr_src = sf.read(r.wav_path, dtype="float32")
            ovr_src = dnsmos_ovr_if_available(y_src, sr_src)
            # قبول مشروط بهامش صافٍ (لا ثقة عمياء بعملية الفصل)
            if ovr_sep is not None and ovr_src is not None:
                accepted = (ovr_sep - ovr_src) >= CONFIG["music"]["accept_margin_dnsmos_ovrl"]
            else:
                accepted = None  # لا DNSMOS → مراجعة بشرية (لا قرار آلي)
            sep_results.append({"fid": r.fid, "orig_name": r.orig_name,
                                "separated": True, "model": _MODEL_REGISTRY["loaded_model"],
                                "dnsmos_ovr_before": round(ovr_src, 3) if ovr_src else None,
                                "dnsmos_ovr_after": round(ovr_sep, 3) if ovr_sep else None,
                                "accepted": accepted,
                                "vocals_path": str(out_wav)})
            STATE.decision(r.fid, "separation", "accepted" if accepted else
                           ("needs_review" if accepted is None else "rejected"),
                           f"ovr {ovr_src}→{ovr_sep}")
            del y_sep, y_src
        except (OptionalDependencyMissing, ModelLoadError) as e:
            LOGGER.error("تخطي فصل %s: %s", r.orig_name, e)
            sep_results.append({"fid": r.fid, "orig_name": r.orig_name, "separated": False,
                                "error": f"[{type(e).__name__}] {e}"})
        finally:
            memory_cleanup()

SEP_RESULTS = pd.DataFrame([{"fid": s.get("fid"), "orig_name": s.get("orig_name"),
                             "separated": s.get("separated", False),
                             "model": s.get("model"), 
                             "dnsmos_ovr_before": s.get("dnsmos_ovr_before"),
                             "dnsmos_ovr_after": s.get("dnsmos_ovr_after"),
                             "accepted": s.get("accepted"),
                             "error": s.get("error")} for s in sep_results])
if len(SEP_RESULTS):
    SEP_RESULTS.to_csv(PATHS["reports"] / "separation_report.csv", index=False)
    print(SEP_RESULTS.to_string(index=False))

# الفروع الصوتية المقبولة تحل محل المصدر في المراحل التالية (تتبع كامل في الـ Manifest)
VOCALS_OVERRIDE = {s["fid"]: s["vocals_path"] for s in sep_results if s.get("accepted")}
if VOCALS_OVERRIDE:
    LOGGER.info("فُصل وقُبل %d ملفًا — المصدر يستبدل بالصوت البشري (موثقًا)", len(VOCALS_OVERRIDE))

## STAGE 11 — Adaptive Speech Enhancement (Multi-Branch Engine)

**إصلاح جذري:** لا تنقية إجبارية على الجميع. التنقية **بوابة قياسية + فروع مستقلة + مقارنة عادلة**:

**1. البوابة (من يقفز إلى المعالجة أصلًا؟):** ملف يدخل فقط إذا وُجد دليل قياسي: `DNSMOS BAK < dnsmos_bak_below` **أو** أرضية ضوضاء مقدرة أعلى من الحد، مع **استبعاد كلام هادئ من البوابة السلبية** (Quiet Speech Guard: كلام هادئ كثير النسبة ليس ضوضاء).

**2. الفروع المستقلة (كلها تبدأ من نفس المصدر — مقارنة عادلة):**

| الفرع | المكتبة | التوفر |
|---|---|---|
| `uvr_denoise` | audio-separator + `UVR-DeNoise.pth` | مثبت هنا — يعمل |
| `noisereduce` | Spectral gating | مثبت هنا — يعمل |
| `deepfilternet` | DeepFilterNet3 (48kHz) | اختياري — يُكتشف إن وُجد |
| `mossformer2` | ClearerVoice MossFormer2_SE_48K | اختياري — يُكتشف إن وُجد |
| `resemble` | Resemble-Enhance denoiser | اختياري — يُكتشف إن وُجد |

**3. لكل فرع يُسجَّل Benchmark حقيقي:** زمن المعالجة، DNSMOS SIG/BAK/OVRL، بصمة ECAPA مقارنة بالمصدر، مسافة طيفية، والقص المستحدث — **النتيجة تُقبل فقط إذا فاضت الأصل بهامش موثق** وإلا **يبقى الأصل** (المبدأ: عند الشك احتفظ بالأصل).

**مخرجات:** `ENHANCE_REPORT` (كل فرع × ملف: مقاييس كاملة) + `reports/enhancement_report.csv` — والقرار النهائي يتخذه STAGE 13 (محرك الاختيار) وليس هنا.

In [ ]:
# -*- coding: utf-8 -*-
"""STAGE 11 — محرك التنقية التكيفية متعدد الفروع (كل فرع من نفس المصدر)."""
import tracemalloc as _tracemalloc

ND = CONFIG["noise"]["detection"]

def needs_denoise_gate(fid):
    """بوابة قياسية: دليل واحد يكفي — لكن كلام هادئ محمي من البوابة السلبية."""
    dn = DNSMOS_DF.set_index("fid").loc[fid] if fid in DNSMOS_DF["fid"].values else None
    d = DIAG.set_index("fid").loc[fid] if fid in DIAG["fid"].values else None
    srow = SPEECH_STATS.set_index("fid").loc[fid] if fid in SPEECH_STATS["fid"].values else None
    bak_ok = dn is not None and dn["BAK"] is not None and float(dn["BAK"]) < ND["dnsmos_bak_below"]
    floor = float(d["noise_floor_est_db"]) if d is not None and d.get("noise_floor_est_db") is not None else -np.inf
    floor_hit = np.isfinite(floor) and floor > ND["floor_estimate_above_db"]
    quiet = bool(srow is not None and srow["quiet_speech_flag"])
    reason = None
    if bak_ok:
        reason = f"DNSMOS BAK={dn['BAK']:.2f} < {ND['dnsmos_bak_below']}"
    elif floor_hit and not quiet:
        reason = f"noise_floor_est={floor:.1f}dB > {ND['floor_estimate_above_db']}dB"
    return bool(reason), reason

# ── تنفيذات الفروع (كل واحدة: (y, sr) → y' أو ترفع OptionalDependencyMissing) ──

def branch_uvr_denoise(y, sr):
    """UVR-DeNoise عبر audio-separator (نموذج VR Arch المخصص لإزالة الضوضاء)."""
    if not CAPS["audio_separator"]:
        raise OptionalDependencyMissing("audio_separator غير متاح")
    out = PATHS["cache"] / "branch_uvrdn"
    out.mkdir(parents=True, exist_ok=True)
    tmp_in = out / "in.wav"
    atomic_write_audio(tmp_in, y, sr, subtype="FLOAT")
    sep = separator_instance()
    sep.output_dir = str(out)
    if _MODEL_REGISTRY["loaded_model"] != "UVR-DeNoise.pth":
        sep.load_model("UVR-DeNoise.pth")
        _MODEL_REGISTRY["loaded_model"] = "UVR-DeNoise.pth"
    stems = sep.separate(str(tmp_in))
    clean_stems = [s for s in stems if "No Noise" in s or "No_Reverb" in s or "Instrumental" in s]
    pick = None
    for name in clean_stems:
        p = out / name
        if p.exists():
            pick = p
            break
    if pick is None:
        raise ModelLoadError("لم ينتج UVR-DeNoise stem نظيفًا")
    y2, sr2 = sf.read(str(pick), dtype="float32")
    y2 = y2.mean(axis=1) if y2.ndim > 1 else y2
    return align_stem_to_source(y2, sr2, y, sr)   # VR Arch: 44.1k داخليًا → إعادة لمعدل المصدر

def branch_noisereduce(y, sr):
    """Spectral gating (noisereduce) — خفيف ويعمل على CPU دائمًا."""
    if not CAPS["noisereduce"]:
        raise OptionalDependencyMissing("noisereduce غير متاح")
    import noisereduce as nr
    out = nr.reduce_noise(y=y, sr=sr, stationary=False,
                          prop_decrease=CONFIG["noise"].get("prop_decrease", 0.75))
    return np.asarray(out, dtype=np.float32)

### فروع التنقية الاختيارية الثقيلة

DeepFilterNet3 (48kHz) / MossFormer2_SE_48K (ClearerVoice) / resemble-enhance — واجهاتها الرسمية مع توحيد معدل العينة وحدود أمان الأطوال. تُكتشف إن وُجدت ولا تُثبّت تلقائيًا أبدًا (قرار مستخدم واعٍ).

In [ ]:
def branch_deepfilternet(y, sr):
    """DeepFilterNet3 (48kHz) — اختياري (يُكتشف إن وُجد — لا تثبيت تلقائي).
    
    الواجهة الرسمية df.enhance.enhance/init_df — يدعم 48kHz أصلاً.
    """
    if not CAPS.get("opt_deepfilternet"):
        raise OptionalDependencyMissing("DeepFilterNet غير مثبت (اختياري)")
    from df.enhance import enhance, init_df
    import torch
    model, df_state, _ = init_df()  # DeepFilterNet3 افتراضيًا
    wav = torch.from_numpy(np.ascontiguousarray(y, dtype=np.float32))
    if sr != df_state.sr():
        wav = torch.from_numpy(soxr.resample(y, sr, df_state.sr(), quality="VHQ"))
    with torch.inference_mode():
        out = enhance(model, df_state, wav)
    out = out.cpu().numpy().astype(np.float32)
    if isinstance(out, np.ndarray) and out.ndim == 2:
        out = out.mean(axis=0)
    if df_state.sr() != sr:
        out = soxr.resample(out, df_state.sr(), sr, quality="VHQ")
    return out.astype(np.float32)

_CV_CACHE = {}
def branch_mossformer2(y, sr):
    """ClearerVoice Studio — MossFormer2_SE_48K (اختياري — يُكتشف إن وُجد).
    
    يستخدم واجهة ClearVoice الرسمية مع معالجة مرنة لأشكال المخرجات
    (dict {اسم: tensor} أو tensor مباشرة) وتوحيد معدل العينة عند اللزوم.
    """
    if not CAPS.get("opt_clearervoice"):
        raise OptionalDependencyMissing("ClearerVoice غير مثبت (اختياري)")
    from clearvoice import ClearVoice
    import torch
    if "MossFormer2_SE_48K" not in _CV_CACHE:
        _CV_CACHE["MossFormer2_SE_48K"] = ClearVoice(model_name="MossFormer2_SE_48K")
    cv = _CV_CACHE["MossFormer2_SE_48K"]
    in_dir = PATHS["cache"] / "branch_cv_in"
    in_dir.mkdir(parents=True, exist_ok=True)
    tmp_in = in_dir / "in.wav"
    sr_in = sr if sr in (16000, 48000) else 48000  # النموذج يعمل عند 48k
    atomic_write_audio(tmp_in, y if sr_in == sr else soxr.resample(y, sr, sr_in, "VHQ"), sr_in, "FLOAT")
    out = cv.run(path=str(tmp_in), online=False, overwrite=True)
    if isinstance(out, dict):   # {filename: tensor|ndarray}
        out = next(iter(out.values()))
    if hasattr(out, "cpu"):
        out = out.detach().cpu().numpy()
    out = np.asarray(out, dtype=np.float32)
    if out.ndim == 2:
        out = out.mean(axis=0) if out.shape[0] > 1 else out[0]
    out = out.reshape(-1)
    if sr_in != sr:
        out = soxr.resample(out, sr_in, sr, quality="VHQ")
    if abs(len(out) - len(y)) / max(len(y), 1) > 0.05:
        out = out[: len(y)] if len(out) > len(y) else np.pad(out, (0, len(y) - len(out)))
    return out.astype(np.float32)

def branch_resemble(y, sr):
    """Resemble-Enhance — denoiser فقط (اختياري — يُكتشف إن وُجد).
    
    الواجهة الرسمية resemble_enhance.enhancer.inference.denoise.
    ملاحظة: وفق سياسة الخط نستخدم Denoiser وحده دون Enhancer —
    إعادة التوليد قد تُخترع تفاصيل صوتية اصطناعية (مخاطرة على الهوية).
    """
    if not CAPS.get("opt_resemble"):
        raise OptionalDependencyMissing("resemble-enhance غير مثبت (اختياري)")
    from resemble_enhance.enhancer.inference import denoise
    import torch
    dwav = torch.from_numpy(np.ascontiguousarray(y, dtype=np.float32))
    out, new_sr = denoise(dwav, sr, "cuda" if DEVICE == "cuda" else "cpu")
    out = out.cpu().numpy().astype(np.float32)
    if new_sr != sr:
        out = soxr.resample(out, int(new_sr), sr, quality="VHQ")
    return out.astype(np.float32)

BRANCH_IMPLS = {
    "uvr_denoise": branch_uvr_denoise,
    "noisereduce": branch_noisereduce,
    "deepfilternet": branch_deepfilternet,
    "mossformer2": branch_mossformer2,
    "resemble": branch_resemble,
}

def spectral_distance(y_a, y_b, sr):
    """مسافة طيفية (Log-Mel) بين نسختين محاذيتين زمنيًا — 0 = تطابق تام."""
    n = min(len(y_a), len(y_b))
    if n < sr * 0.2:
        return None
    import librosa as lr
    S = lr.feature.melspectrogram(y=y_a[:n], sr=sr, n_mels=64)
    S2 = lr.feature.melspectrogram(y=y_b[:n], sr=sr, n_mels=64)
    A, B = lr.power_to_db(S + 1e-10), lr.power_to_db(S2 + 1e-10)
    return float(np.mean(np.abs(A - B)))

def speaker_sim_between(y_a, sr_a, y_b, sr_b):
    """تشابه ECAPA بين إشارتين (قياس حفظ الهوية بين نسختين)."""
    if not CAPS["speechbrain"]:
        return None
    e1 = ecapa_embedding(y_a, sr_a)
    e2 = ecapa_embedding(y_b, sr_b)
    return cosine(e1, e2)

### تشغيل بوابة التنقية والفروع

بوابة قياسية (DNSMOS BAK أو أرضية ضوضاء، مع حماية الكلام الهادئ) ثم فروع مستقلة من نفس المصدر مع Benchmark حقيقي لكل فرع: زمن/ذاكرة/DNSMOS/تشابه هوية/مسافة طيفية.

In [ ]:
# ══ تشغيل الفروع للملفات التي عبرت البوابة ══
enh_rows, branch_audio = [], {}
if CONFIG["noise"]["policy"] == "off":
    LOGGER.info("سياسة التنقية: معطلة من CONFIG — لا فروع")
    GATE_FIDS = []
else:
    GATE_FIDS = []
    for r in KEPT.itertuples():
        hit, reason = needs_denoise_gate(r.fid)
        if hit:
            GATE_FIDS.append((r, reason))
            STATE.decision(r.fid, "enhance_gate", "process", reason)
        else:
            STATE.decision(r.fid, "enhance_gate", "skip", reason or "نظيف قياسيًا")
    LOGGER.info("بوابة التنقية: %d/%d ملفًا يحتاج فحص فروع",
                len(GATE_FIDS), len(KEPT))

for r, reason in tqdm(GATE_FIDS, desc="تنقية متعددة الفروع", unit="ملف"):
    src_path = VOCALS_OVERRIDE.get(r.fid, r.wav_path)
    y_src, sr = sf.read(str(src_path), dtype="float32")
    y_src = y_src.mean(axis=1) if y_src.ndim > 1 else y_src
    dn_src = dnsmos_full(y_src, sr) or {}
    for branch_name in CONFIG["noise"]["branches"]:
        impl = BRANCH_IMPLS.get(branch_name)
        if impl is None:
            continue
        out_path = PATHS["cache"] / "branches" / branch_name / f"{r.fid}.wav"
        _BRCFG = {"branch": branch_name, "src_sha": r.file_sha256,
                  "dnsmos": CONFIG["quality"]["dnsmos"],
                  "accept": CONFIG["noise"]["accept_margin"],
                  "prop_decrease": CONFIG["noise"].get("prop_decrease", 0.75)}
        def _branch_cached():
            def _run_branch():
                t0 = _time.time()
                _tracemalloc.start()
                try:
                    y_out = impl(y_src, sr)
                    peak_mem = _tracemalloc.get_traced_memory()[1] / 1e6
                finally:
                    _tracemalloc.stop()
                dur = _time.time() - t0
                dn = dnsmos_full(y_out, sr) or {}
                clip = detect_source_clipping(y_out)
                sim = speaker_sim_between(y_src, sr, y_out, sr)
                sdist = spectral_distance(y_src, y_out, sr)
                atomic_write_audio(out_path, y_out, sr, subtype="FLOAT")
                return {"path": str(out_path), "time_s": dur, "peak_ram_mb": peak_mem,
                        "dn": dn, "clip_ratio": clip["clip_ratio"], "sim": sim, "sdist": sdist}
            try:
                return _run_branch()
            except (OptionalDependencyMissing, NotImplementedError) as e:
                return {"status": f"skipped: {type(e).__name__}"}
            except Exception as e:
                STATE.log_error(r.fid, "enhance", type(e).__name__, "error", f"{branch_name}: {e}")
                return {"status": f"failed: {e}"}
        res = file_cache(f"enh_branch_{branch_name}", r.fid, _BRCFG, _branch_cached)
        if res.get("status") and not res.get("path"):
            row = {"fid": r.fid, "orig_name": r.orig_name, "branch": branch_name,
                   "gate_reason": reason, "status": res["status"]}
            _warn_once(f"فرع {branch_name}: {res['status']}")
        elif not res.get("path") or not _Path(res["path"]).exists():
            # مخرج الفرع مفقود (تلاشى الـ Cache) — إعادة السطر كفاشل موثق
            row = {"fid": r.fid, "orig_name": r.orig_name, "branch": branch_name,
                   "gate_reason": reason, "status": "failed: branch_output_missing"}
        else:
            branch_audio[(r.fid, branch_name)] = res["path"]
            row = {"fid": r.fid, "orig_name": r.orig_name, "branch": branch_name,
                   "gate_reason": reason,
                   "src_dnsmos_sig": dn_src.get("SIG"), "src_dnsmos_bak": dn_src.get("BAK"),
                   "src_dnsmos_ovrl": dn_src.get("OVRL"),
                   "out_dnsmos_sig": res["dn"].get("SIG"), "out_dnsmos_bak": res["dn"].get("BAK"),
                   "out_dnsmos_ovrl": res["dn"].get("OVRL"),
                   "speaker_sim_vs_src": res["sim"],
                   "spectral_distance": res["sdist"],
                   "out_clip_ratio": res["clip_ratio"],
                   "proc_time_s": round(res["time_s"], 2),
                   "peak_ram_mb": round(res["peak_ram_mb"], 1),
                   "status": "ok"}
            for k in ("out_dnsmos_sig", "out_dnsmos_bak", "out_dnsmos_ovrl", "speaker_sim_vs_src"):
                if row[k] is not None:
                    STATE.metric(r.fid, f"enh_{branch_name}", k, row[k])
        memory_cleanup()
        enh_rows.append(row)
    del y_src

ENHANCE_REPORT = pd.DataFrame(enh_rows)
ENHANCE_REPORT.to_csv(PATHS["reports"] / "enhancement_report.csv", index=False)
if len(ENHANCE_REPORT):
    _cols = [c for c in ["orig_name", "branch", "src_dnsmos_ovrl", "out_dnsmos_ovrl",
                         "speaker_sim_vs_src", "spectral_distance", "proc_time_s", "status"]
             if c in ENHANCE_REPORT.columns]
    print(ENHANCE_REPORT[_cols].round(3).to_string(index=False))
else:
    print("لا ملفات عبرت بوابة التنقية — الأصل يبقى (لا تنقية إجبارية)")

## STAGE 12 — Conditional Dereverberation

**القاعدة:** لا إزالة صدى على تسجيل سليم — إزالة الصدى من صوت نظيف تجففه وتدمر الطبيعية.

**البوابة (كلا الشرطين):**
1. **تصنيف** `speech_plus_reverb` أو `mixed_conditions` من STAGE 09 (مؤشرات نعومة الغلاف/الذيل)،
2. **معايرة قياسية بعد المعالجة**: يشغّل الخط `UVR-DeEcho-DeReverb` فرعًا مستقلًا ثم **يقيس الفرق** — الفرض يُقبل فقط إذا: DNSMOS OVRL تحسّن بهامش موثق **و** بصمة ECAPA تجاه المصدر ≥ `min_speaker_sim` **و** المسافة الطيفية معقولة.

هذا نمط "المعالجة المشروطة بالقياس البعدي" — لا قرار آلي أعمى ولا ثقة عمياء بالنموذج.

**مخرجات:** `DEREVERB_REPORT` + `reports/dereverb_report.csv` + `DEREVERB_OVERRIDE` (فروض مقبولة تحل محل المصدر، موثقة في الـ Manifest).

In [ ]:
# -*- coding: utf-8 -*-
"""STAGE 12 — إزالة صدى مشروطة بقياس قبل/بعد (لا قرار أعمى)."""
DER = CONFIG["dereverb"]

def branch_uvr_dereverb(y, sr):
    """UVR-DeEcho-DeReverb عبر audio-separator (نموذج VR مخصص)."""
    if not CAPS["audio_separator"]:
        raise OptionalDependencyMissing("audio_separator غير متاح")
    out = PATHS["cache"] / "branch_uvrdv"
    out.mkdir(parents=True, exist_ok=True)
    tmp_in = out / "in.wav"
    atomic_write_audio(tmp_in, y, sr, subtype="FLOAT")
    sep = separator_instance()
    sep.output_dir = str(out)
    if _MODEL_REGISTRY["loaded_model"] != DER["model"]:
        sep.load_model(DER["model"])
        _MODEL_REGISTRY["loaded_model"] = DER["model"]
    stems = sep.separate(str(tmp_in))
    pick = None
    for name in stems:  # الصوت البشري عادة "No Echo" أو الاسم بلا قوسين "(...)"
        if "(Echo)" not in name and "(Reverb)" not in name:
            p = out / name
            if p.exists():
                pick = p
                break
    if pick is None:
        raise ModelLoadError("لم ينتج UVR-DeEcho-DeReverb stem صوتيًا")
    y2, sr2 = sf.read(str(pick), dtype="float32")
    y2 = y2.mean(axis=1) if y2.ndim > 1 else y2
    return align_stem_to_source(y2, sr2, y, sr)

DRV = CONFIG["dereverb"]["detection"]
drv_rows, DEREVERB_OVERRIDE = [], {}
if CONFIG["dereverb"]["policy"] == "off":
    LOGGER.info("سياسة إزالة الصدى: معطلة من CONFIG")
else:
    drv_fids = [(r, c) for r in KEPT.itertuples()
                for c in [CLASSIFICATION[CLASSIFICATION["fid"] == r.fid]["condition"].values]
                if c and c[0] in ("speech_plus_reverb", "mixed_conditions")]
    LOGGER.info("بوابة الصدى: %d ملفًا عليه مؤشر", len(drv_fids))
    for r, _ in tqdm(drv_fids, desc="إزالة الصدى المشروطة", unit="ملف"):
        src_path = VOCALS_OVERRIDE.get(r.fid, r.wav_path)
        try:
            y_src, sr = sf.read(str(src_path), dtype="float32")
            y_src = y_src.mean(axis=1) if y_src.ndim > 1 else y_src
            y_drv = branch_uvr_dereverb(y_src, sr)
            dn_before = dnsmos_full(y_src, sr) or {}
            dn_after = dnsmos_full(y_drv, sr) or {}
            sim = speaker_sim_between(y_src, sr, y_drv, sr)
            sdist = spectral_distance(y_src, y_drv, sr)
            ovr_gain = (dn_after.get("OVRL") or 0) - (dn_before.get("OVRL") or 0)
            sim_ok = sim is None or sim >= DER["min_speaker_sim"]
            accepted = bool(dn_after and ovr_gain >= DER["accept_margin_dnsmos_ovrl"] and sim_ok
                            and (sdist is None or sdist < 12.0))
            out_path = PATHS["cache"] / "dereverb" / f"{r.fid}.wav"
            if accepted:
                atomic_write_audio(out_path, y_drv, sr, subtype="FLOAT")
                DEREVERB_OVERRIDE[r.fid] = str(out_path)
            drv_rows.append({"fid": r.fid, "orig_name": r.orig_name,
                             "dnsmos_ovr_before": round(dn_before.get("OVRL"), 3) if dn_before else None,
                             "dnsmos_ovr_after": round(dn_after.get("OVRL"), 3) if dn_after else None,
                             "ovrl_gain": round(ovr_gain, 3),
                             "speaker_sim": round(sim, 4) if sim is not None else None,
                             "spectral_distance": round(sdist, 2) if sdist is not None else None,
                             "accepted": accepted})
            STATE.decision(r.fid, "dereverb", "accepted" if accepted else "rejected",
                           f"ovrl_gain={ovr_gain:.3f} sim={sim}")
            del y_src, y_drv
        except (OptionalDependencyMissing, ModelLoadError) as e:
            LOGGER.error("تخطي إزالة صدى %s: %s", r.orig_name, e)
            drv_rows.append({"fid": r.fid, "orig_name": r.orig_name,
                             "accepted": False, "error": f"[{type(e).__name__}] {e}"})
        finally:
            memory_cleanup()

DEREVERB_REPORT = pd.DataFrame(drv_rows)
if len(DEREVERB_REPORT):
    DEREVERB_REPORT.to_csv(PATHS["reports"] / "dereverb_report.csv", index=False)
    print(DEREVERB_REPORT.to_string(index=False))
    LOGGER.info("فُرض وقُبل %d إزالة صدى من %d مرشحًا",
                len(DEREVERB_OVERRIDE), len(drv_fids))

## STAGE 13 — Enhancement Selection Engine (Multi-Criteria Decision)

**الفكرة الجوهرية:** لا يُفرض مسار واحد. لكل ملف عبر البوابة، يجمع المحرك **كل المرشحين**:

| المرشح | المصدر |
|---|---|
| `original` | المصدر (أو vocals المفصول المقبول) |
| `original_dereverb` | فرع إزالة الصدى إن قُبل |
| `uvr_denoise` / `noisereduce` / `deepfilternet` / … | فروع STAGE 11 |

**قرار موزون موثق (لا أوزان سرية):**

| المعيار | الوزن | الاتجاه | القياس |
|---|---|---|---|
| DNSMOS OVRL | 0.30 | أعلى أفضل | جودة إدراكية كلية |
| DNSMOS SIG | 0.25 | أعلى أفضل | جودة حفظ الكلام نفسه |
| Speaker Sim | 0.20 | أعلى أفضل (باب 0.80) | حفظ بصمة الهوية ECAPA |
| Spectral Distance | 0.15 | أدنى أفضل | أقل تغيير طيفي عن الأصل |
| Clipping | 0.10 | لا قص أفضل | سلامة الإشارة |

**قواعد حرجة (موثقة في كل قرار):**
1. **Original له أفضلية التعادل** (`accept_margin`): المرشح يتفوق صافيًا وإلا يبقى الأصل — عند الشك احتفظ بالأصل.
2. **Speaker Sim دون 0.80 = رفض مطلق** لأي مرشح معالج (الهوية فوق الجودة).
3. **تعارض قوي** (فرع يفوق في OVRL ويتخلف في SIG ≥0.3) → `needs_review` بديلًا من القرار الآلي.
4. الملفات التي لم تعبر البوابة: تظل على `original` — **معلنًا لا صامتًا**.

**مخرجات:** `SELECTION` (لكل ملف: المرشح الفائز + درجاته + سبب القرار) + `reports/selection_report.csv` + `SELECTED_PATH` (المسار الفائز لكل ملف).

In [ ]:
# -*- coding: utf-8 -*-
"""STAGE 13 — محرك اختيار متعدد المعايير بأوزان موثقة وقواعد حماية."""
SELECTION_WEIGHTS = {
    "dnsmos_ovrl": 0.30, "dnsmos_sig": 0.25, "speaker_sim": 0.20,
    "spectral_distance": 0.15, "clipping": 0.10,
}
ACCEPT = CONFIG["noise"]["accept_margin"]

def candidate_score(row):
    """درجة موزونة موثقة — تُعاد المعايير نفسها للتقرير (لا صندوق أسود)."""
    ovl = row.get("dnsmos_ovrl"); sig = row.get("dnsmos_sig")
    sim = row.get("speaker_sim"); sdist = row.get("spectral_distance")
    clip = row.get("clip_ratio", 0.0)
    s = 0.0
    if ovl is not None: s += SELECTION_WEIGHTS["dnsmos_ovrl"] * ((ovl - 2.5) / 2.5)
    if sig is not None: s += SELECTION_WEIGHTS["dnsmos_sig"] * ((sig - 2.5) / 2.5)
    if sim is not None: s += SELECTION_WEIGHTS["speaker_sim"] * ((sim - 0.5) / 0.5)
    if sdist is not None: s += SELECTION_WEIGHTS["spectral_distance"] * max(0.0, (15.0 - sdist) / 15.0)
    s += SELECTION_WEIGHTS["clipping"] * (1.0 - min(clip / 0.001, 1.0) if clip else 1.0)
    return round(s, 4)

def gather_candidates(fid):
    """كل المرشحين لملف: original + dereverb + فروع التنقية (بيانات موحدة)."""
    cands = []
    src_path = DEREVERB_OVERRIDE.get(fid) or VOCALS_OVERRIDE.get(fid)
    base_label = "original_dereverb" if fid in DEREVERB_OVERRIDE else (
        "original_vocals_separated" if fid in VOCALS_OVERRIDE else "original")
    src_path = src_path or VOCALS_OVERRIDE.get(fid) or KEPT[KEPT["fid"] == fid]["wav_path"].iloc[0]
    y, sr = sf.read(str(src_path), dtype="float32")
    dn = dnsmos_full(y, sr) or {}
    d = DIAG[DIAG["fid"] == fid]
    cands.append({"candidate": base_label, "path": str(src_path),
                  "dnsmos_ovrl": dn.get("OVRL"), "dnsmos_sig": dn.get("SIG"),
                  "dnsmos_bak": dn.get("BAK"),
                  "speaker_sim": 1.0,  # المرجع هو نفسه
                  "spectral_distance": 0.0,
                  "clip_ratio": float(d["clip_ratio"].iloc[0]) if len(d) else 0.0})
    del y
    for _, er in ENHANCE_REPORT[(ENHANCE_REPORT["fid"] == fid) &
                                 (ENHANCE_REPORT.get("status") == "ok")].iterrows() if len(ENHANCE_REPORT) else []:
        cands.append({"candidate": er["branch"], "path": branch_audio.get((fid, er["branch"])),
                      "dnsmos_ovrl": er.get("out_dnsmos_ovrl"), "dnsmos_sig": er.get("out_dnsmos_sig"),
                      "dnsmos_bak": er.get("out_dnsmos_bak"),
                      "speaker_sim": er.get("speaker_sim_vs_src"),
                      "spectral_distance": er.get("spectral_distance"),
                      "clip_ratio": er.get("out_clip_ratio", 0.0)})
    return cands

sel_rows, SELECTED_PATH, SELECTED_LABEL = [], {}, {}
for r in KEPT.itertuples():
    fid = r.fid
    cands = gather_candidates(fid)
    base = cands[0]
    if len(cands) == 1:
        decision, chosen, reason = "original", base, "لم يعبر بوابة المعالجة أو لا مرشحين صالحين"
    else:
        # ── البوابات الحاكمة أولاً (accept_margin من CONFIG — لا نتاج أوزان) ──
        passing, rejected = [], []
        for c in cands[1:]:
            sim = c.get("speaker_sim")
            ovr_gain = (c.get("dnsmos_ovrl") or -9) - (base.get("dnsmos_ovrl") or -9)
            if sim is not None and sim < ACCEPT["min_speaker_sim"]:
                rejected.append((c, f"هوية sim={sim:.3f} < {ACCEPT['min_speaker_sim']}"))
                continue
            if ovr_gain < ACCEPT["dnsmos_ovrl"]:
                rejected.append((c, f"هامش OVRL {ovr_gain:+.3f} < +{ACCEPT['dnsmos_ovrl']}"))
                continue
            if (c.get("clip_ratio") or 0) > CONFIG["quality"]["max_clipping_ratio"] * 5:
                rejected.append((c, f"قص مصدَّر={c.get('clip_ratio'):.4f}"))
                continue
            passing.append(c)
        if not passing:
            decision, chosen = "original", base
            reason = ("لا مرشح فاق هامش القبول "
                      f"({ACCEPT['dnsmos_ovrl']:+.2f} OVRL وsim>={ACCEPT['min_speaker_sim']}) — "
                      "يُبقى الأصل" +
                      (f" (مثال رفض: {rejected[0][1]})" if rejected else ""))
        else:
            # بين العابرين: أعلى درجة موزونة (الجودة ثم الهوية ثم أقل تشوّه)
            for c in passing:
                c["_score"] = candidate_score(c)
            passing.sort(key=lambda c: -c["_score"])
            best = passing[0]
            # تعارض قوي؟ (فائز OVRL لكن SIG يتراجع بشدة مقابل الأصل)
            sig_gap = (best.get("dnsmos_sig") or 5) - (base.get("dnsmos_sig") or 5)
            ovr_gain = (best.get("dnsmos_ovrl") or 0) - (base.get("dnsmos_ovrl") or 0)
            if ovr_gain >= 0.15 and sig_gap <= -0.30:
                decision, chosen, reason = "needs_review", base, (
                    f"تعارض قوي: {best['candidate']} رفع OVRL {ovr_gain:+.2f} لكن SIG {sig_gap:+.2f} — "
                    "قرار بشري (خطر على وضوح الكلام)")
            else:
                decision, chosen, reason = "enhanced", best, (
                    f"{best['candidate']} فاز: OVRL {ovr_gain:+.3f} (هامش +{ACCEPT['dnsmos_ovrl']}) "
                    f"sim={best.get('speaker_sim')}")
    if decision == "needs_review":
        STATE.log_error(fid, "selection", "ConflictDetected", "review", reason)
    else:
        STATE.decision(fid, "selection", decision, reason)
    SELECTED_PATH[fid] = chosen["path"]
    SELECTED_LABEL[fid] = chosen["candidate"]
    sel_rows.append({"fid": fid, "orig_name": r.orig_name, "selected": chosen["candidate"],
                     "decision": decision, "reason": reason,
                     "dnsmos_ovrl": chosen.get("dnsmos_ovrl"),
                     "dnsmos_sig": chosen.get("dnsmos_sig"),
                     "speaker_sim": chosen.get("speaker_sim"),
                     "spectral_distance": chosen.get("spectral_distance"),
                     "clip_ratio": chosen.get("clip_ratio"),
                     "score": candidate_score(chosen)})

SELECTION = pd.DataFrame(sel_rows)
SELECTION.to_csv(PATHS["reports"] / "selection_report.csv", index=False)
print(SELECTION[["orig_name", "selected", "decision", "dnsmos_ovrl",
                  "speaker_sim", "score"]].round(3).to_string(index=False))
print("\nتوزع القرارات:", SELECTION["decision"].value_counts().to_dict())

## STAGE 14 — Speaker Identity Preservation Validation

**الهدف:** بعد كل معالجة — وقبل التصدير — نتحقق أن النسخة المختارة **لم تغيّر هوية المتحدث**:

- لكل ملف مُعالج (قرار `enhanced` من STAGE 13 أو فرع فصل/صدى مقبول): بصمة ECAPA للنسخة النهائية مقابل بصمة المصدر الخام — **تشابه كوساين** يُقاس ويُسجّل.
- **تفسير حذر مضمّن:** التشابه المنخفض قد يكون بسبب ضوضاء المصدر (وليس تغيير هوية!) — لذلك النتيجة تُقرأ مع سياق الملف (أرضية الضوضاء، طول الكلام)، والقرارات الحرجة تذهب للمراجعة البشرية لا الاستبعاد الآلي.
- الملفات الباقية على الأصل: تظل تُفحص (بصمات قبل/بعد متطابقة بالبناء — تُسجَّل كمرجع سلامة).

**مخرجات:** `IDENTITY` + دمج النتائج في الـ Manifest + `reports/identity_preservation.csv`.

In [ ]:
# -*- coding: utf-8 -*-
"""STAGE 14 — تحقق حفظ هوية المتحدث عبر المعالجة (قبل/بعد)."""
id_rows = []
_IDCFG = {"min_sim": CONFIG["quality"]["min_speaker_sim_processed"],
          "label_policy": {f: SELECTED_LABEL.get(f) for f in KEPT["fid"]},
          "nf": {f: (float(DIAG.set_index("fid").loc[f, "noise_floor_est_db"])
                     if f in DIAG["fid"].values and DIAG.set_index("fid").loc[f, "noise_floor_est_db"] is not None
                     else None) for f in KEPT["fid"]}}
for r in tqdm(KEPT.itertuples(), total=len(KEPT), desc="حفظ الهوية", unit="ملف"):
    fid = r.fid
    sel_path = SELECTED_PATH[fid]
    try:
        def _ident():
            y_src, sr_src = sf.read(r.wav_path, dtype="float32")
            y_sel, sr_sel = sf.read(str(sel_path), dtype="float32")
            e_src = ecapa_embedding(y_src, sr_src)
            e_sel = ecapa_embedding(y_sel, sr_sel)
            sim = cosine(e_src, e_sel) if (e_src is not None and e_sel is not None) else None
            del y_src, y_sel
            return {"sim": None if sim is None else round(float(sim), 4)}
        res = file_cache("identity", fid, _IDCFG, _ident)
        sim = res["sim"]
        label = SELECTED_LABEL[fid]
        nf = _IDCFG["nf"][fid]
        nf = nf if nf is not None and np.isfinite(nf) else -np.inf
        ambiguous = bool(sim is not None and sim < CONFIG["quality"]["min_speaker_sim_processed"]
                         and np.isfinite(nf) and nf > -35.0)
        verdict = ("preserved" if (sim is None or sim >= CONFIG["quality"]["min_speaker_sim_processed"])
                   else ("ambiguous_needs_review" if ambiguous else "degraded"))
        id_rows.append({"fid": fid, "orig_name": r.orig_name, "selected": label,
                        "identity_sim": None if sim is None else round(sim, 4),
                        "verdict": verdict,
                        "note": "" if sim is None else
                                (f"أرضية ضوضاء {nf:.0f}dB — التشابه غير حاسم" if ambiguous else "")})
        STATE.metric(fid, "identity", "sim", sim if sim is not None else -1)
        if verdict != "preserved":
            STATE.decision(fid, "identity", verdict, f"sim={sim}")
    except OptionalDependencyMissing:
        id_rows.append({"fid": fid, "orig_name": r.orig_name, "selected": SELECTED_LABEL[fid],
                        "identity_sim": None, "verdict": "unchecked_no_engine",
                        "note": "speechbrain غير متاح — تحقق الهوية معطّل"})
    except Exception as e:
        LOGGER.warning("فحص هوية %s فشل: %s", r.orig_name, e)
        id_rows.append({"fid": fid, "orig_name": r.orig_name, "selected": SELECTED_LABEL[fid],
                        "identity_sim": None, "verdict": "error", "note": str(e)[:120]})

IDENTITY = pd.DataFrame(id_rows)
IDENTITY.to_csv(PATHS["reports"] / "identity_preservation.csv", index=False)
if len(IDENTITY):
    print(IDENTITY[IDENTITY["verdict"] != "preserved"].to_string(index=False)
          if (IDENTITY["verdict"] != "preserved").any() else "✓ الهوية محفوظة في كل الملفات المعالجة")
IDENTITY_REVIEW = IDENTITY[IDENTITY["verdict"].isin(["degraded", "ambiguous_needs_review"])]

## STAGE 15 — Intelligent Speech Activity Detection (للتقطيع)

**الهدف:** فواصل كلام **نهائية عالية الجودة** للتقطيع — Silero (هستيريسيس كامل التوصيل من `CONFIG['vad']`) أو البديل الطاقي، مع تدابير ذكاء إضافية:

1. **Hysteresis فعلي:** عتبتا صعود/هبوط (`threshold`/`neg_threshold`) تمنعان التذبذب عند الحد — **كل القيم من CONFIG** (إصلاح الترميز الثابت التاريخي مثل `min_silence_duration_ms=100`).
2. **حماية الكلام الهادئ:** امتدادات قصيرة متفرقة بين موجات كلام → تُدمج لا تُهمل (فحص مستوى الغلاف المحيط عند الحدّ).
3. **Ensemble عند الشك:** فجوة تنفصل عند Silero لكن مستوى طاقتها فوق الأرضية بوضوح → تُدمج (كلام مهموس محتمل مثل حروف احتكاكية).
4. الحد الأقصى للجملة الواحدة (`max_speech_s`) يُطبق عند الحاجة داخل محرك التقطيع (STAGE 16) لا هنا — فصل المسؤوليات.

**مخرجات:** `FINAL_SPANS` (لكل ملف: فواصل الكلام النهائية) — أساس التقطيع الكامل.

In [ ]:
# -*- coding: utf-8 -*-
"""STAGE 15 — VAD النهائي للتقطيع: هستيريسيس + Ensemble + حماية الكلام الهادئ."""
V = CONFIG["vad"]

def vad_with_energy_ensemble(y, sr, spans, fid):
    """Ensemble: فجوة تنفصل عند VAD لكن طاقتها فوق الأرضية بوضوح → تُدمج.
    
    يحمي الكلام المهموس/الاحتكاكي المنخفض RMS الذي يفلته Silero أحيانًا.
    """
    if not V["quiet_speech_rescue"] or len(spans) < 2:
        return spans
    d = DIAG[DIAG["fid"] == fid]
    nf = float(d["noise_floor_est_db"].iloc[0]) if len(d) and d["noise_floor_est_db"].iloc[0] is not None else -45.0
    if not np.isfinite(nf):
        nf = -45.0
    hop = int(sr * 0.01)
    merged = [list(spans[0])]
    for s, e in spans[1:]:
        gap_start, gap_end = merged[-1][1], s
        if gap_end <= gap_start:
            merged.append([s, e]); continue
        gs, ge = int(gap_start * sr), int(gap_end * sr)
        seg = y[gs: ge]
        if len(seg) < hop:
            merged.append([s, e]); continue
        gap_level = 20 * np.log10(np.sqrt(np.mean(seg ** 2)) + 1e-12)
        if gap_level > nf + 10:  # الفجوة أعلى من الأرضية بوضوح: كلام محتمل مهمل
            merged[-1][1] = e    # دمج
        else:
            merged.append([s, e])
    return [(s, e) for s, e in merged]

def final_speech_spans(y, sr, fid):
    """الفواصل النهائية: Silero هستيريسيس + Ensemble طاقي + حماية الكلام الهادئ."""
    spans = speech_spans(y, sr)  # كل العتبات من CONFIG.vad (الخلية الموحدة STAGE 07)
    spans = vad_with_energy_ensemble(y, sr, spans, fid)
    return spans

FINAL_SPANS = {}
_FVCFG = {"vad": dict(CONFIG["vad"]),
          "selected": {f: SELECTED_LABEL.get(f) for f in KEPT["fid"]}}
for r in tqdm(KEPT.itertuples(), total=len(KEPT), desc="VAD النهائي", unit="ملف"):
    def _fvad():
        y, sr = sf.read(str(SELECTED_PATH[r.fid]), dtype="float32")  # على النسخة المختارة
        spans = final_speech_spans(y, sr, r.fid)
        del y
        return [[round(a, 4), round(b, 4)] for a, b in spans]
    FINAL_SPANS[r.fid] = [tuple(s) for s in
                          file_cache("final_vad", r.fid, _FVCFG, _fvad)]
    STATE.metric(r.fid, "final_vad", "spans", len(FINAL_SPANS[r.fid]))
_total = sum(spans_total(v) for v in FINAL_SPANS.values())
LOGGER.info("VAD النهائي: %d جملة | إجمالي كلام %.1f دقيقة",
            sum(len(v) for v in FINAL_SPANS.values()), _total / 60)

## STAGE 16 — Utterance Segmentation (حدود طبيعية لا أهدأ نافذة)

**إصلاح جذري للتقطيع التاريخي:** لا تقطيع عند "أهدأ نافذة 300ms وسطية" عمياء. الخوارزمية هنا **متعددة الخطوات الملتزمة بـ CONFIG حرفيًا**:

1. **دمج قريب الجوار:** فواصل يفصلها أقل من `merge_gap_s` تُدمج (نفس الجملة عمليًا).
2. **تقسيم الطويل بأمان:** المقطع الأطول من `max_clip_s` يُقسم عند **أفضل حد طبيعي** داخل نافذة بحث `boundary_search_s` حول النقاط المرشحة: فجوات VAD الفعلية أولًا (صمت طبيعي بين كلمات)، ثم أدنى طاقة غير كلامية — **لا قطع داخل كلمة أو حرف احتكاكي** (الحد يُختار خارج فواصل الكلام الحالية).
3. **سياسة القصير (كل القيم موصولة):** أقصر من `drop_below_s` → يُهمل (موثقًا)؛ بين `drop_below_s` و`min_clip_s` → **يُحاول دمجه بجاره** إن جمعهما ≤ `max_clip_s`، وإلا يُحفظ مع وسم `short=True` إذا `keep_short_between` (لا إسقاط كلام مفيد آليًا).
4. **تدقيق الحدود:** انكماش داخل الصمت لتقليل الصمت الرائد/اللاحق ضمن سماحيات `leading/trailing_silence_max_s` — مع هامش `speech_pad_ms` محفوظ دائمًا للتنفس الختامي.

**كل حد من `CONFIG['segmentation']` موصول فعليًا** — لا إعدادات شكلية.

**مخرجات:** `SEGMENTS` (كل مقطع: fid + بداية/نهاية + مدة + أعلام التدقيق) + `reports/segmentation.csv`.

In [ ]:
# -*- coding: utf-8 -*-
"""STAGE 16 — تقطيع بحدود طبيعية موصول كليًا بـ CONFIG['segmentation']."""
SEG = CONFIG["segmentation"]
VADP = CONFIG["vad"]

def merge_close_spans(spans, max_gap):
    """دمج فواصل تفصلها فجوة أقل من max_gap (نفس الجملة عمليًا)."""
    if not spans:
        return []
    merged = [list(spans[0])]
    for s, e in spans[1:]:
        if s - merged[-1][1] <= max_gap:
            merged[-1][1] = e
        else:
            merged.append([s, e])
    return [(s, e) for s, e in merged]

def best_split_point(y, sr, spans, span_lo, span_hi, center, search, pad_s, raw_spans=None):
    """أفضل حد قطع طبيعي: فجوات VAD الأصلية أولًا ثم فجوة طاقة هادئة فعليًا.
    
    لا يكفي أن يكون الحد "الأهدأ نسبيًا" — يجب أن يكون فجوة حقيقية (طاقة منخفضة
    المطلق) وإلا قطعنا داخل ذيل مقطع كلمة متضائل. يعيد (cut, forced).
    """
    lo, hi = max(span_lo, center - search), min(span_hi, center + search)
    # 1) فجوات VAD الأصلية (pre-merge) — الصمت الطبيعي الحقيقي بين الجمل
    gaps = []
    src = raw_spans if raw_spans else spans
    for i in range(1, len(src)):
        g0, g1 = src[i - 1][1], src[i][0]
        if lo <= (g0 + g1) / 2 <= hi and g1 > g0:
            gaps.append((g0 + g1) / 2)
    if gaps:
        gaps.sort(key=lambda g: abs(g - center))
        return gaps[0], False
    # 2) فجوة طاقة هادئة فعليًا: الأدنى المحلي AND تحت عتبة هدوء مطلقة
    gs, ge = int(lo * sr), int(min(len(y), hi * sr))
    if ge - gs < int(0.05 * sr):
        return None, True
    hop = max(1, int(sr * 0.02))
    seg = y[gs:ge]
    n = max(1, len(seg) // hop)
    env = 20 * np.log10(np.sqrt((seg[: n * hop].reshape(n, hop) ** 2).mean(axis=1)) + 1e-12)
    guard = max(1, int(0.1 / 0.02))  # استبعاد حواف البحث (خطر قطع داخل كلمة)
    if len(env) <= 2 * guard + 1:
        return None, True
    inner = env[guard:-guard]
    quiet_floor = min(np.percentile(env, 25), np.percentile(inner, 20) + 12.0)
    cand = guard + int(np.argmin(inner))
    if env[cand] <= quiet_floor:      # فجوة هادئة فعلية — ليست ذيل كلمة متضائل
        return gs / sr + cand * 0.02, False
    return None, True   # لا فجوة آمنة في النافذة — قطع اضطراري موسوم

def split_overlong(span, y, sr, spans, max_len, search, pad_s, raw_spans=None):
    """تقسيم متكرر عند حدود طبيعية — كل مقطع ناتج ≤ max_len. القطع الاضطراري يوسم."""
    pieces, stack = [], [(span, False)]
    while stack:
        (s, e), forced = stack.pop()
        if e - s <= max_len:
            pieces.append(((s, e), forced)); continue
        center = s + (e - s) / 2
        cut, was_forced = best_split_point(y, sr, spans, s, e, center, search, pad_s,
                                          raw_spans=raw_spans)
        if cut is None or cut <= s + 0.3 or cut >= e - 0.3:  # لا حد آمن: قسم اضطراريًا
            cut, was_forced = center, True
        stack.append(((cut, e), forced or was_forced))
        stack.append(((s, cut), forced or was_forced))
    return pieces

def refine_boundaries(piece, y, sr, lead_max, trail_max, pad_s):
    """انكماش داخل الصمت: تقليل الصمت الرائد/اللاحق مع حفظ هامش التنفس."""
    s, e = piece
    gs, ge = int(s * sr), int(min(len(y), e * sr))
    hop = max(1, int(sr * 0.01))
    seg = y[gs:ge]
    if len(seg) < 3 * hop:
        return s, e, 0.0, 0.0
    n = len(seg) // hop
    env_db = 20 * np.log10(np.sqrt((seg[: n * hop].reshape(n, hop) ** 2).mean(axis=1)) + 1e-12)
    lead_n, trail_n = int(lead_max * 100), int(trail_max * 100)
    new_s = 0
    for i in range(min(lead_n, n - 1)):
        if env_db[i] > env_db[max(0, i - 5): i + 6].max() - 12 or env_db[i] > np.percentile(env_db, 60):
            new_s = i
            break
    new_e = n
    for i in range(min(trail_n, n - 1)):
        if env_db[n - 1 - i] > env_db[max(0, n - 6 - i): n - i + 5].max() - 12 or env_db[n - 1 - i] > np.percentile(env_db, 60):
            new_e = n - i
            break
    lead_s = new_s * 0.01
    trail_s = (n - new_e) * 0.01
    return s + lead_s, e - trail_s, lead_s, trail_s

### دوال الحدود الطبيعية للتقطيع

دمج قريب الجوار، البحث عن حد قطع آمن (فجوات VAD أولًا ثم أدنى طاقة بلا قطع داخل كلمة)، وتقسيم المقاطع الطويلة جدًا.

In [ ]:
def build_segments(y, sr, spans_raw, fid):
    """بناء المقاطع من فواصل VAD وفق سياسة CONFIG كاملة.
    
    ملاحظة جوهرية: نقاط القطع تُختار من الفواصل الأصلية (pre-merge) — دمج
    الجوار لا يجعل الفجوات الحقيقية بين الجمل تختفي من خيارات القطع.
    """
    padded = [(max(0.0, s - VADP["speech_pad_ms"] / 1000),
               min(len(y) / sr, e + VADP["speech_pad_ms"] / 1000))
              for s, e in spans_raw]
    spans = merge_close_spans(padded, SEG["merge_gap_s"])
    pieces = []
    for s, e in spans:
        if e - s < 0.05:
            continue
        pieces.extend(split_overlong((s, e), y, sr, spans, SEG["max_clip_s"],
                                     SEG["boundary_search_s"], VADP["speech_pad_ms"] / 1000,
                                     raw_spans=padded))
    # سياسة القصير (موصولة فعليًا بـ min_clip_s/drop_below_s): دمج بجار أو حفظ موسومًا
    final = []
    for piece, forced in pieces:
        s_, e_ = piece
        if final and (e_ - final[-1][0][0]) <= SEG["max_clip_s"] and \
           (s_ - final[-1][0][1]) <= SEG["merge_gap_s"] * 2 and \
           ((e_ - s_) < SEG["min_clip_s"] or
            (final[-1][0][1] - final[-1][0][0]) < SEG["min_clip_s"]):
            final[-1] = ((final[-1][0][0], e_), final[-1][1] or forced)
        else:
            final.append((piece, forced))
    segs = []
    for (s, e), forced in final:
        dur = e - s
        if dur < SEG["drop_below_s"]:
            STATE.decision(fid, "segmentation", "dropped_span", f"المدة {dur:.2f}s < drop_below_s")
            continue
        rs, re_, lead, trail = refine_boundaries((s, e), y, sr, SEG["leading_silence_max_s"],
                                                  SEG["trailing_silence_max_s"],
                                                  VADP["speech_pad_ms"] / 1000)
        segs.append({"fid": fid, "start_s": round(rs, 3), "end_s": round(re_, 3),
                     "dur_s": round(re_ - rs, 3),
                     "lead_sil_s": round(lead, 3), "trail_sil_s": round(trail, 3),
                     "short": dur < SEG["min_clip_s"],
                     "overlong_split": dur > SEG["max_clip_s"],
                     "forced_cut": bool(forced)})
    return segs

### بناء المقاطع النهائي

`build_segments` يطبق سياسة `CONFIG['segmentation']` كاملة: دمج، سياسة القصير المتصلة فعليًا بالحدود، حذف المتناهي الصغر، وتدقيق الحدود داخل الصمت.

In [ ]:
seg_rows = []
_SEGCFG = {"segmentation": dict(CONFIG["segmentation"]), "vad": dict(CONFIG["vad"])}
for r in tqdm(KEPT.itertuples(), total=len(KEPT), desc="التقطيع", unit="ملف"):
    def _seg():
        y, sr = sf.read(str(SELECTED_PATH[r.fid]), dtype="float32")
        segs = build_segments(y, sr, FINAL_SPANS[r.fid], r.fid)
        del y
        return segs
    segs = file_cache("segments", r.fid, _SEGCFG, _seg)
    for i, sg in enumerate(segs):
        sg["seg_id"] = f"{r.fid}_s{i:03d}"
        sg["orig_name"] = r.orig_name
        seg_rows.append(dict(sg))
    STATE.metric(r.fid, "segmentation", "segments", len(segs))

SEGMENTS = pd.DataFrame(seg_rows)
SEGMENTS.to_csv(PATHS["reports"] / "segmentation.csv", index=False)
if len(SEGMENTS):
    print(f"مقاطع: {len(SEGMENTS)} | مدة متوسطة {SEGMENTS['dur_s'].mean():.1f}s | "
          f"قصيرة {int(SEGMENTS['short'].sum())} | إجمالي {SEGMENTS['dur_s'].sum()/60:.1f} دقيقة")
    print(SEGMENTS["dur_s"].describe().round(2).to_string())
else:
    raise DataIntegrityError("لا مقاطع أُنتجت — فشل التقطيع (تحقق من إعدادات VAD)")

## STAGE 17 — Segment Integrity Verification

**الهدف:** كل مقطع يفحص فرديًا قبل التصدير (لا افتراض أن التقطيع سليم):

| الفحص | المعيار | الفعل عند الفشل |
|---|---|---|
| بداية/نهاية سليمة | لا طاقة قصّ/انقطاع حاد في أول/آخر 50ms | وسم `needs_review` |
| اكتمال الكلام | نسبة كلام داخل المقطع ≥ 50% | وسم `low_speech` |
| صمت رائد/لاحق | ضمن سماحيات CONFIG | تقليص إضافي |
| قص رقمي | بلا قص داخل المقطع | وسم `clipped` |
| الطول | ضمن [drop_below, max] | وسم حسب السياسة |
| تداخل | لا تداخل بين مقاطع نفس الملف | إصلاح تلقائي + توثيق |

**مخرجات:** `SEGMENTS` يُخصَّل بالفحوص + `seg_verify` أعلام لكل مقطع تدخل QA النهائي والـ Manifest.

In [ ]:
# -*- coding: utf-8 -*-
"""STAGE 17 — فحص سلامة كل مقطع فرديًا (اكتمال/حدود/قص/تداخل)."""
Q = CONFIG["quality"]

ver_rows = []
_y_cache = {}
for sg in tqdm(SEGMENTS.itertuples(), total=len(SEGMENTS), desc="فحص المقاطع", unit="مقطع"):
    fid = sg.fid
    if fid not in _y_cache:
        _y_cache.clear()  # ملف واحد في الذاكرة في كل مرة (حد موارد)
        y, sr = sf.read(str(SELECTED_PATH[fid]), dtype="float32")
        _y_cache[fid] = (y, sr)
    y, sr = _y_cache[fid]
    s, e = int(sg.start_s * sr), min(int(sg.end_s * sr), len(y))
    seg = y[s:e]
    flags = []
    if len(seg) < sr * 0.1:
        ver_rows.append({"seg_id": sg.seg_id, "verify": "rejected_empty", "flags": "empty"})
        continue
    # 1) حدود طبيعية — تُفحص بسياق المصدر لا بجسد المقطع وحده:
    #    حد طبيعي = يسبقه/يليه صمت فعلي في المصدر (وقفة جملة) وليس كلام متواصل.
    #    قص داخل كلمة = ما قبل الحد في المصدر بطاقة كلام.
    hop_v = max(1, int(0.02 * sr))
    n_f = max(2, len(seg) // hop_v)
    frames = seg[: n_f * hop_v].reshape(n_f, hop_v)
    fr_db = 20 * np.log10(np.sqrt((frames ** 2).mean(axis=1)) + 1e-12)
    speech_level = np.percentile(fr_db, 60)   # مستوى الكلام داخل المقطع
    ctx = int(0.06 * sr)                       # السياق المباشر: 60ms قبل/بعد الحد
    pre = y[max(0, s - ctx): s] if sg.start_s > 0 else None
    post = y[e: min(len(y), e + ctx)] if sg.end_s < len(y) / sr - 0.01 else None
    lvl = lambda a: 20 * np.log10(np.sqrt(np.mean(a ** 2)) + 1e-12)
    # وقفة كلمة طبيعية قد تليها كلمة جديدة خلال ~100ms — لذا يُفحص المباشر فقط
    if pre is not None and len(pre) >= ctx:
        if lvl(pre) > speech_level - 8.0:      # ما قبل الحد مباشرةً كلامٌ — قص داخل كلام
            flags.append("hard_onset")
    if post is not None and len(post) >= ctx:
        if lvl(post) > speech_level - 8.0:
            flags.append("hard_offset")
    # 2) اكتمال الكلام
    spans_in = [(a, b) for a, b in FINAL_SPANS[fid] if b > sg.start_s and a < sg.end_s]
    inner = sum(min(b, sg.end_s) - max(a, sg.start_s) for a, b in spans_in)
    speech_ratio = inner / max(sg.dur_s, 1e-9)
    if speech_ratio < 0.5:
        flags.append("low_speech")
    # 3) قص رقمي
    clip = detect_source_clipping(seg)
    if clip["clipped"]:
        flags.append("clipped")
    # 4) الطول
    if sg.short:
        flags.append("short")
    verdict = "pass" if not flags else (
        "needs_review" if any(f in flags for f in ("hard_onset", "hard_offset", "clipped")) else "pass_with_warnings")
    ver_rows.append({"seg_id": sg.seg_id, "verify": verdict,
                     "speech_ratio": round(speech_ratio, 3),
                     "clip_ratio": clip["clip_ratio"], "flags": ";".join(flags)})

VERIFIED = pd.DataFrame(ver_rows)
SEGMENTS = SEGMENTS.merge(VERIFIED, on="seg_id", how="left")
SEGMENTS.to_csv(PATHS["reports"] / "segmentation.csv", index=False)
_counts = SEGMENTS["verify"].value_counts().to_dict()
print("فحص المقاطع:", _counts)
if SEGMENTS["verify"].eq("rejected_empty").any():
    LOGGER.warning("مقاطع فارغة: %d — ستستبعد", int(SEGMENTS['verify'].eq('rejected_empty').sum()))

## STAGE 18 — Audio Format & Sample Rate Preparation

**السياسة (بلا خرافات):**
1. **لا إعادة عينة متكررة أبدًا:** التتبع الكامل لمعدل المصدر → تحويل **واحد** إلى معدل الهدف عند التصدير فقط (SoXR VHQ) — لا سلسلة تحويلات.
2. **رفع المعدل لا يعيد المعلومات:** ملف أصله 16kHz يُرفع إلى 48kHz دون ادعاء استعادة تفاصيل — يوسم في الـ Manifest (`upconverted: true`) ليعرف المستخدم حقيقة مصدره.
3. **التحقق بعد التحويل:** عدد العينات/المدة مطابقان للمرجع (تفاوت >2 عينة/نافذة = خطأ سلامة).

**مخرجات:** `resample_once()` الموثوقة + تتبع معدل المصدر لكل ملف يدخل الـ Manifest.

In [ ]:
# -*- coding: utf-8 -*-
"""STAGE 18 — إعادة عينة واحدة (SoXR VHQ) مع تحقق سلامة بعد التحويل."""

def resample_once(y, src_sr, dst_sr):
    """تحويل واحد VHQ مع تحقق طول/مدة — لا تسامح مع انزياح العينات."""
    if src_sr == dst_sr:
        return y, False
    out = soxr.resample(np.asarray(y, dtype=np.float32), src_sr, dst_sr, quality="VHQ")
    # تحقق السلامة: الطول المتوقع ± نافذة إعادة العينة
    expected = int(round(len(y) * dst_sr / src_sr))
    if abs(len(out) - expected) > max(4, int(0.001 * dst_sr)):
        raise DataIntegrityError(
            f"إعادة عينة خرقت الطول: {len(out)} مقابل متوقع {expected} "
            f"({src_sr}→{dst_sr})")
    if not np.all(np.isfinite(out)):
        raise DataIntegrityError("NaN/Inf بعد إعادة العينة")
    return out.astype(np.float32), True

def prepare_for_export(y, src_sr):
    """تجهيز نهائي: تحويل واحد إلى TARGET_SR + تتبع حقيقة التحويل."""
    out, converted = resample_once(y, src_sr, TARGET_SR)
    return out, {"resampled": converted, "src_sr": src_sr, "dst_sr": TARGET_SR}

LOGGER.info("معدل الهدف: %d Hz (نمط %s) — التحويل الواحد يحدث في STAGE 19 فقط عند الحاجة",
            TARGET_SR, CONFIG["audio"]["rvc_profile"])
SRC_SR_MAP = dict(zip(REGISTRY["fid"], REGISTRY["dec_sr"]))

## STAGE 19 — Final Dataset Export

**شجرة المخرجات المفصولة بدقة (لا خلط أنواع مختلفة في مجلد واحد):**

```
RVC_Audio_Dataset/
├── master/original_preserved/     # المصدر الخام كما فُك (بمعدله الأصلي) — لا يُمسّ
├── master/processed/              # Master المعالج (النسخ المختارة) بمعدل الهدف
├── clips/accepted/                # مقاطع التدريب المقبولة (TARGET_SR, PCM bit_depth)
├── clips/needs_review/            # مقاطع الحالات الحدودية — قرار بشري
├── evaluation/held_out/           # تسجيلات مُحجوزة لاختبار النموذج بعد التدريب
├── quarantine/{wrong_speaker,corrupted,low_quality,duplicate,uncertain}/
├── reports/…csv                   # كل تقارير المراحل
├── visualizations/…png            # الرسوم
├── manifest.json                  # Provenance كامل لكل مخرج
└── dataset_index.csv              # فهرس قابل للفرز
```

**قرارات التصدير الموصلة فعليًا (لا إعدادات شكلية):**
- `bit_depth` (16/24) يتحكم حرفيًا في subtype التصدير `PCM_16/PCM_24`,
- حماية True-Peak: تخفيض **فقط** إذا تجاوزت القمة الحقيقية السقف (لا ربح/تطبيع إجباري),
- إدارة الجهارة الاختيارية `gain_only`: ربح ساكن محدود للشواذ فقط (بلا تطبيع LUFS إجباري),
- كل ملف يُقرأ **بعد الكتابة** ويتحقق (فك → فحص sr/channels/subtype/NaN) قبل اعتباره ناجحًا.

**مخرجات:** مجلدات كاملة + تقرير تصدير لكل ملف.

In [ ]:
# -*- coding: utf-8 -*-
"""STAGE 19 — التصدير النهائي: master + clips مع حماية قمم فعلية وتحقق بعد الكتابة."""
AUD = CONFIG["audio"]
SUBTYPE = f"PCM_{AUD['bit_depth']}"

def protect_true_peak(y, sr):
    """حماية True-Peak: تخفيض فقط عند تجاوز السقف (لا ربح إجباري أبدًا)."""
    tp_db = true_peak_db(y, sr)
    if tp_db > AUD["true_peak_ceiling_db"]:
        gain = 10 ** ((AUD["true_peak_ceiling_db"] - tp_db) / 20) * 0.995
        LOGGER.debug("تخفيض قمة حقيقية: %.2f → ≤ %.2f dBFS", tp_db, AUD["true_peak_ceiling_db"])
        return y * gain, -tp_db
    return y, 0.0

def optional_gain(y, sr, lufs_default=None):
    """إدارة جهارة اختيارية: gain_only — ربح ساكن محدود للشواذ فقط (لا تطبيع قسري)."""
    L = AUD["loudness"]
    if L["mode"] != "gain_only" or not L["target_lufs"]:
        return y, 0.0
    cur = lufs_default if lufs_default is not None else float(pyloudness_lufs(y, sr))
    delta = L["target_lufs"] - cur
    delta = float(np.clip(delta, -L["max_boost_db"], L["max_boost_db"]))
    if abs(delta) < 0.5:
        return y, 0.0
    y2 = y * (10 ** (delta / 20))
    tp = true_peak_db(y2, sr)
    if tp > AUD["true_peak_ceiling_db"]:
        y2 *= 10 ** ((AUD["true_peak_ceiling_db"] - tp) / 20) * 0.995
        delta = 20 * np.log10(max(np.max(np.abs(y2)), 1e-9) / max(np.max(np.abs(y)), 1e-9))
    return y2, delta

def pyloudness_lufs(y, sr):
    """تقدير LUFS مدمج بوزن K المرحلي (high-shelf ITU-R BS.1770) — يوسم تقديريًا، ليس قياس بث معتمدًا."""
    from scipy.signal import lfilter
    # مرشح الرف العالي لوزن K (معاملات BS.1770 المرحلية الرقمية عند 48kHz تقريبًا)
    b1 = np.array([1.53512485958697, -2.69169618940638, 1.19839281085285])
    a1 = np.array([1.0, -1.69065929318241, 0.73248077421585])
    w = lfilter(b1, a1, y, axis=0)
    # بوابة العتبة -70 LUFS (BS.1770-4 مبسطة)
    block = int(0.4 * sr)
    if len(w) < block:
        return -np.inf
    n = len(w) // block
    ms = (w[: n * block].reshape(n, block) ** 2).mean(axis=1)
    loud = -0.691 + 10 * np.log10(ms + 1e-12)
    keep = loud > -70.0
    if not keep.any():
        return -np.inf
    # مدمج: وزن كل البلوكات الفعالة
    return float(-0.691 + 10 * np.log10((10 ** (loud[keep] / 10)).mean() + 1e-12))

def verify_written(path, expect_sr, expect_ch=1, expect_subtype=None):
    """قراءة ما بعد الكتابة — الملف لا يعتبر ناجحًا قبل فحصه فعليًا."""
    info = sf.info(str(path))
    y, s = sf.read(str(path), dtype="float32")
    ok = (s == expect_sr)
    problems = []
    if info.channels != expect_ch: problems.append(f"channels={info.channels}")
    if expect_subtype and info.subtype != expect_subtype: problems.append(f"subtype={info.subtype}")
    if not np.all(np.isfinite(y)): problems.append("nan_inf")
    if len(y) == 0: problems.append("empty")
    return (len(problems) == 0), problems, info

### تنفيذ التصدير: Master (أصل محفوظ + معالج) ثم Clips (accepted/needs_review) مع تحقق بعد الكتابة لكل ملف.

In [ ]:
E = CONFIG["export"]
export_rows = []

# ── A. Master: المصدر الخام محفوظ + المعالج بمعدل الهدف ──
if E["master"]:
    orig_dir = PATHS["export_root"] / "master" / "original_preserved"
    proc_dir = PATHS["export_root"] / "master" / "processed"
    for d in (orig_dir, proc_dir):
        d.mkdir(parents=True, exist_ok=True)
    for r in KEPT.itertuples():
        # 1) الأصل المفكوك (معدله الأصلي — لا تحويل)
        src_dec = PATHS["work"] / "01_decoded" / f"{r.fid}.wav"
        dst = orig_dir / f"{r.fid}.wav"
        if not dst.exists():
            _shutil.copy2(src_dec, dst)
        # 2) المعالج: النسخة المختارة → معدل الهدف + bit-depth + حماية قمم
        y, sr = sf.read(str(SELECTED_PATH[r.fid]), dtype="float32")
        y, rr = prepare_for_export(y, sr)
        y, applied = protect_true_peak(y, TARGET_SR)
        y, gain = optional_gain(y, TARGET_SR)
        out = proc_dir / f"{r.fid}.wav"
        atomic_write_audio(out, y, TARGET_SR, subtype=SUBTYPE)
        ok, problems, _info = verify_written(out, TARGET_SR, 1, SUBTYPE)
        export_rows.append({"kind": "master", "fid": r.fid, "orig_name": r.orig_name,
                            "path": str(out), "sr": TARGET_SR, "subtype": SUBTYPE,
                            "verify": "pass" if ok else "fail:" + ",".join(problems),
                            "true_peak_reduction_db": round(applied, 2) if applied else None,
                            "gain_db": round(gain, 2) if gain else None})
        del y

# ── B. Clips: المقبولة + الحالات الحدودية للمراجعة ──
if E["clips"]:
    acc_dir = PATHS["export_root"] / "clips" / "accepted"
    rev_dir = PATHS["export_root"] / "clips" / "needs_review"
    for d in (acc_dir, rev_dir):
        d.mkdir(parents=True, exist_ok=True)
    src_y = {}
    for sg in tqdm(SEGMENTS.itertuples(), total=len(SEGMENTS), desc="تصدير المقاطع", unit="مقطع"):
        if sg.verify == "rejected_empty":
            continue
        if sg.fid not in src_y:
            src_y.clear()
            y, sr = sf.read(str(SELECTED_PATH[sg.fid]), dtype="float32")
            src_y[sg.fid] = (y, sr)
        y, sr = src_y[sg.fid]
        s, e = int(sg.start_s * sr), min(int(sg.end_s * sr), len(y))
        clip = y[s:e]
        clip, rr = prepare_for_export(clip, sr)
        clip, applied = protect_true_peak(clip, TARGET_SR)
        clip, gain = optional_gain(clip, TARGET_SR)
        # وجهة المقطع: مقبول أو مراجعة (أعلام الفحص + فئات الجودة لاحقًا في QA)
        needs_review = (sg.verify == "needs_review") or bool(sg.short and not SEG["keep_short_between"])
        out = (rev_dir if needs_review else acc_dir) / f"{sg.seg_id}.wav"
        atomic_write_audio(out, clip, TARGET_SR, subtype=SUBTYPE)
        ok, problems, _info = verify_written(out, TARGET_SR, 1, SUBTYPE)
        export_rows.append({"kind": "clip_accepted" if not needs_review else "clip_review",
                             "fid": sg.fid, "seg_id": sg.seg_id, "orig_name": sg.orig_name,
                             "path": str(out), "sr": TARGET_SR, "subtype": SUBTYPE,
                             "dur_s": round(sg.dur_s, 2),
                             "verify": "pass" if ok else "fail:" + ",".join(problems),
                             "true_peak_reduction_db": round(applied, 2) if applied else None,
                             "gain_db": round(gain, 2) if gain else None})
    del src_y

EXPORT_REPORT = pd.DataFrame(export_rows)
EXPORT_REPORT.to_csv(PATHS["reports"] / "export_report.csv", index=False)
_n = EXPORT_REPORT["verify"].str.startswith("pass").sum()
LOGGER.info("التصدير: %d/%d ملفًا اجتاز التحقق بعد الكتابة", _n, len(EXPORT_REPORT))
if _n < len(EXPORT_REPORT):
    LOGGER.error("ملفات فشلت في التحقق بعد الكتابة: %s",
                 EXPORT_REPORT[~EXPORT_REPORT["verify"].str.startswith("pass")]["path"].tolist())
print(EXPORT_REPORT.groupby("kind").agg(n=("path", "count"),
      dur=("dur_s", "sum")).round(1).to_string() if "dur_s" in EXPORT_REPORT else
      EXPORT_REPORT["kind"].value_counts().to_string())

## STAGE 20 — Multi-Dimensional Quality Assurance (كل الملفات لا عينة منها)

**إصلاح جذري:** لا فحص "أول خمسة ملفات" — **كل ملف مخرَج يُفحص من القرص فعليًا**.

### طبقات الفحص الخمس (لكل ملف)
1. **تقني:** وجود/قراءة/معدل/قنوات/subtype/مدة/NaN/DC/قمة/**True-Peak**/قص تصدير.
2. **كلامي:** وجود كلام فعلي داخل المقطع (VAD)، نسبة الكلام، اكتمال الحدود.
3. **متحدث:** تشابه بصمة ECAPA مع المصدر الخام (فحص قبل/بعد حقيقي، لا ادعاء).
4. **إدراكي:** DNSMOS SIG/BAK/OVRL مع عتبات CONFIG (reject/review).
5. **طيفي:** مسافة طيفية عن المصدر + سلامة الترددات العالية (لا "تحسين" أكل التفاصيل).

### فئات النتيجة الخمس (لا ثنائية نجاح/فشل فجة)
`PASS` / `PASS_WITH_WARNINGS` / `NEEDS_REVIEW` / `REJECTED` / `PROCESSING_FAILED`

**قاعدة صارمة:** أي مخرج بلا سجل QA = خطأ سلامة (لا ملف يفلت من الفحص).

**مخرجات:** `QA` (DataFrame لكل ملف + أسبابه) + `reports/final_quality.csv`.

In [ ]:
# -*- coding: utf-8 -*-
"""STAGE 20 — ضمان جودة شامل لكل الملفات المصدرة من القرص فعليًا."""
Q = CONFIG["quality"]

def qa_one_file(row, y_src_cache=None):
    """فحص خمسي الأبعاد لملف مصدر واحد (master أو clip)."""
    p = Path(row["path"])
    if not p.exists():
        return {"result": "PROCESSING_FAILED", "reasons": "file_missing"}
    try:
        info = sf.info(str(p))
        y, sr = sf.read(str(p), dtype="float32")
    except Exception as e:
        return {"result": "PROCESSING_FAILED", "reasons": f"unreadable: {e}"}
    reasons, warnings = [], []
    # 1) تقني
    if sr != TARGET_SR: reasons.append(f"sr={sr}")
    if info.channels != 1: reasons.append(f"ch={info.channels}")
    if info.subtype != SUBTYPE: reasons.append(f"subtype={info.subtype}")
    if not np.all(np.isfinite(y)): reasons.append("nan_inf")
    dc = float(np.mean(y)); 
    if abs(dc) > Q["max_dc_offset"]: warnings.append(f"dc={dc:.3f}")
    tp = true_peak_db(y, sr)
    if tp > Q["max_true_peak_db"]: warnings.append(f"true_peak={tp:.2f}")
    clip = detect_source_clipping(y)
    if clip["clip_ratio"] > Q["max_clipping_ratio"]: reasons.append(f"export_clipping={clip['clip_ratio']:.4f}")
    if len(y) / sr < 0.2: reasons.append("too_short")
    # 2) كلامي + 5) طيفي + 3) متحدث
    spans = speech_spans(y, sr)
    speech_ratio = spans_total(spans) / max(len(y) / sr, 1e-9)
    if speech_ratio < 0.3 and row["kind"].startswith("clip"):
        reasons.append(f"no_speech={speech_ratio:.2f}")
    dn = dnsmos_full(y, sr) or {}
    ovr = dn.get("OVRL")
    if row["kind"] == "master" and ovr is not None:
        # ملف كامل: قياس DNSMOS صادق هنا (بلا حشو) — أرضية القمامة الفعلية
        if ovr < Q["min_file_dnsmos_ovrl"]:
            reasons.append(f"dnsmos_ovrl={ovr:.2f} (ملف تحت أرضية الجودة)")
    if row["kind"].startswith("clip") and ovr is not None:
        # مقارنة صادقة مثل بمثل: المقطع مقابل **نفس مدته في مصدره المختار** —
        # DNSMOS P.835 يقيس النوافذ 9.01s، والمقاطع الأقصر تُقاس بحشو صمت، فيهبط
        # الرقم هبوطًا منهجيًا لا علاقة له بالجودة. لذا لا تُستخدم أرضية مطلقة
        # على المقاطع المبطنة — بل يُكشف التدهور الحقيقي (تصدير/تحويل/تكميم):
        seg = row.get("_seg_bounds")
        span_ovr = _src_span_dnsmos(fid, seg) if seg else None
        if span_ovr is not None and ovr < span_ovr - Q["degradation_reject"]:
            reasons.append(f"dnsmos_ovrl={ovr:.2f} << نفس المدى بالمصدر {span_ovr:.2f} (تدهور تصدير)")
        elif span_ovr is not None and ovr < span_ovr - Q["degradation_review"]:
            warnings.append(f"dnsmos_ovrl={ovr:.2f} < نفس المدى بالمصدر {span_ovr:.2f}")
    # الهوية: المقطع يُقارن ببصمة **نفس مدته** في مصدره (مثل بمثل) —
    # المقارنة بالملف الكامل زائفة إحصائيًا للنوافذ القصيرة
    id_sim = None
    if CAPS["speechbrain"] and y_src_cache is not None:
        e_out = ecapa_embedding(y, sr)
        id_sim = cosine(e_out, y_src_cache) if (e_out is not None and y_src_cache is not None) else None
        if id_sim is not None and id_sim < Q["min_speaker_sim_processed"]:
            reasons.append(f"identity_sim={id_sim:.2f}")
    # الحكم
    result = ("REJECTED" if reasons else
              "NEEDS_REVIEW" if warnings else "PASS")
    return {"result": result, "reasons": ";".join(reasons), "warnings": ";".join(warnings),
            "true_peak_db": round(tp, 2) if np.isfinite(tp) else None,
            "dc_offset": round(dc, 4), "speech_ratio": round(speech_ratio, 3),
            "dnsmos_ovrl": round(ovr, 3) if ovr else None,
            "dnsmos_sig": round(dn["SIG"], 3) if dn.get("SIG") else None,
            "identity_sim": round(id_sim, 4) if id_sim is not None else None}

_SPAN_CACHE = {}
def _src_span_dnsmos(fid, seg_bounds):
    """OVRL لنفس المدة الزمنية من المصدر المختار — مقارنة مثل بمثل.
    
    يكشف هذا الفحص أضرار التصدير فعلًا: إعادة العينة، تكميم PCM، القص الرقمي —
    لأن المحتوى المرجعي هو نفسه وضعه المقطع قبل التصدير.
    """
    if seg_bounds is None:
        return None
    key = (fid, round(seg_bounds[0], 3), round(seg_bounds[1], 3))
    if key in _SPAN_CACHE:
        return _SPAN_CACHE[key]
    try:
        y, sr = sf.read(str(SELECTED_PATH[fid]), dtype="float32")
        s_i, e_i = int(seg_bounds[0] * sr), min(int(seg_bounds[1] * sr), len(y))
        res = dnsmos_full(y[s_i:e_i], sr)
        del y
        _SPAN_CACHE[key] = res.get("OVRL") if res else None
    except Exception:
        _SPAN_CACHE[key] = None
    return _SPAN_CACHE[key]

qa_rows = []
emb_src_cache = {}   # بصمة المصدر الخام لكل fid (masters)
span_emb_cache = {}  # بصمة المصدر لنفس المدة الزمنية لكل مقطع (مثل بمثل)

def _src_emb(fid):
    """بصمة المصدر الخام — مرجع فحص الهوية للملفات الكاملة (masters)."""
    if fid not in emb_src_cache:
        emb_src_cache[fid] = None
        try:
            r = KEPT[KEPT["fid"] == fid]
            if len(r):
                y, sr = sf.read(r["wav_path"].iloc[0], dtype="float32")
                emb_src_cache[fid] = ecapa_embedding(y, sr)
                del y
        except Exception:
            pass
    return emb_src_cache[fid]

def _span_emb(fid, seg_id, bounds):
    """بصمة المصدر المختار لنفس المدة الزمنية للمقطع — مقارنة مثل بمثل.

    مقارنة مقطع 8s بملف كامل 5 دقائق تعطي تشابهًا منخفضًا زائفًا (متوسط نوافذ
    مختلفة) — الفحص الصادق: المقطع مقابل نفس مدته في مصدره.
    """
    key = (fid, seg_id)
    if key not in span_emb_cache:
        span_emb_cache[key] = None
        try:
            y, sr = sf.read(str(SELECTED_PATH[fid]), dtype="float32")
            s_i, e_i = int(bounds[0] * sr), min(int(bounds[1] * sr), len(y))
            span_emb_cache[key] = ecapa_embedding(y[s_i:e_i], sr)
            del y
        except Exception:
            pass
    return span_emb_cache[key]

for row in tqdm(EXPORT_REPORT.to_dict("records"), total=len(EXPORT_REPORT),
                desc="ضمان الجودة", unit="ملف"):
    fid = row["fid"]
    seg_bounds = None
    seg_row = SEGMENTS[SEGMENTS["seg_id"] == row.get("seg_id")] if row.get("seg_id") else None
    if seg_row is not None and len(seg_row):
        seg_bounds = (float(seg_row["start_s"].iloc[0]), float(seg_row["end_s"].iloc[0]))
    _QACFG = {"quality": {k: CONFIG["quality"][k] for k in
                          ("min_file_dnsmos_ovrl", "degradation_reject",
                           "degradation_review", "min_speaker_sim_processed",
                           "max_true_peak_db", "max_dc_offset", "max_clipping_ratio")},
              "target_sr": TARGET_SR, "subtype": SUBTYPE,
              "identity_ref": "span_v2"}   # مثل بمثل: مقطع↔نفس مدته بالمصدر
    if row["kind"].startswith("clip") and seg_bounds:
        _emb_ref = _span_emb(fid, row.get("seg_id"), seg_bounds)
    else:
        _emb_ref = _src_emb(fid)
    def _qa():
        return qa_one_file({**row, "_seg_bounds": seg_bounds}, y_src_cache=_emb_ref)
    out = file_cache("qa", f"{fid}_{row.get('seg_id') or 'master'}", _QACFG, _qa)
    qa_rows.append({"fid": fid, "path": row["path"], "kind": row["kind"],
                    "seg_id": row.get("seg_id"), **out})
    if out["result"] in ("REJECTED", "PROCESSING_FAILED"):
        STATE.log_error(fid, "qa", "QARejected", "warn",
                        f"{out['result']}: {out['reasons']}")
    STATE.metric(fid, "qa", "ovrl", out.get("dnsmos_ovrl") or -1)

emb_src_cache.clear()
memory_cleanup()
QA = pd.DataFrame(qa_rows)
QA.to_csv(PATHS["reports"] / "final_quality.csv", index=False)
_counts = QA["result"].value_counts().to_dict()
print("نتائج QA:", _counts)
_n_bad = int(QA["result"].isin(["REJECTED", "PROCESSING_FAILED"]).sum())
if _n_bad:
    print(QA[QA["result"].isin(["REJECTED", "PROCESSING_FAILED"])][
          ["kind", "path", "reasons"]].to_string(index=False))

## STAGE 21 — Dataset Diversity Analysis

**السؤال الجوهري:** هل بياناتنا **متنوعة فعلًا** أم تكرار مجموعة محدودة من العبارات؟ مجموعة بيانات مكررة = نموذج يحفظ ولا يعمم.

**ما يقاس (من الملفات النهائية المقبولة فعليًا):**
- المدد: المصدر / الكلام / المقبول — وتوزيع مدد المقاطع,
- **الطبقة الصوتية (F0):** توزيع متوسط/وسيط F0 لكل ملف (pyin — مقياس موثوق للكلام) + كشف الشواذ (تسجيل مريض/معدل سرعة خطأ),
- الجهارة/الضوضاء/الجودة: توزيعات LUFS التقديرية وأرضيات الضوضاء وDNSMOS,
- **الفونيمات حصريًا بالعربية عند توفر ASR** — وإلا (الافتراضي) يوثّق أن تحليل المحتوى اللغوي غير منفذ (لا ادعاءات بأداة غير مختبرة على اللهجة),
- التكرارات النهائية (المدة الفريدة مقابل الإجمالية).

**مخرجات:** `DIVERSITY` + `reports/diversity_report.csv` + رسوم توزيع إلى `visualizations/` (عينات تمثيلية لا 1000 رسم فوق بعضها).

In [ ]:
# -*- coding: utf-8 -*-
"""STAGE 21 — تحليل تنوع مجموعة البيانات الفعلية (لا الافتراضية)."""
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import librosa as lr

acc = QA[(QA["kind"].str.startswith("clip")) & (QA["result"] == "PASS")]
div_rows = []
_F0CFG = {"sr_mode": "pyin_60_500"}
for r in tqdm(KEPT.itertuples(), total=len(KEPT), desc="تنوّع: F0", unit="ملف"):
    def _f0():
        y, sr = sf.read(str(SELECTED_PATH[r.fid]), dtype="float32")
        spans = FINAL_SPANS[r.fid]
        ys = np.concatenate([y[int(a * sr): int(min(b, len(y) / sr) * sr)] for a, b in spans
                             if int(min(b, len(y) / sr) * sr) > int(a * sr)]) if spans else y
        if len(ys) < sr // 2:
            ys = y
        try:
            f0, voiced_flag, _ = lr.pyin(ys, fmin=60, fmax=500, sr=sr,
                                         frame_length=1024 if sr <= 24000 else 2048)
            f0v = f0[~np.isnan(f0)]
        except Exception as e:
            LOGGER.warning("pyin فشل لـ %s: %s", r.orig_name, e)
            f0v = np.array([np.nan])
        del y, ys
        nanmean = float(np.nanmean(f0v)) if len(f0v) else np.nan
        nanmed = float(np.nanmedian(f0v)) if len(f0v) else np.nan
        return {"f0_mean_hz": round(nanmean, 1) if np.isfinite(nanmean) else None,
                "f0_median_hz": round(nanmed, 1) if np.isfinite(nanmed) else None,
                "f0_p10": round(float(np.nanpercentile(f0v, 10)), 1) if len(f0v) > 3 else None,
                "f0_p90": round(float(np.nanpercentile(f0v, 90)), 1) if len(f0v) > 3 else None,
                "voiced_ratio": round(float(np.mean(~np.isnan(f0))), 3) if len(f0) else None}
    d = file_cache("f0", r.fid, _F0CFG, _f0)
    div_rows.append({"fid": r.fid, "orig_name": r.orig_name, **d})

PITCH = pd.DataFrame(div_rows)
DIVERSITY = PITCH.merge(
    DIAG[["fid", "dur_s", "rms_db", "noise_floor_est_db", "env_smoothness"]], on="fid", how="left")
if len(DNSMOS_DF):
    DIVERSITY = DIVERSITY.merge(DNSMOS_DF[["fid", "OVRL", "SIG", "BAK"]], on="fid", how="left")

# كشف شواذ الطبقة (تسجيل خاطئ السرعة/متحدث مختلف محتمل)
if DIVERSITY["f0_median_hz"].notna().sum() >= 3:
    med = DIVERSITY["f0_median_hz"].dropna()
    q1, q3 = med.quantile(0.25), med.quantile(0.75)
    iqr = q3 - q1
    outliers = med[(med < q1 - 1.5 * iqr) | (med > q3 + 1.5 * iqr)]
    DIVERSITY["f0_outlier"] = DIVERSITY["f0_median_hz"].isin(outliers)
    if len(outliers):
        LOGGER.warning("شواذ F0 (احتمال متحدث آخر/سرعة خاطئة): %s",
                       DIVERSITY[DIVERSITY['f0_outlier']]['orig_name'].tolist())

DIVERSITY.to_csv(PATHS["reports"] / "diversity_report.csv", index=False)

def r_path_of(fid):
    """مسار الملف الخام المفكوك."""
    return str(PATHS["work"] / "01_decoded" / f"{fid}.wav")

### لوحة الرسوم التمثيلية: توزيعات المدة/الطبقة/الجهارة/الضوضاء/الجودة/المقاطع + مثال قبل/بعد (موجة وطيف).

In [ ]:
# ── الرسوم: توزيعات تمثيلية مفيدة (لا تراكم) ──
if CONFIG["export"]["visualizations"]:
    V = PATHS["visualizations"]
    fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
    fig.suptitle("Dataset Diversity — توزيعات المدة/الطبقة/الجهارة/الضوضاء/الجودة/المقاطع", fontsize=13)
    axes[0, 0].hist(DIVERSITY["dur_s"], bins=min(20, max(5, len(DIVERSITY))), color="#4C72B0")
    axes[0, 0].set_title("مدة الملفات (s)")
    if DIVERSITY["f0_median_hz"].notna().any():
        axes[0, 1].hist(DIVERSITY["f0_median_hz"].dropna(), bins=15, color="#DD8452")
    axes[0, 1].set_title("F0 وسيط (Hz)")
    axes[0, 2].hist(DIVERSITY["rms_db"].dropna(), bins=15, color="#55A868")
    axes[0, 2].set_title("RMS (dBFS)")
    axes[1, 0].hist(DIVERSITY["noise_floor_est_db"].dropna(), bins=15, color="#C44E52")
    axes[1, 0].set_title("أرضية الضوضاء المقدرة (dB)")
    if DIVERSITY.get("OVRL") is not None and DIVERSITY["OVRL"].notna().any():
        axes[1, 1].hist(DIVERSITY["OVRL"].dropna(), bins=15, color="#8172B3")
    axes[1, 1].set_title("DNSMOS OVRL")
    if len(SEGMENTS):
        axes[1, 2].hist(SEGMENTS["dur_s"], bins=30, color="#937860")
    axes[1, 2].set_title("مدد المقاطع (s)")
    for ax in axes.ravel():
        ax.grid(alpha=0.25)
    fig.savefig(V / "diversity_dashboard.png", dpi=110)
    plt.close(fig)
    # قبل/بعد لمثال واحد ممثل (الأطول كلامًا)
    try:
        ex = max(KEPT.itertuples(), key=lambda r: spans_total(FINAL_SPANS[r.fid]))
        y_raw, sr_r = sf.read(r_path_of(ex.fid), dtype="float32")
        y_sel, sr_s = sf.read(str(SELECTED_PATH[ex.fid]), dtype="float32")
        fig, axes = plt.subplots(2, 2, figsize=(14, 6), constrained_layout=True)
        fig.suptitle(f"قبل/بعد: {ex.orig_name}", fontsize=13)
        t_r = np.arange(len(y_raw)) / sr_r; t_s = np.arange(len(y_sel)) / sr_s
        axes[0, 0].plot(t_r[:: max(1, len(t_r) // 5000)], y_raw[:: max(1, len(y_raw) // 5000)], lw=0.4, color="#888")
        axes[0, 0].set_title("الموجة الخام")
        axes[0, 1].plot(t_s[:: max(1, len(t_s) // 5000)], y_sel[:: max(1, len(y_sel) // 5000)], lw=0.4, color="#C44E52")
        axes[0, 1].set_title("الموجة النهائية")
        S1 = lr.amplitude_to_db(np.abs(lr.stft(y_raw, n_fft=1024)), ref=1.0)
        S2 = lr.amplitude_to_db(np.abs(lr.stft(y_sel, n_fft=1024)), ref=1.0)
        axes[1, 0].imshow(S1, origin="lower", aspect="auto", cmap="magma")
        axes[1, 0].set_title("طيف الخام")
        axes[1, 1].imshow(S2, origin="lower", aspect="auto", cmap="magma")
        axes[1, 1].set_title("طيف النهائي")
        for ax in axes[1]: ax.set_xlabel("إطار"); ax.set_ylabel("تردد")
        fig.savefig(V / "before_after_example.png", dpi=110)
        plt.close(fig)
        del y_raw, y_sel
    except Exception as e:
        LOGGER.warning("رسم قبل/بعد فشل: %s", e)

_dcols = [c for c in ["orig_name", "dur_s", "f0_median_hz", "f0_outlier", "voiced_ratio",
                      "noise_floor_est_db", "rms_db", "OVRL"] if c in DIVERSITY.columns]
print(DIVERSITY[_dcols].round(2).to_string(index=False))
_total_speech = SEGMENTS["dur_s"].sum() if len(SEGMENTS) else 0
print(f"\nمقاطع مقبولة: {len(acc)} | إجمالي مدتها: {_total_speech/60:.1f} دقيقة | "
      f"معدل مدة المقطع: {(SEGMENTS['dur_s'].mean() if len(SEGMENTS) else 0):.1f}s")

## STAGE 22 — Training / Evaluation / Quarantine Split

**القاعدة الذهبية — منع التسريب:** التقسيم على **مستوى التسجيل المصدر** لا المقطع — كل مقاطع التسجيل الواحد تذهب لنفس الجهة (مقاطع من نفس التسجيل موزعة بين التدريب والتقييم = تقييم زائف مخادع).

- **النسب قابلة للضبط** (`split.train_ratio`/`eval_ratio`) وتتأقلم مع الحجم: أرشيف صغير يحجز تقييمًا أصغر (حد أدنى عملي) — لا نسبة عمياء تُفرَغ مجموعة صغيرة من محتواها.
- **الحجز للاختبار:** تسجيل كامل (وليس مقاطع) يُحجز في `evaluation/held_out/` لاختبار النموذج **بعد** التدريب (الغاية: قياس موضوعي للتحويل على كلام لم يرَه التدريب).
- **Quarantine مُصنّف بالأسباب:** wrong_speaker / corrupted / low_quality / duplicate / uncertain — كل ملف مستبعد **موجود فعليًا** في مجلده مع سببه في الـ Manifest (لا حذف).
- **تسجيل كامل:** مدد وعدد وتوزع وأسباب كل جهة — قابلة للمراجعة.

**مخرجات:** `SPLIT` (لكل مقطع: جهته) + مجلد `evaluation/held_out/` + `reports/split_report.csv`.

In [ ]:
# -*- coding: utf-8 -*-
"""STAGE 22 — تقسيم مقاوم للتسريب (مستوى المصدر) + حجر صحي مُصنّف."""
SPLIT = {"train_ratio": 0.9, "eval_ratio": 0.1, "min_eval_recordings": 1, "only_if_recordings_ge": 4}

def source_level_split(fids):
    """تقسيم على مستوى التسجيل: كل مقاطع المصدر لجهة واحدة (منع التسريب)."""
    import random as rnd
    rnd = __import__("random")
    fids = list(fids)
    if len(fids) < SPLIT["only_if_recordings_ge"]:
        LOGGER.info("أرشيف صغير (%d تسجيلات) — لا حجز تقييم (كل البيانات للتدريب)", len(fids))
        return {f: "train" for f in fids}, {}
    rng = rnd.Random(CONFIG["runtime"]["seed"])
    shuffled = fids[:]; rng.shuffle(shuffled)
    n_eval = max(SPLIT["min_eval_recordings"], int(round(len(shuffled) * SPLIT["eval_ratio"])))
    n_eval = min(n_eval, max(1, len(shuffled) - 1))  # لا نفرغ التدريب
    eval_fids = set(shuffled[:n_eval])
    assign = {f: ("eval" if f in eval_fids else "train") for f in fids}
    return assign, eval_fids

all_fids = KEPT["fid"].tolist()
ASSIGN, EVAL_FIDS = source_level_split(all_fids)

# ── تصنيف الحجر الصحي من كل قرارات الخط ──
q_dir = lambda sub: (PATHS["export_root"] / "quarantine" / sub)

def quarantine_file(fid, sub, reason):
    d = q_dir(sub); d.mkdir(parents=True, exist_ok=True)
    src = PATHS["work"] / "01_decoded" / f"{fid}.wav"
    if src.exists():
        dst = d / src.name
        if not dst.exists():
            _shutil.copy2(src, dst)
    STATE.decision(fid, "split", f"quarantine_{sub}", reason)

# 1) متحدث خاطئ (تصنيف mismatch حاسم فقط — الحدودية للمراجعة البشرية)
if len(SPEAKER):
    for _, sp in SPEAKER[SPEAKER["speaker_class"] == "mismatch"].iterrows():
        quarantine_file(sp["fid"], "wrong_speaker", f"speaker_sim={sp['speaker_sim']}")
# 2) تكرارات (نُقلت في STAGE 06)
# 3) جودة منخفضة (QA رفض)
for _, qrow in QA[QA["result"].isin(["REJECTED", "PROCESSING_FAILED"])].iterrows():
    if qrow["kind"].startswith("clip") and qrow.get("reasons") and "dnsmos" in str(qrow["reasons"]):
        quarantine_file(qrow["fid"], "low_quality", str(qrow["reasons"])[:120])
# 4) غير مؤكد: هوية/حالات حدية
for _, irow in (IDENTITY_REVIEW.iterrows() if len(IDENTITY_REVIEW) else pd.DataFrame().iterrows()):
    if irow.get("verdict") == "degraded":
        quarantine_file(irow["fid"], "uncertain", f"identity_sim={irow['identity_sim']}")
    else:
        quarantine_file(irow["fid"], "uncertain", f"identity ambiguous (أرضية ضوضاء)")

# ── التسجيلات المحجوزة للتقييم بعد التدريب ──
held_dir = PATHS["export_root"] / "evaluation" / "held_out"
held_dir.mkdir(parents=True, exist_ok=True)
HELD_OUT = []
for fid in EVAL_FIDS:
    src = PATHS["export_root"] / "master" / "processed" / f"{fid}.wav"
    if src.exists():
        dst = held_dir / src.name
        if not dst.exists():
            _shutil.copy2(src, dst)
        HELD_OUT.append(str(dst))

# ── توزيع المقاطع النهائية ──
seg_assign = []
for sg in SEGMENTS.itertuples():
    side = ASSIGN.get(sg.fid, "train")
    qa_row = QA[QA["seg_id"] == sg.seg_id]["result"]
    qa_res = qa_row.iloc[0] if len(qa_row) else None
    if qa_res in ("REJECTED", "PROCESSING_FAILED"):
        side_final = "quarantine"
    elif qa_res == "NEEDS_REVIEW":
        side_final = "needs_review"
    else:
        side_final = side
    seg_assign.append({"seg_id": sg.seg_id, "fid": sg.fid, "dur_s": sg.dur_s,
                       "split": side_final})
SPLIT_DF = pd.DataFrame(seg_assign)

_split_summary = SPLIT_DF.groupby("split").agg(clips=("seg_id", "count"), dur_s=("dur_s", "sum")).round(1)
SPLIT_DF.to_csv(PATHS["reports"] / "split_report.csv", index=False)
print("توزيع المقاطع:")
print(_split_summary.to_string())
print(f"\nتسجيلات محجوزة للتقييم بعد التدريب: {len(HELD_OUT)} | "
      f"مقاطعها: {int((SPLIT_DF['split'] == 'eval').sum())}")
for p in HELD_OUT:
    print("  -", Path(p).name)

## STAGE 23 — Provenance, Manifest & Packaging

**الهدف:** كل مقطع نهائي قابل للتتبع **إلى التسجيل الأصلي وسلسلة العمليات كاملة**:

- **لكل مخرج** (master + clips): معرف المصدر وبصمته (SHA-256)، مساره الأصلي داخل الأرشيف، المراحل المطبقة والمتخطاة (مع أسباب التخطي)، النماذج المستخدمة، إعدادات المعالجة (هاش CONFIG)، تحويل معدل العينة (من←إلى وهل كان رفعًا)، حدود المقاطع، مقاييس الجودة النهائية، نتيجة QA وأسبابها، بصمة المخرج، وزمن المعالجة والتحذيرات.
- **لا بيانات غير موجودة:** كل حقل يوثق قيمة فعلية أو `null` صريح — الواجهة البرمجية للفحص: `MANIFEST["outputs"][...]`.
- **نسختان:** `manifest.json` تقنية كاملة + `dataset_index.csv` مختصرة للفرز السريع.
- **التغليف:** `rvc_dataset.zip` يحوي الاثنين والمخرجات — **لا يُغلَّف إلا بعد فحص QA** (لا تغليف فوق أخطاء حرجة موثقة).

**مخرجات:** `MANIFEST` + `dataset_index.csv` + `RVC_Audio_Dataset/rvc_dataset.zip`.

In [ ]:
# -*- coding: utf-8 -*-
"""STAGE 23 — Manifest كامل + فهرس + تغليف مشروط بسلامة QA."""
import zipfile as zipfile_mod

def stage_history(fid):
    """تاريخ مراحل الملف من قاعدة الحالة (قابل للتتبع الفعلي)."""
    rows = STATE.conn.execute(
        "SELECT stage, status, duration_s, error FROM stages WHERE fid=? ORDER BY rowid", (fid,)).fetchall()
    return [{"stage": s, "status": st, "duration_s": d, "error": e} for s, st, d, e in rows]

def models_used_for(fid):
    """النماذج الفعلية الموثقة لهذا الملف."""
    models = {}
    if SELECTED_LABEL.get(fid) == "original_vocals_separated":
        models["separation"] = _MODEL_REGISTRY.get("loaded_model")
    if SELECTED_LABEL.get(fid) == "original_dereverb":
        models["dereverb"] = CONFIG["dereverb"]["model"]
    if SELECTED_LABEL.get(fid, "").startswith(("uvr", "noisereduce", "deepfilternet", "mossformer", "resemble")):
        models["enhancement"] = SELECTED_LABEL[fid]
    return models

def output_manifest_row(row):
    """سجل Manifest لمخرج واحد (master أو clip)."""
    fid = row["fid"]
    reg = KEPT[KEPT["fid"] == fid]
    reg_row = reg.iloc[0].to_dict() if len(reg) else {}
    qa = QA[QA["path"] == row["path"]]
    qa_row = qa.iloc[0].to_dict() if len(qa) else {}
    out_path = Path(row["path"])
    seg = SEGMENTS[SEGMENTS["seg_id"] == row.get("seg_id")].iloc[0].to_dict() if row.get("seg_id") and len(SEGMENTS[SEGMENTS["seg_id"] == row.get("seg_id")]) else None
    return {
        "output_path": str(out_path.relative_to(PATHS["export_root"])),
        "output_sha256": sha256_file(out_path),
        "kind": row["kind"],
        "source": {
            "fid": fid,
            "original_member_path": reg_row.get("member_path"),
            "original_name": reg_row.get("orig_name"),
            "source_sha256": reg_row.get("file_sha256"),
            "decoded_from": reg_row.get("codec"),
            "source_sample_rate": reg_row.get("dec_sr"),
        },
        "processing": {
            "selected_candidate": SELECTED_LABEL.get(fid),
            "models": models_used_for(fid),
            "config_hash": stage_cache_key("export-all", {"profile": CONFIG["audio"]}),
            "target_sr": TARGET_SR,
            "resampled": reg_row.get("dec_sr") != TARGET_SR,
            "upconverted": bool(reg_row.get("dec_sr") and reg_row["dec_sr"] < TARGET_SR),
            "bit_depth": CONFIG["audio"]["bit_depth"],
            "true_peak_reduction_db": row.get("true_peak_reduction_db"),
            "gain_db": row.get("gain_db"),
        },
        "segment": ({"seg_id": seg["seg_id"], "start_s": seg["start_s"], "end_s": seg["end_s"],
                     "dur_s": seg["dur_s"], "short": bool(seg["short"]),
                     "verify_flags": seg.get("flags")}
                    if seg else None),
        "quality": {
            "qa_result": qa_row.get("result"), "qa_reasons": qa_row.get("reasons"),
            "qa_warnings": qa_row.get("warnings"),
            "dnsmos_ovrl": qa_row.get("dnsmos_ovrl"), "dnsmos_sig": qa_row.get("dnsmos_sig"),
            "true_peak_db": qa_row.get("true_peak_db"),
            "identity_sim": qa_row.get("identity_sim"),
            "speech_ratio": qa_row.get("speech_ratio"),
        },
        "split": (SPLIT_DF[SPLIT_DF["seg_id"] == row.get("seg_id")]["split"].iloc[0]
                  if row.get("seg_id") and len(SPLIT_DF[SPLIT_DF["seg_id"] == row.get("seg_id")]) else None),
        "stage_history": stage_history(fid),
        "warnings": [w for w in [qa_row.get("warnings")] if w],
    }

### بناء الـ Manifest الكامل + الفهرس المختصر + التغليف المشروط بسلامة QA (لا ZIP فوق أخطاء حرجة موثقة).

In [ ]:
_manifest_outputs = [output_manifest_row(r) for r in tqdm(
    EXPORT_REPORT.to_dict("records"), total=len(EXPORT_REPORT), desc="Manifest", unit="مخرج")]

MANIFEST = {
    "pipeline_version": PIPELINE_VERSION,
    "created_at": now_iso(),
    "rvc_profile": CONFIG["audio"]["rvc_profile"],
    "target_sr": TARGET_SR,
    "bit_depth": CONFIG["audio"]["bit_depth"],
    "environment": {"device": DEVICE, "lib_versions": LIB_VERSIONS},
    "counts": {
        "source_files": int(len(INVENTORY)),
        "decoded": int(len(REGISTRY)),
        "after_dedup": int(len(KEPT)),
        "duplicates_removed": int(len(QUARANTINE_DUP)) if "QUARANTINE_DUP" in globals() else 0,
        "clips": int(len(SEGMENTS)),
        "outputs": int(len(_manifest_outputs)),
    },
    "split_summary": {k: {"clips": int(v["clips"]), "dur_s": float(v["dur_s"])}
                      for k, v in SPLIT_DF.groupby("split").agg(
                          clips=("seg_id", "count"), dur_s=("dur_s", "sum")).iterrows()},
    "held_out_recordings": [str(Path(p).name) for p in HELD_OUT],
    "outputs": _manifest_outputs,
}

atomic_write_json(PATHS["export_root"] / "manifest.json", MANIFEST)

# فهرس مختصر قابل للفرز
_idx = pd.DataFrame([{
    "output": m["output_path"], "kind": m["kind"], "fid": m["source"]["fid"],
    "source": m["source"]["original_member_path"],
    "candidate": m["processing"]["selected_candidate"],
    "resampled": m["processing"]["resampled"],
    "upconverted": m["processing"]["upconverted"],
    "qa": m["quality"]["qa_result"],
    "dnsmos_ovrl": m["quality"]["dnsmos_ovrl"],
    "identity_sim": m["quality"]["identity_sim"],
    "split": m["split"], "dur_s": (m["segment"] or {}).get("dur_s"),
} for m in _manifest_outputs])
_idx.to_csv(PATHS["export_root"] / "dataset_index.csv", index=False)

# ── التغليف: مشروط بسلامة QA (لا تغليف فوق أخطاء حرجة غير موثقة) ──
if CONFIG["export"]["package_zip"]:
    _critical = int(QA["result"].eq("PROCESSING_FAILED").sum())
    if _critical:
        LOGGER.error("لا تغليف: %d مخرجًا PROCESSING_FAILED — راجع reports/final_quality.csv أولاً", _critical)
    else:
        zip_path = PATHS["export_root"] / CONFIG["export"]["zip_name"]
        with zipfile_mod.ZipFile(zip_path, "w", zipfile_mod.ZIP_DEFLATED, compresslevel=6) as zf:
            for p in PATHS["export_root"].rglob("*"):
                if p.is_file() and p != zip_path:
                    zf.write(p, p.relative_to(PATHS["export_root"]))
        LOGGER.info("غُلّفت المجموعة: %s (%.1f MB)", zip_path.name, zip_path.stat().st_size / 1e6)
        print(f"📦 {zip_path} — {zip_path.stat().st_size/1e6:.1f} MB")
print(f"manifest.json: {len(_manifest_outputs)} مخرجًا موثقًا | dataset_index.csv: {len(_idx)} صفًا")

## STAGE 24 — Final Audit & Completion Report

**الفحص الأخير الصارم (لا يُصدَّق الدفتر قبلها):**
1. **اكتمال فعلي:** كل مخرجات Manifest موجودة على القرص وبصماتها مطابقة (عينة تحقق كاملة للأعداد + تحقق بصمة لكل مقطع نهائي).
2. **فئات QA موزونة:** لا PROCESSING_FAILED غير موثق، وأعداد REJECTED/NEEDS_REVIEW معروضة للقرار البشري.
3. **الإجابة عن الأسئلة العشرين** لكل ملف (موثقة عبر الـ Manifest + التقارير — الـ SQL يجيب عنها كوديًا لا نصيًا).
4. **ملخص الإنجاز:** مصدر ← جرد ← إزالة تكرار ← تحقق متحدث ← معالجة مشروطة ← تقطيع ← QA ← تقسيم — بأرقام حقيقية مقيسة.

**مخرجات:** تقرير إنجاز نهائي مطبوع + `reports/completion_report.json`.

In [ ]:
# -*- coding: utf-8 -*-
"""STAGE 24 — التدقيق النهائي: اكتمال + بصمات + ملخص موثوق."""

audit = {"checks": [], "critical": 0}

def check(name, ok, detail=""):
    audit["checks"].append({"check": name, "ok": bool(ok), "detail": str(detail)[:200]})
    if not ok:
        audit["critical"] += 1
    return ok

# 1) اكتمال المخرجات (كلها لا عينة)
_missing = [m for m in MANIFEST["outputs"]
           if not (PATHS["export_root"] / m["output_path"]).exists()]
check("outputs_exist", not _missing, f"مفقود: {[m['output_path'] for m in _missing][:5]}")

# 2) بصمات المخرجات النهائية (كل clip)
_bad_sha = []
for m in MANIFEST["outputs"]:
    p = PATHS["export_root"] / m["output_path"]
    if p.exists() and sha256_file(p) != m["output_sha256"]:
        _bad_sha.append(m["output_path"])
check("output_sha_match", not _bad_sha, f"مخالف: {_bad_sha[:5]}")

# 3) QA بلا مخرجات فاشلة موثقة
_pf = int(QA["result"].eq("PROCESSING_FAILED").sum())
check("no_processing_failed", _pf == 0, f"PROCESSING_FAILED = {_pf}")

# 4) التقارير الأساسية موجودة
_required_reports = ["inventory.csv", "diagnostics.csv", "deduplication.csv",
                     "noise_analysis.csv", "segmentation.csv", "final_quality.csv",
                     "split_report.csv", "identity_preservation.csv", "selection_report.csv"]
_missing_reports = [r for r in _required_reports
                    if not (PATHS["reports"] / r).exists()]
check("reports_exist", not _missing_reports, f"مفقود: {_missing_reports}")

# 5) المانيفست قابل للتحميل
check("manifest_valid", len(MANIFEST["outputs"]) == len(EXPORT_REPORT))

# 6) الإجابة الكودية عن الأسئلة العشرين (عبر عرض مثال ملف كامل التتبع)
_ex = MANIFEST["outputs"][0] if MANIFEST["outputs"] else None
check("provenance_example", _ex is not None and _ex["source"]["source_sha256"] is not None)

audit["completed_at"] = now_iso()
audit["pipeline_version"] = PIPELINE_VERSION
audit["counts"] = MANIFEST["counts"]
audit["qa_distribution"] = QA["result"].value_counts().to_dict()
audit["split_summary"] = MANIFEST["split_summary"]
atomic_write_json(PATHS["reports"] / "completion_report.json", audit)

print("═" * 64)
print("FINAL AUDIT — التدقيق النهائي")
print("═" * 64)
for c in audit["checks"]:
    print(f"  [{'✓' if c['ok'] else '✗'}] {c['check']}" + (f" — {c['detail']}" if c["detail"] and not c["ok"] else ""))
print("─" * 64)
_ncrit = audit["critical"]
_verdict = "✔ ناجح — لا فشل حرج" if _ncrit == 0 else f"✘ {_ncrit} فحصًا حرجًا فشل"
print(f"النتيجة: {_verdict}")
print(f"المصادر: {audit['counts']['source_files']} → فريدة: {audit['counts']['after_dedup']} "
      f"(تكرارات: {audit['counts']['duplicates_removed']})")
print(f"المقاطع: {audit['counts']['clips']} | المخرجات: {audit['counts']['outputs']} | "
      f"QA: {audit['qa_distribution']}")
print(f"التقسيم: " + ", ".join(f"{k}={v['clips']}" for k, v in audit["split_summary"].items()))
if _ex:
    print("\nمثال تتبع كامل (أول مخرج):")
    print(f"  { _ex['output_path']} ← {_ex['source']['original_member_path']}"
          f" (sha {str(_ex['source']['source_sha256'])[:10]}…)")
    print(f"  المرشح: {_ex['processing']['selected_candidate']} | QA: {_ex['quality']['qa_result']}"
          f" | split: {_ex['split']}")
print("═" * 64)

## الملحق أ — Human Review Interface (المراجعة البشرية)

**الفلسفة:** القرارات الحساسة الحدودية **لا تُحسم آليًا** — تُعرض للمراجعة بأدوات استماع مدمجة.

**قوائم المراجعة المجمعة تلقائيًا:**
1. مقاطع `needs_review` من فحص المقاطع (حدود قاسية/قص/كلام قليل),
2. ملفات هوية متحدث غير محسومة (borderline/ambiguous),
3. تعارضات محرك الاختيار (OVRL↑ مقابل SIG↓),
4. مقاطع قصيرة محفوظة بعلم `short`,
5. شواذ F0 (احتمال متحدث آخر).

**الاستخدام في Jupyter/Colab:** مشغلات صوت مدمجة (A/B: أصلي/معالج) عبر `IPython.display.Audio` — تعمل بلا ipywidgets. في بيئات بلا واجهة: القوائم تصدر CSV للمراجعة اليدوية (لا يعيق عملها غياب الواجهة).

In [ ]:
# -*- coding: utf-8 -*-
"""الملحق أ — واجهة المراجعة البشرية: قوائم + مشغلات A/B مدمجة (اختيارية الواجهة)."""

def build_review_queue():
    """تجميع كل قوائم المراجعة من كل المراحل (بأسبابها)."""
    queue = []
    if len(SEGMENTS):
        for sg in SEGMENTS[SEGMENTS["verify"] == "needs_review"].itertuples():
            queue.append({"item_type": "segment", "id": sg.seg_id, "fid": sg.fid,
                          "why": f"فحص المقاطع: {sg.flags}",
                          "audio": str(PATHS["export_root"] / "clips" / "needs_review" / f"{sg.seg_id}.wav")})
        for sg in SEGMENTS[SEGMENTS["short"] == True].itertuples() if SEGMENTS["short"].any() else []:
            queue.append({"item_type": "segment", "id": sg.seg_id, "fid": sg.fid,
                          "why": f"مقطع قصير ({sg.dur_s:.1f}s < min_clip_s) — قرار: إبقاء/دمج/إسقاط",
                          "audio": ""})
    if len(REVIEW_SPEAKER):
        for _, sp in REVIEW_SPEAKER.iterrows():
            queue.append({"item_type": "speaker", "id": sp["fid"], "fid": sp["fid"],
                          "why": f"هوية متحدث: {sp['speaker_class']} (sim={sp['speaker_sim']})",
                          "audio": r_path_of(sp["fid"])})
    if len(IDENTITY_REVIEW):
        for _, irow in IDENTITY_REVIEW.iterrows():
            queue.append({"item_type": "identity", "id": irow["fid"], "fid": irow["fid"],
                          "why": f"هوية بعد المعالجة: {irow['verdict']} (sim={irow['identity_sim']})",
                          "audio": SELECTED_PATH.get(irow["fid"], "")})
    if len(DIVERSITY) and DIVERSITY.get("f0_outlier") is not None:
        for _, drow in DIVERSITY[DIVERSITY["f0_outlier"] == True].iterrows() if DIVERSITY["f0_outlier"].any() else []:
            queue.append({"item_type": "f0", "id": drow["fid"], "fid": drow["fid"],
                          "why": f"طبقة صوتية شاذة: F0={drow['f0_median_hz']}Hz — متحدث آخر محتمل؟",
                          "audio": r_path_of(drow["fid"])})
    return queue

REVIEW_QUEUE = build_review_queue()
REVIEW_DF = pd.DataFrame(REVIEW_QUEUE)
if len(REVIEW_DF):
    REVIEW_DF.to_csv(PATHS["reports"] / "human_review_queue.csv", index=False)
    print(f"📋 قائمة المراجعة البشرية: {len(REVIEW_DF)} عنصرًا (نُسخت إلى reports/human_review_queue.csv)")
    print(REVIEW_DF[["item_type", "why"]].to_string(index=False))
else:
    print("✓ لا عناصر تحتاج مراجعة بشرية")

def review_player(path_a, path_b, title="A/B"):
    """مشغل A/B داخل Jupyter (يعمل بلا ipywidgets — يتحلل بلطف خارجها)."""
    try:
        from IPython.display import Audio, display
        print(f"▶ {title}: [A] الأصلي")
        display(Audio(filename=str(path_a)))
        if path_b:
            print(f"▶ {title}: [B] المعالج")
            display(Audio(filename=str(path_b)))
    except ImportError:
        print(f"(خارج Jupyter — استمع يدويًا إلى: {path_a} / {path_b})")

if len(REVIEW_QUEUE):
    _n_show = min(3, len(REVIEW_QUEUE))  # عينة عرض: أول 3 (البقية في CSV)
    for _item in REVIEW_QUEUE[:_n_show]:
        review_player(_item["audio"],
                      SELECTED_PATH.get(_item["fid"]) if _item["item_type"] == "speaker" else None,
                      title=_item["why"][:70])

## الملحق ب — التدريب على المخرجات في RVC v2 / Applio

### 1) أي مخرجات تُدخل للتدريب؟

| المخرج | متى تستخدمه | لماذا |
|---|---|---|
| `RVC_Audio_Dataset/clips/accepted/` | **الافتراضي** — مقاطع منسّقة 2–15s بمعدل الهدف | RVC يقبلها مباشرة (لن يعيد تقطيعها إذا كانت أقصر من حده) |
| `RVC_Audio_Dataset/master/processed/` | بديل مفضّل لدى البعض | دع RVC/Applio يقطّع بنفسه (slice_db) — التقطيع هناك مبني على عتبة صمت الطاقة فقط، أقل وعيًا بحدود الكلام من تقطيعنا |
| `evaluation/held_out/` | **ليس للتدريب** | تقييم التحويل بعد التدريب على كلام لم يره النموذج |

**توصية هندسية:** ابدأ بـ`clips/accepted`؛ إن وجدت التدريب يهمل كلامًا مهمًا (مقاطع مراجعة)، أنجز قائمة المراجعة ثم أعد التدريب.

### 2) خطوات التدريب (Applio — الواجهة الأبسط)
1. Model name ← اسم نموذجك.
2. **Sample rate** ← نفس `rvc_profile` (48k افتراضيًا هنا — تأكد من التوافق مع ملفات G/D المسبقة).
3. Upload dataset ← ارفع مجلد (أو محتوى) `clips/accepted` (أو أرشفة `rvc_dataset.zip`).
4. استخدم **RVC v2** + pretrained base المناسب لنفس معدل العينة.
5. عيّن `speaker_id=0` (متحدث واحد).
6. درّب (batch صغير إذا كانت الذاكرة محدودة؛ GPU ≥8GB مريحة).

### 3) خطوات التدريب (RVC الأصلي — WebUI)
- ضع المجلد تحت `datasets/<اسم>/`
- `python infer-web.py` → Config training: sampling_rate مطابق، `n_spk=1`، ثم preprocess (سيستخرج F0 والمعالم على المقاطع كما هي) → Feature extraction → Train.

### 4) بعد التدريب — التقييم الموضوعي
1. حوّل 3–5 ملفات من `evaluation/held_out/` بالنموذج الناتج.
2. قارن بصمة ECAPA للناتج مع بصمة الملفات الأصلية نفسها (سكربت الخط يحسبها في `IDENTITY`).
3. استمع لطبيعية التنفس والسواكن (الحروف الاحتكاكية أول ضحايا التنقية المفرطة — إن كانت باهتة، راجع قرارات STAGE 13).

### 5) ملاحظات توافق دقيقة
- **48kHz يستهلك ذاكرة أكثر** في التدريب؛ 40k خيار معقول إذا كانت توليفاتك 40k — غيّر `audio.rvc_profile` وأعد تشغيل المراحل من STAGE 18 (الذاكرة المؤقتة تُبطل تلقائيًا لأن المفتاح تغيّر).
- لا تخلط معدلات مختلفة في مجلد واحد — RVC يفترض التوحيد.
- `manifest.json` يوثق كل شيء — عند شك، `dataset_index.csv` يجيب خلال ثوانٍ عن مصدر أي مقطع.

## الملحق ج — الملاحظات الهندسية والمصادر

### قرارات هندسية موثقة (مع البدائل المرفوضة)
| القرار | البديل المرفوض | السبب |
|---|---|---|
| True Peak بفرط تعيين 4× | تسمية Sample Peak بالـ True Peak | تسمية خاطئة تاريخيًا — القمة الحقيقية تفوق العينات بمواقع بين-عينية تصل 1dB+ |
| أرضية ضوضاء من إطارات اللاكلام (VAD-aware) | المئين 10 من كل الإطارات | الأخير يخلط الكلام الهادئ بالضوضاء |
| إزالة تكرار بمحتوى الموجة فقط | استخدام بصمة ECAPA | نفس الشخص = تشابه عالٍ دوماً — ليس دليل تكرار |
| فصل/تنقية/صدى مشروطة ببوابات قياسية | معالجة إجبارية للجميع | كل معالجة تخاطر بالهوية والتفاصيل — لا تُطبق إلا بدليل |
| تقسيم train/eval بمستوى المصدر | تقسيم بالمقاطع | تسريب مقاطع التسجيل الواحد = تقييم مُخادع |
| إعادة عينة واحدة عند التصدير فقط | التحويل 48k عند الفك | تحويلات متتالية تتراكم أخطاء + يفقد معلومة المصدر |
| LUFS اختيارية `gain_only` | تطبيع -20 LUFS إجباري | يرفع الضوضاء مع الكلام ويكبّر الديناميكية صناعيًا |

### النماذج والمصادر المعتمدة
- **Silero VAD** — snakers4/silero-vad (MIT) — هستيريسيس مدمج v5.
- **ECAPA-TDNN** — speechbrain/spkrec-ecapa-voxceleb (Apache-2.0).
- **DNSMOS P.835/P.808** — microsoft/DNS-Challenge ONNX (أوزان مفتوحة للبحث/التقييم).
- **audio-separator** — nomadkaraoke/python-audio-separator (MIT) — BS-RoFormer/MelBand/UVR/MDX.
- **SoXR VHQ** — dofuuz/python-soxr (LGPL).
- **noisereduce** — spectral gating.
- اختياري: DeepFilterNet3 (MIT/Apache)، ClearerVoice MossFormer2_SE_48K (ModelScope License)، resemble-enhance (MIT).

### حدود معلنة بشفافية
- **F0 بـ pyin**: أدق المتاح دون تنزيل RMVPE؛ للأداء الأعلى في التدريب الفعلي يستخدم RVC RMVPE الخاص به داخليًا — تحليلنا للتقارير لا للتدريب.
- **كشف المتحدث المتنافس**: مؤشرات أولية (استدلال VAD + ECAPA) — يراجع بشريًا، لا حكم آلي.
- **النماذج الاختيارية الثقيلة** (DFN/ClearerVoice/Resemble/WhisperX): مدمجة الواجهة وتُكتشف إن وُجدت؛ لا تُثبّت تلقائيًا (قرار مستخدم واعٍ).
- **مرجع المتحدث في وضع Bootstrap**: وسم صريح يحتاج تأكيدك — للدقة القصوى ضع 3+ تسجيلات مؤكدة في `speaker.reference_dir`.